# Section 1 — Environment Setup and Dependency Verification

This section installs the required dependencies for the Santali → English vocabulary-extension and LoRA experiments.

We pin `transformers==4.46.3` and `tokenizers==0.20.3` to maintain a consistent environment with the intended IndicTrans2 implementation.

After installation, the Kaggle session must be restarted before importing the libraries. We will then verify the installed versions and GPU availability.

**Important:** Do not load the model, tokenizer, or dataset in this section. Those tasks belong to later sections.

In [ ]:
# Section 1.2 — Install Pinned Dependencies

%pip install -q --no-cache-dir \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "IndicTransToolkit==1.1.1" \
    peft accelerate sacrebleu sentencepiece kagglehub


 **MANDATORY STEP: Restart SessionClick Restart Session in Kaggle's top menu bar. This flushes Python's in-memory module cache while keeping the newly installed files on disk.**


In [ ]:
# Section 1.3 — Environment and GPU Verification

import sys
import importlib.metadata as metadata
import torch

print("=== Package Version Verification ===")

packages = [
    "transformers",
    "tokenizers",
    "IndicTransToolkit",
    "torch",
    "peft",
    "accelerate",
    "sacrebleu",
    "sentencepiece",
]

for package in packages:
    try:
        print(f"{package:22s}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:22s}: NOT INSTALLED")

print("\n=== Runtime Information ===")
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory / (1024**3)
    )
    print(f"GPU memory: {gpu_memory:.2f} GiB")

assert metadata.version("transformers") == "4.46.3"
assert metadata.version("tokenizers") == "0.20.3"

print("\nEnvironment verification passed.")

## Section 2 — Load the Shared Preprocessed Dataset

This experiment uses the published `common-tribal-preprocessing` Kaggle Dataset as the single source of truth for all data splits.

We will load the existing Santali training, validation, and internal test splits without performing additional cleaning or splitting.

- **Training split:** used for LoRA fine-tuning.
- **Validation split:** used for monitoring training, early stopping, and selecting the best checkpoint.
- **Internal test split:** reserved for final evaluation after model selection.
- **Competition test data:** kept separate and not used for training or model selection.

We will verify the loaded files, column names, row counts, missing values, and duplicate source-target pairs before proceeding.

In [ ]:
# Section 2.2 — Locate the Published Dataset

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("Top-level input folders:")
for path in sorted(INPUT_ROOT.iterdir()):
    print(" -", path.name)

# Search recursively because Kaggle may mount datasets under /kaggle/input/datasets/
matches = [
    path
    for path in INPUT_ROOT.rglob("santali_train.parquet")
    if path.parent.name == "splits"
]

assert len(matches) == 1, (
    f"Expected one santali_train.parquet file, found {len(matches)}. "
    "Check the attached dataset and input directory."
)

SPLIT_DIR = matches[0].parent
DATASET_DIR = SPLIT_DIR.parent

print("\nDataset directory:", DATASET_DIR)
print("Split directory:", SPLIT_DIR)

print("\nAvailable split files:")
for path in sorted(SPLIT_DIR.iterdir()):
    print(" -", path.name)

## Section 3 — Data Validation and Split Integrity

Validate the preprocessed training, validation, and internal test splits for Bhili, Gondi, Mundari, and Santali.

We will check expected columns, data types, missing or empty sentences, duplicate source-target pairs, and overlap between splits. The saved data will not be modified.

All checks must pass before tokenization analysis begins. The internal test splits remain reserved for final evaluation.

In [ ]:
# Section 3.2 — Validate All Languages and Splits

import pandas as pd

language_columns = {
    "bhili": ("Hindi", "Bhili"),
    "gondi": ("Hindi", "Gondi"),
    "mundari": ("Hindi", "Mundari"),
    "santali": ("English", "Santali"),
}

split_names = ("train", "validation", "test")

# Reuse SPLIT_DIR established in Section 2
all_data = {
    language: {
        split: pd.read_parquet(
            SPLIT_DIR / f"{language}_{split}.parquet"
        )
        for split in split_names
    }
    for language in language_columns
}

validation_results = []

for language, splits in all_data.items():
    source_col, target_col = language_columns[language]

    # Validate each split
    for split_name, df in splits.items():
        checks = {
            "expected_columns": (
                source_col in df.columns
                and target_col in df.columns
            ),
            "missing_values": (
                not df[[source_col, target_col]].isna().any().any()
            ),
            "empty_sentences": (
                not df[source_col].astype(str).str.strip().eq("").any()
                and not df[target_col].astype(str).str.strip().eq("").any()
            ),
            "string_types": (
                df[source_col].map(lambda x: isinstance(x, str)).all()
                and df[target_col].map(lambda x: isinstance(x, str)).all()
            ),
            "duplicate_pairs": (
                not df.duplicated(
                    subset=[source_col, target_col]
                ).any()
            ),
        }

        validation_results.append({
            "language": language,
            "split": split_name,
            "rows": len(df),
            **checks,
        })

    # Check exact source-target pair overlap between splits
    pair_sets = {
        split: set(
            map(
                tuple,
                df[[source_col, target_col]].itertuples(
                    index=False, name=None
                )
            )
        )
        for split, df in splits.items()
    }

    overlap_train_val = len(
        pair_sets["train"] & pair_sets["validation"]
    )
    overlap_train_test = len(
        pair_sets["train"] & pair_sets["test"]
    )
    overlap_val_test = len(
        pair_sets["validation"] & pair_sets["test"]
    )

    print(f"\n{language.upper()} — Exact pair overlap")
    print("Train ↔ Validation:", overlap_train_val)
    print("Train ↔ Test:", overlap_train_test)
    print("Validation ↔ Test:", overlap_val_test)

results_df = pd.DataFrame(validation_results)

print("\n=== Split Validation Results ===")
display(results_df)

check_columns = [
    "expected_columns",
    "missing_values",
    "empty_sentences",
    "string_types",
    "duplicate_pairs",
]

all_checks_passed = results_df[check_columns].all().all()

print("\nAll individual split checks passed:", all_checks_passed)

assert all_checks_passed, (
    "Validation failed. Inspect the results before proceeding."
)

print("Basic data validation passed.")

## 4. Model Selection and Tokenization Research

### 4.1 Research Configuration

This section investigates whether vocabulary extension improves neural machine translation for low-resource Indian tribal languages. The experiments will compare a standard LoRA fine-tuning baseline against a vocabulary-extended LoRA model while keeping the data splits and other experimental settings consistent.

The target language directions are:

- Hindi → Bhili
- Hindi → Gondi
- Hindi → Mundari
- English → Santali

Two model-size categories will be investigated:

1. A larger approximately 1B-parameter model fine-tuned using LoRA.
2. A smaller distilled model to assess the trade-off between computational cost and translation quality.

Before training, we will verify language support, source and target tokenizers, vocabulary dimensions, and GPU memory requirements. Tokenization fertility will be measured across all four target languages, and the most suitable pilot setup will be selected before implementing vocabulary extension.

The main experimental comparison will use the same training and evaluation data, random seed, LoRA configuration, and training budget wherever technically possible. Validation performance will guide checkpoint selection, while the test set will be reserved for final evaluation.

In [ ]:
# Section 4.1 — Environment and existing research objects

import torch
import transformers
import tokenizers
import pandas as pd

print("Python/PyTorch environment")
print("---------------------------")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3),
            2
        ),
        "GiB"
    )

print("\nExisting research objects")
print("-------------------------")

for name in ["all_data", "language_columns", "SPLIT_DIR",
             "tokenizer", "processor", "MODEL_NAME"]:
    print(f"{name}: {'available' if name in globals() else 'not defined'}")

if "all_data" in globals():
    print("\nDataset split sizes:")
    for language, splits in all_data.items():
        print(
            f"{language}:",
            {split: len(df) for split, df in splits.items()}
        )

if "MODEL_NAME" in globals():
    print("\nPreviously loaded model checkpoint:", MODEL_NAME)

### 4.2 Dataset Structure and Translation Directions

The experiments use four language pairs: Hindi–Bhili, Hindi–Gondi, Hindi–Mundari, and English–Santali. The source and target columns must be verified before selecting a model because the translation direction determines which tokenizer and language configuration are appropriate.

We will also verify that all three data splits contain the expected columns and that the existing dataset objects are consistent. This step reuses the previously loaded data rather than reading the files again.

In [ ]:
# Section 4.2 — Verify dataset columns and translation directions

expected_splits = {"train", "validation", "test"}

print("Dataset direction and schema verification")
print("=" * 55)

for language, (source_col, target_col) in language_columns.items():
    splits = all_data[language]

    assert set(splits.keys()) == expected_splits, (
        f"{language}: unexpected split names {list(splits.keys())}"
    )

    print(f"\nLanguage: {language.upper()}")
    print(f"Translation direction: {source_col} -> {target_col}")

    for split_name in ("train", "validation", "test"):
        df = splits[split_name]

        missing_cols = {
            source_col, target_col
        } - set(df.columns)

        assert not missing_cols, (
            f"{language}/{split_name}: missing columns {missing_cols}"
        )

        assert df[[source_col, target_col]].notna().all().all(), (
            f"{language}/{split_name}: missing source or target text"
        )

        print(
            f"  {split_name:10s}: {len(df):5d} rows | "
            f"source={source_col}, target={target_col}"
        )

    # Display one example to verify the actual direction.
    example = splits["train"].iloc[0]
    print("\n  Example source:", str(example[source_col])[:200])
    print("  Example target:", str(example[target_col])[:200])

print("\nAll dataset direction and schema checks passed.")

### 4.3 Candidate Model Checkpoints

We will investigate both the larger IndicTrans2 checkpoint family and the smaller distilled variants.

The purpose is to identify checkpoints that are appropriate for the required translation directions and feasible on a Tesla T4 GPU. LoRA reduces the number of trainable parameters, but it does not remove the memory cost of the frozen base model.

Candidate families include:

- Indic-to-English 1B: the checkpoint currently loaded in the notebook, retained for comparison but not assumed to be suitable.
- English-to-Indic 1B: a candidate for English-to-Santali, subject to language support.
- Indic-to-Indic 1B: a candidate for Hindi-to-Bhili, Gondi, and Mundari, subject to language support.
- Distilled English-to-Indic and Indic-to-Indic checkpoints: smaller alternatives for feasibility and performance comparisons.

Before selecting a checkpoint, we will inspect its configuration, vocabulary dimensions, architecture, and declared language metadata. A checkpoint's general Indic-language capability does not automatically imply support for every tribal language or script in this dataset.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face authentication restored.")

In [ ]:
# Section 4.3 — Inspect candidate checkpoint configurations

from transformers import AutoConfig

CANDIDATE_MODELS = {
    "Indic -> English 1B": "ai4bharat/indictrans2-indic-en-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic distilled": "ai4bharat/indictrans2-en-indic-dist-200M",
    "Indic -> Indic distilled": "ai4bharat/indictrans2-indic-indic-dist-320M",
}

config_results = {}

for label, model_id in CANDIDATE_MODELS.items():
    print(f"\n{'=' * 70}\n{label}\n{model_id}")

    try:
        config = AutoConfig.from_pretrained(
            model_id,
            trust_remote_code=True,
            token=hf_token,
        )

        config_results[label] = config

        print("Model type:", getattr(config, "model_type", None))
        print("Architecture:", getattr(config, "architectures", None))
        print("Encoder-decoder:", getattr(config, "is_encoder_decoder", None))
        print("Source vocab size:", getattr(config, "vocab_size", None))
        print("Target vocab size:", getattr(config, "tgt_vocab_size", None))
        print("Hidden size:", getattr(config, "d_model", None))
        print("Encoder layers:", getattr(config, "encoder_layers", None))
        print("Decoder layers:", getattr(config, "decoder_layers", None))

        # Print likely language metadata fields if present.
        for key in ("src_lang", "tgt_lang", "languages", "lang_code_to_id"):
            value = getattr(config, key, None)
            if value is not None:
                if isinstance(value, dict) and len(value) > 20:
                    value = f"<mapping with {len(value)} entries>"
                print(f"{key}:", value)

    except Exception as e:
        print("CONFIG INSPECTION FAILED:", type(e).__name__, str(e)[:400])

print("\nConfiguration inspection complete.")

#### 4.3.2 Compare Model Architecture and Vocabulary Configuration

The candidate checkpoints are now accessible. We will compare their configuration fields, including model dimensions, encoder and decoder settings, vocabulary sizes, and parameter-count metadata where available.

This comparison will help us identify the differences between the larger and distilled models. We will also check whether each checkpoint uses separate source and target vocabularies, since that distinction is critical for vocabulary extension.

Configuration metadata is preliminary evidence only. The final decision will also depend on tokenizer behavior, language support, memory requirements, and the ability to fine-tune the model reliably.

In [ ]:
# Section 4.3.2 — Compare available configuration fields

from transformers import AutoConfig

fields_to_compare = [
    "d_model",
    "hidden_size",
    "encoder_embed_dim",
    "decoder_embed_dim",
    "encoder_ffn_dim",
    "decoder_ffn_dim",
    "encoder_layers",
    "decoder_layers",
    "encoder_attention_heads",
    "decoder_attention_heads",
    "vocab_size",
    "tgt_vocab_size",
    "max_position_embeddings",
    "tie_word_embeddings",
]

comparison_rows = []

for label, model_id in CANDIDATE_MODELS.items():
    config = config_results.get(label)

    if config is None:
        try:
            config = AutoConfig.from_pretrained(
                model_id,
                trust_remote_code=True,
                token=hf_token,
            )
            config_results[label] = config
        except Exception as e:
            print(f"Could not inspect {label}: {type(e).__name__}: {e}")
            continue

    row = {
        "candidate": label,
        "model_id": model_id,
    }

    for field in fields_to_compare:
        value = getattr(config, field, None)
        if isinstance(value, (list, dict)):
            value = f"<{type(value).__name__}: {len(value)} items>"
        row[field] = value

    comparison_rows.append(row)

config_comparison = pd.DataFrame(comparison_rows).set_index("candidate")
display(config_comparison.T)

### 4.4 Tokenizer and Language Configuration

Before measuring tokenization fertility, we must identify the tokenizer associated with each candidate checkpoint.

IndicTrans2 uses language-specific source and target processing. Therefore, we must not assume that a single SentencePiece vocabulary or a generic tokenizer interface fully describes both sides of the translation model.

For each accessible checkpoint, we will inspect:

- Tokenizer class and vocabulary size.
- Source and target SentencePiece vocabulary sizes, where available.
- Special-token IDs.
- Language-code handling.
- Whether the tokenizer can be initialized without loading the model weights.

These checks will help establish which tokenizer is appropriate for Hindi-to-tribal and English-to-Santali translation, and what changes vocabulary extension would require.

In [ ]:
# Section 4.4 — Inspect candidate tokenizers

from transformers import AutoTokenizer

tokenizer_results = {}

for label, model_id in CANDIDATE_MODELS.items():
    print(f"\n{'=' * 70}")
    print(label)
    print(model_id)

    try:
        tok = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            token=hf_token,
        )
        tokenizer_results[label] = tok

        print("Tokenizer class:", type(tok).__name__)
        print("len(tokenizer):", len(tok))
        print("Tokenizer vocab_size:", getattr(tok, "vocab_size", None))
        print("Special tokens:", tok.special_tokens_map)

        for attr in ("src_spm", "tgt_spm"):
            spm = getattr(tok, attr, None)
            if spm is not None:
                try:
                    print(f"{attr} piece count:", spm.GetPieceSize())
                except Exception:
                    print(f"{attr}: available, piece count unavailable")
            else:
                print(f"{attr}: not exposed")

        print("Special token IDs:")
        for token_name in (
            "bos_token_id",
            "eos_token_id",
            "pad_token_id",
            "unk_token_id",
        ):
            print(f"  {token_name}:", getattr(tok, token_name, None))

        # Inspect language-related tokenizer attributes without assuming
        # a particular custom-tokenizer implementation.
        for attr in ("lang_code_to_id", "supported_languages", "src_lang", "tgt_lang"):
            value = getattr(tok, attr, None)
            if value is not None:
                if isinstance(value, dict):
                    print(f"{attr}: {len(value)} entries")
                    print("  Sample:", list(value.items())[:8])
                else:
                    print(f"{attr}:", value)

    except Exception as e:
        print("TOKENIZER INSPECTION FAILED:", type(e).__name__, str(e)[:350])

### 4.5 Tokenization Fertility Across Tribal Languages

Tokenization fertility measures how many subword tokens a tokenizer produces for a given amount of text. High fertility can indicate that words are fragmented into many small pieces, potentially making training less efficient.

We will evaluate the source and target SentencePiece models separately on the tribal-language text for Bhili, Gondi, Mundari, and Santali. This is a diagnostic comparison: the source tokenizer applied to a target-language sentence is not necessarily the tokenizer the translation model would use during normal inference.

For each language and tokenizer, we will calculate:

- Average tokens per sentence.
- Average tokens per whitespace-separated word.
- Word split rate: the percentage of whitespace-separated words represented by more than one SentencePiece token.
- Unknown-token rate, where the tokenizer exposes an unknown-token ID.

A fixed random seed and sample size will be used to make the comparisons reproducible. These measurements will help identify vocabulary gaps, but they do not independently establish translation quality.

In [ ]:
# Section 4.5 — Compare source and target SentencePiece fertility

import numpy as np
import pandas as pd

SAMPLE_SIZE = 2000
RANDOM_SEED = 42

fertility_rows = []

for language, (_, target_col) in language_columns.items():
    df = all_data[language]["train"]
    texts = (
        df[target_col]
        .dropna()
        .astype(str)
        .sample(
            n=min(SAMPLE_SIZE, len(df)),
            random_state=RANDOM_SEED,
        )
        .tolist()
    )

    # Both models expose these attributes through IndicTransTokenizer.
    # This compares the SP models diagnostically on the same text.
    for model_label in [
        "Indic -> English 1B",
        "English -> Indic 1B",
        "Indic -> Indic 1B",
        "English -> Indic distilled",
        "Indic -> Indic distilled",
    ]:
        tok = tokenizer_results[model_label]

        for spm_name in ["src_spm", "tgt_spm"]:
            spm = getattr(tok, spm_name, None)
            if spm is None:
                continue

            sentence_token_counts = []
            total_words = 0
            total_word_tokens = 0
            split_words = 0
            unknown_pieces = 0
            total_pieces = 0

            for text in texts:
                pieces = spm.EncodeAsPieces(text)
                sentence_token_counts.append(len(pieces))
                total_pieces += len(pieces)

                words = text.split()
                total_words += len(words)

                for word in words:
                    word_pieces = spm.EncodeAsPieces(word)
                    total_word_tokens += len(word_pieces)

                    if len(word_pieces) > 1:
                        split_words += 1

                    unknown_pieces += sum(
                        piece == "<unk>" for piece in word_pieces
                    )

            fertility_rows.append({
                "language": language,
                "text_column": target_col,
                "model": model_label,
                "spm": spm_name,
                "sentences": len(texts),
                "avg_tokens_per_sentence": round(
                    np.mean(sentence_token_counts), 3
                ),
                "tokens_per_word": round(
                    total_word_tokens / max(total_words, 1), 3
                ),
                "word_split_rate_pct": round(
                    100 * split_words / max(total_words, 1), 3
                ),
                "unknown_piece_rate_pct": round(
                    100 * unknown_pieces / max(total_pieces, 1), 4
                ),
            })

fertility_df = pd.DataFrame(fertility_rows)

display(
    fertility_df.sort_values(
        ["language", "model", "spm"]
    ).reset_index(drop=True)
)

### 4.5.1 Identifying Distinct SentencePiece Vocabularies

The previous fertility table contains repeated measurements across model
checkpoints. However, tokenizer vocabularies cannot be considered identical
merely because they have the same number of pieces.

We will compute a SHA-256 fingerprint from the ordered SentencePiece
vocabulary of each source and target tokenizer. Matching fingerprints
indicate identical ordered piece lists.

This allows us to identify genuinely distinct vocabularies before comparing
tokenization fertility across the four target languages.

In [ ]:
# Section 4.5.1 — Fingerprint actual SentencePiece vocabularies

import hashlib
import pandas as pd

spm_fingerprints = {}

for model_name, tok in tokenizer_results.items():
    for spm_name in ("src_spm", "tgt_spm"):
        spm = getattr(tok, spm_name)
        piece_count = spm.GetPieceSize()

        hasher = hashlib.sha256()

        for piece_id in range(piece_count):
            piece = spm.IdToPiece(piece_id)
            hasher.update(piece.encode("utf-8"))
            hasher.update(b"\n")

        fingerprint = hasher.hexdigest()

        spm_fingerprints[(model_name, spm_name)] = {
            "model": model_name,
            "spm": spm_name,
            "piece_count": piece_count,
            "fingerprint": fingerprint,
        }

fingerprint_df = pd.DataFrame(spm_fingerprints.values())

display(
    fingerprint_df[
        ["model", "spm", "piece_count", "fingerprint"]
    ].sort_values(["piece_count", "model", "spm"]).reset_index(drop=True)
)

print(
    "\nNumber of distinct SentencePiece vocabularies:",
    fingerprint_df["fingerprint"].nunique()
)

print("\nVocabulary groups:")
display(
    fingerprint_df.groupby(
        ["piece_count", "fingerprint"], as_index=False
    ).agg(
        models=("model", lambda x: ", ".join(x)),
        tokenizer_sides=("spm", lambda x: ", ".join(x)),
        occurrences=("model", "size"),
    ).sort_values(["piece_count", "occurrences"], ascending=[True, False])
)

### 4.5.2 Comparing Distinct SentencePiece Vocabularies

The fingerprint analysis identified two distinct SentencePiece vocabularies:
one containing 32,000 pieces and another containing 128,000 pieces.

We will now compare their tokenization behavior on the target-language
sentences. This comparison is diagnostic: lower fertility may indicate
more efficient tokenization, but it does not by itself establish better
translation quality or model compatibility.

In [ ]:
# Section 4.5.2 — Compare the two distinct vocabularies

import pandas as pd
import numpy as np

# Select one representative tokenizer for each actual fingerprint.
unique_fingerprints = fingerprint_df.drop_duplicates(
    subset=["fingerprint"]
)

spm_32k = None
spm_128k = None

for _, row in unique_fingerprints.iterrows():
    tok = tokenizer_results[row["model"]]
    spm = getattr(tok, row["spm"])
    piece_count = spm.GetPieceSize()

    if piece_count == 32000:
        spm_32k = spm
    elif piece_count == 128000:
        spm_128k = spm

assert spm_32k is not None, "Could not locate the 32k vocabulary."
assert spm_128k is not None, "Could not locate the 128k vocabulary."

rows = []

for language, (_, target_col) in language_columns.items():
    texts = (
        all_data[language]["train"][target_col]
        .dropna()
        .astype(str)
        .head(2000)
        .tolist()
    )

    for vocab_size, spm in [
        (32000, spm_32k),
        (128000, spm_128k),
    ]:
        total_tokens = 0
        total_words = 0
        split_words = 0

        for text in texts:
            total_tokens += len(spm.EncodeAsPieces(text))

            for word in text.split():
                total_words += 1
                if len(spm.EncodeAsPieces(word)) > 1:
                    split_words += 1

        rows.append({
            "language": language,
            "vocabulary_size": vocab_size,
            "sentences": len(texts),
            "avg_tokens_per_sentence": round(
                total_tokens / max(len(texts), 1), 3
            ),
            "tokens_per_word": round(
                total_tokens / max(total_words, 1), 3
            ),
            "word_split_rate_pct": round(
                100 * split_words / max(total_words, 1), 3
            ),
        })

comparison_df = pd.DataFrame(rows)

display(
    comparison_df.sort_values(
        ["language", "vocabulary_size"]
    ).reset_index(drop=True)
)

In [ ]:
# Section 4.6 — Check existing tokenizer objects and language-code support

print("Candidate tokenizers available:", len(tokenizer_results))
print("Candidate checkpoints:", list(tokenizer_results.keys()))

print("\nLanguage-related tokenizer attributes:")
for label, tok in tokenizer_results.items():
    print(f"\n{label}")

    for attr in dir(tok):
        if any(term in attr.lower() for term in ("lang_code", "supported_languages")):
            try:
                value = getattr(tok, attr)
                if not callable(value):
                    print(f"  {attr}: {str(value)[:300]}")
            except Exception:
                pass

print("\nProcessor class:", type(processor).__name__)

In [ ]:
# Section 4.6 — Inspect the compiled IndicProcessor API

import inspect

print("Processor methods and signatures:\n")

for name in dir(processor):
    if name.startswith("_"):
        continue

    try:
        obj = getattr(processor, name)
        if callable(obj):
            try:
                signature = inspect.signature(obj)
            except (TypeError, ValueError):
                signature = "(signature unavailable)"

            print(f"{name}{signature}")
    except Exception as e:
        print(f"{name}: inspection failed ({type(e).__name__})")

In [ ]:
# Section 4.6 — Search installed toolkit for language-code definitions

from pathlib import Path
import IndicTransToolkit

package_dir = Path(IndicTransToolkit.__file__).parent

print("Toolkit directory:", package_dir)

keywords = ("bhili", "bhili", "gondi", "mundari", "santali",
            "hin_deva", "eng_latn", "language_codes")

matches = []

for path in package_dir.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".py", ".json", ".txt", ".yaml", ".yml"}:
        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
            found = [word for word in keywords if word in content.lower()]
            if found:
                matches.append((str(path.relative_to(package_dir)), found))
        except OSError:
            pass

if matches:
    for filename, found in matches:
        print(f"{filename}: {', '.join(sorted(set(found)))}")
else:
    print("No matching text files found in the installed toolkit.")

In [ ]:
# Section 4.6 — Inspect checkpoint language metadata

for name, config in config_results.items():
    print(f"\n{'=' * 60}")
    print(name)

    if isinstance(config, dict):
        for key in ("model_name", "name_or_path", "architectures",
                    "model_type", "language_codes", "langs",
                    "lang_code_to_id", "id_to_lang_code"):
            if key in config:
                print(f"{key}: {config[key]}")
    else:
        print("Configuration object:", type(config).__name__)
        for key in ("name_or_path", "architectures", "model_type",
                    "language_codes", "langs",
                    "lang_code_to_id", "id_to_lang_code"):
            value = getattr(config, key, None)
            if value is not None:
                print(f"{key}: {value}")

In [ ]:
# Section 4.6 — Inspect tokenizer API for language handling

for name, tok in tokenizer_results.items():
    print(f"\n{'=' * 60}")
    print(name)
    print("Class:", type(tok).__name__)

    for attr in dir(tok):
        if any(term in attr.lower() for term in (
            "lang", "language", "src_spm", "tgt_spm",
            "preprocess", "special"
        )):
            try:
                value = getattr(tok, attr)
                if not callable(value):
                    print(f"{attr}: {str(value)[:200]}")
            except Exception:
                pass

In [ ]:
# Section 4.6 — Prepare one example per translation task
# This cell only inspects the existing data; it does not train or load models.

for language, (src_col, tgt_col) in language_columns.items():
    example = all_data[language]["validation"].iloc[0]

    print(f"\n{'=' * 55}")
    print(f"Task: {src_col} -> {tgt_col}")
    print("Language key:", language)
    print("Source column:", src_col)
    print("Target column:", tgt_col)
    print("Source example:", str(example[src_col])[:250])
    print("Target example:", str(example[tgt_col])[:250])

# Section 4.6 results and analysis — Language and Model Compatibility Analysis

## 1. Research Objective

Our research investigates whether vocabulary extension and informed embedding initialization can improve neural machine translation for four low-resource Indian tribal languages: Bhili, Gondi, Mundari, and Santali.

We will compare two approaches:

1. **Baseline:** Fine-tune a pretrained IndicTrans2 model using LoRA while retaining its original vocabulary.
2. **Proposed method:** Fine-tune the same model using LoRA after extending its vocabulary and applying a selected embedding-initialization strategy.

For a fair comparison, both approaches must use the same checkpoint, training data, data splits, evaluation procedure, and comparable training settings.

## 2. Translation Tasks and Dataset Examples

The prepared dataset contains four translation tasks:

| Language | Translation direction | Source language | Target language |
|---|---|---|---|
| Bhili | Hindi → Bhili | Hindi | Bhili |
| Gondi | Hindi → Gondi | Hindi | Gondi |
| Mundari | Hindi → Mundari | Hindi | Mundari |
| Santali | English → Santali | English | Santali |

The validation examples inspected in the notebook confirm that the expected source and target columns are available.

The Santali target example uses the **Ol Chiki script**. The Bhili, Gondi, and Mundari examples use their respective dataset representations, which include Devanagari-script text.

These examples confirm the dataset structure, but do not establish translation quality or pretrained-model compatibility.

## 3. Findings from the Official Language Information

The IndicTrans2 language information examined during this investigation lists Santali with the language code `sat_Olck`, corresponding to the Ol Chiki script.

Bhili, Gondi, and Mundari were not found in the supported-language lists examined for the candidate IndicTrans2 checkpoints.

This leads to the following distinction:

- **Santali:** A listed language, with a documented script-specific language code. We still need to verify the complete inference pipeline for the selected checkpoint.
- **Bhili, Gondi, and Mundari:** Languages for which we must investigate adaptation beyond the officially listed language coverage. We must not assume that inventing or assigning language codes will make these languages supported.

The distinction is important because tokenizer compatibility, language-code acceptance, and actual translation capability are separate properties.

## 4. Candidate Checkpoints

Five IndicTrans2 checkpoints were inspected.

| Checkpoint | Approximate model size | Intended translation direction |
|---|---:|---|
| `ai4bharat/indictrans2-indic-en-1B` | 1B | Indic → English |
| `ai4bharat/indictrans2-en-indic-1B` | 1B | English → Indic |
| `ai4bharat/indictrans2-indic-indic-1B` | 1B | Indic → Indic |
| `ai4bharat/indictrans2-en-indic-dist-200M` | 200M | English → Indic |
| `ai4bharat/indictrans2-indic-indic-dist-320M` | 320M | Indic → Indic |

All five inspected configurations use the `IndicTransForConditionalGeneration` architecture and have an encoder-decoder structure.

The 1B checkpoints use 1024-dimensional encoder and decoder embeddings, 8192-dimensional feed-forward layers, 18 encoder layers, 18 decoder layers, and 16 attention heads.

The distilled checkpoints use 512-dimensional embeddings, 2048-dimensional feed-forward layers, 18 encoder layers, 18 decoder layers, and 8 attention heads.

These differences make model size a relevant experimental variable. A larger checkpoint may offer greater capacity, but its

In [ ]:
# Section 4.6 — Smoke-test preprocessing for supported-language examples
# This checks text preparation only, not translation quality.

preprocessing_tests = [
    {
        "task": "English -> Santali",
        "src_lang": "eng_Latn",
        "tgt_lang": "sat_Olck",
        "source": all_data["santali"]["validation"].iloc[0]["English"],
        "target": all_data["santali"]["validation"].iloc[0]["Santali"],
    },
    {
        "task": "Hindi source preprocessing",
        "src_lang": "hin_Deva",
        "tgt_lang": None,
        "source": all_data["bhili"]["validation"].iloc[0]["Hindi"],
        "target": None,
    },
]

for test in preprocessing_tests:
    print(f"\n{'=' * 55}")
    print("Test:", test["task"])

    try:
        processed = processor.preprocess_batch(
            [test["source"]],
            src_lang=test["src_lang"],
            tgt_lang=test["tgt_lang"],
        )
        print("Preprocessing succeeded.")
        print("Result:", processed[0][:300])
    except Exception as exc:
        print("Preprocessing failed:", type(exc).__name__, str(exc)[:500])

In [ ]:
# Section 4.6 — Santali tokenizer round-trip check
# Reuses the tokenizer already loaded in tokenizer_results.

tok = tokenizer_results["English -> Indic 1B"]

row = all_data["santali"]["validation"].iloc[0]
source_text = row["English"]
target_text = row["Santali"]

# Encode the source using the English source SentencePiece model
source_pieces = tok.src_spm.encode(source_text, out_type=str)

# Encode the Santali target using the target SentencePiece model
target_pieces = tok.tgt_spm.encode(target_text, out_type=str)

# Decode each sequence back to text
source_roundtrip = tok.src_spm.decode(source_pieces)
target_roundtrip = tok.tgt_spm.decode(target_pieces)

print("Source SentencePiece pieces:", len(source_pieces))
print("Target SentencePiece pieces:", len(target_pieces))

print("\nOriginal Santali target:")
print(target_text)

print("\nDecoded Santali target:")
print(target_roundtrip)

print("\nSantali round-trip exact match:", target_text == target_roundtrip)
print("Source round-trip exact match:", source_text == source_roundtrip)

In [ ]:
# Section 4.6 — Round-trip and tokenization check for all target languages

tokenizer_for_task = {
    "bhili": "Indic -> Indic 1B",
    "gondi": "Indic -> Indic 1B",
    "mundari": "Indic -> Indic 1B",
    "santali": "English -> Indic 1B",
}

for language, tokenizer_name in tokenizer_for_task.items():
    tok = tokenizer_results[tokenizer_name]
    _, target_col = language_columns[language]
    df = all_data[language]["validation"].head(100)

    token_counts = []
    exact_matches = 0

    for text in df[target_col].astype(str):
        pieces = tok.tgt_spm.encode(text, out_type=str)
        decoded = tok.tgt_spm.decode(pieces)

        token_counts.append(len(pieces))
        exact_matches += (decoded == text)

    print(f"\n{'=' * 55}")
    print(f"Language: {language}")
    print(f"Tokenizer: {tokenizer_name}")
    print(f"Examples checked: {len(df)}")
    print(f"Mean target SentencePiece tokens: {sum(token_counts) / len(token_counts):.2f}")
    print(f"Exact round trips: {exact_matches}/{len(df)}")

In [ ]:
# Section 4.6 — Inspect tokenizer round-trip failures

tokenizer_for_task = {
    "bhili": "Indic -> Indic 1B",
    "gondi": "Indic -> Indic 1B",
    "mundari": "Indic -> Indic 1B",
    "santali": "English -> Indic 1B",
}

for language, tokenizer_name in tokenizer_for_task.items():
    tok = tokenizer_results[tokenizer_name]
    _, target_col = language_columns[language]
    df = all_data[language]["validation"].head(100)

    failures = []

    for text in df[target_col].astype(str):
        pieces = tok.tgt_spm.encode(text, out_type=str)
        decoded = tok.tgt_spm.decode(pieces)

        if decoded != text:
            failures.append({
                "original": text,
                "decoded": decoded,
                "original_repr": repr(text),
                "decoded_repr": repr(decoded),
                "original_length": len(text),
                "decoded_length": len(decoded),
                "pieces": pieces,
            })

    print(f"\n{'=' * 70}")
    print(f"Language: {language} | Failures: {len(failures)}/100")

    for i, item in enumerate(failures[:3], start=1):
        print(f"\nFailure {i}")
        print("Original:     ", item["original"][:250])
        print("Decoded:      ", item["decoded"][:250])
        print("Original repr:", item["original_repr"][:300])
        print("Decoded repr: ", item["decoded_repr"][:300])
        print(
            "Lengths:      ",
            item["original_length"],
            "->",
            item["decoded_length"],
        )
        print("Pieces:       ", item["pieces"][:40])

In [ ]:
import unicodedata
import re

tokenizer_for_task = {
    "bhili": "Indic -> Indic 1B",
    "gondi": "Indic -> Indic 1B",
    "mundari": "Indic -> Indic 1B",
    "santali": "English -> Indic 1B",
}

def normalize_nfc(text):
    return unicodedata.normalize("NFC", text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

for language, tokenizer_name in tokenizer_for_task.items():
    tok = tokenizer_results[tokenizer_name]
    _, target_col = language_columns[language]
    df = all_data[language]["validation"].head(100)

    exact = nfc_only = whitespace_only = both = unresolved = 0

    for original in df[target_col].astype(str):
        decoded = tok.tgt_spm.decode(
            tok.tgt_spm.encode(original, out_type=str)
        )

        if decoded == original:
            exact += 1
        elif normalize_nfc(decoded) == normalize_nfc(original):
            nfc_only += 1
        elif normalize_whitespace(decoded) == normalize_whitespace(original):
            whitespace_only += 1
        elif normalize_whitespace(normalize_nfc(decoded)) == normalize_whitespace(
            normalize_nfc(original)
        ):
            both += 1
        else:
            unresolved += 1

    print(f"\n{language.upper()}")
    print(f"Exact matches:                 {exact}")
    print(f"NFC resolves mismatch:         {nfc_only}")
    print(f"Whitespace resolves mismatch:  {whitespace_only}")
    print(f"Both NFC + whitespace resolve: {both}")
    print(f"Still different:               {unresolved}")

## Section 4.6 — Tokenizer Compatibility and Round-Trip Analysis

### 1. Objective

Before beginning model fine-tuning and vocabulary extension, we investigated the pretrained IndicTrans2 tokenizers to understand their vocabulary structure, tokenization behavior on the four target languages, and ability to preserve text through SentencePiece encoding and decoding.

The target languages are Bhili, Gondi, Mundari, and Santali. The translation directions are Hindi → Bhili, Hindi → Gondi, Hindi → Mundari, and English → Santali.

### 2. Tokenizer and Vocabulary Findings

All five candidate checkpoints use the `IndicTransTokenizer` class. Each tokenizer exposes separate source and target SentencePiece models.

| Candidate checkpoint | Source vocabulary | Target vocabulary |
|---|---:|---:|
| Indic → English 1B | 128k | 32k |
| English → Indic 1B | 32k | 128k |
| Indic → Indic 1B | 128k | 128k |
| English → Indic distilled 200M | 32k | 128k |
| Indic → Indic distilled 320M | 128k | 128k |

The 32k and 128k SentencePiece vocabularies have different fingerprints, confirming that they are distinct vocabulary configurations.

The tokenizer special-token IDs are consistent across the inspected checkpoints: BOS = 0, PAD = 1, EOS = 2, and UNK = 3.

### 3. Tokenization Fertility Analysis

A diagnostic sample of 2,000 target sentences per language was used to compare tokenization with the 32k and 128k SentencePiece vocabularies.

| Language | 32k tokens/sentence | 128k tokens/sentence | 32k tokens/word | 128k tokens/word |
|---|---:|---:|---:|---:|
| Bhili | 105.506 | 30.831 | 4.957 | 1.448 |
| Gondi | 74.555 | 23.679 | 5.532 | 1.757 |
| Mundari | 78.115 | 30.438 | 5.530 | 2.155 |
| Santali | 39.544 | 28.102 | 2.029 | 1.442 |

The 128k vocabulary produced fewer tokens per sentence and per word for all four target languages in this sample. The measured split-token rates also decreased:

| Language | 32k split-token rate | 128k split-token rate |
|---|---:|---:|
| Bhili | 98.264% | 32.350% |
| Gondi | 99.547% | 48.418% |
| Mundari | 97.795% | 62.959% |
| Santali | 99.574% | 27.239% |

These results indicate differences in tokenization efficiency. They do **not** establish which checkpoint will produce better translations, and they do not by themselves prove that vocabulary extension will improve translation quality.

### 4. Language Support and Preprocessing

The inspected IndicTrans2 model-card language lists include Santali with the code `sat_Olck`, referring to Santali in the Ol Chiki script. Bhili, Gondi, and Mundari were not listed among the officially supported languages in the materials examined.

The installed `IndicProcessor` accepts parameters for source and target language codes and can preprocess an English-to-Santali example. The output includes the language-code prefix:

`eng_Latn sat_Olck`

A Hindi-source preprocessing example also succeeded with the `hin_Deva` source code.

The processor implementation is compiled, so direct source inspection using Python's `inspect.getsource()` was not available. The observed preprocessing behavior establishes that the tested calls work; it does not establish that unsupported languages are natively supported by the pretrained model.

### 5. SentencePiece Round-Trip Experiment

For each language, 100 validation targets were encoded using the task-specific checkpoint's target SentencePiece model and then decoded. The decoded strings were compared with the originals using exact string equality.

| Language | Exact matches | Exact-match rate | Initial mismatches |
|---|---:|---:|---:|
| Bhili | 84/100 | 84% | 16 |
| Gondi | 80/100 | 80% | 20 |
| Mundari | 53/100 | 53% | 47 |
| Santali | 97/100 | 97% | 3 |

The initial results showed substantial differences in exact round-trip preservation, especially for Mundari.

### 6. Investigation of Round-Trip Mismatches

Sample inspection revealed several types of differences:

- **Unicode normalization:** Characters that look similar may have different Unicode representations. For example, `अगाड़ी` may decode as `अगाड़ी`.
- **Whitespace normalization:** Repeated spaces may be collapsed into a single space.
- **Zero-width characters:** Some original strings contain invisible characters that may not survive the round trip.
- **Other differences:** A small number of examples remain different even after the normalization checks.

To investigate further, the original and decoded strings were compared using NFC Unicode normalization and whitespace normalization. Each example was assigned to the first category that resolved its mismatch.

| Language | Exact | NFC resolves | Whitespace resolves | Both resolve | Still different |
|---|---:|---:|---:|---:|---:|
| Bhili | 84 | 4 | 8 | 2 | 2 |
| Gondi | 80 | 19 | 1 | 0 | 0 |
| Mundari | 53 | 39 | 4 | 3 | 1 |
| Santali | 97 | 0 | 3 | 0 | 0 |

For this diagnostic sample:

- All 20 Gondi mismatches were resolved by NFC or whitespace normalization.
- 46 of 47 Mundari mismatches were resolved by the tested normalization checks.
- 14 of 16 Bhili mismatches were resolved.
- All three Santali mismatches were resolved by whitespace normalization.

These checks are diagnostic approximations. Because the categories are evaluated in order, an example resolved by NFC is not subsequently counted as a whitespace-resolved example. The remaining differences require further investigation.

### 7. Interpretation and Implications

The round-trip results show that exact string equality alone can underestimate text preservation when a tokenizer normalizes Unicode or whitespace. Consequently, the initial exact-match rates should not be interpreted as direct measures of vocabulary coverage or translation quality.

For subsequent experiments:

1. Preserve the original train, validation, and test data.
2. Do not silently normalize the corpus solely to improve round-trip scores.
3. Document the text-normalization behavior of each tokenizer.
4. Investigate the remaining unexplained mismatches before drawing conclusions about information loss.
5. Measure vocabulary-extension benefits using held-out translation quality and tokenization efficiency, rather than round-trip equality alone.

### 8. Provisional Conclusion

The tokenizer investigation establishes three useful findings. First, the inspected checkpoints use distinct 32k and 128k SentencePiece vocabularies. Second, the 128k vocabulary produces fewer tokens per word in the sampled target texts. Third, many observed round-trip mismatches are explained by Unicode or whitespace normalization.

These findings motivate further investigation of vocabulary-aware adaptation. However, the language-conditioning setup for Bhili, Gondi, and Mundari still needs to be established before selecting the baseline training procedure. No conclusion about translation quality or the effectiveness of vocabulary extension can yet be drawn from these tokenizer diagnostics.

In [ ]:
# Section 4.7 — Inspect language-code preprocessing behavior

from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

source_example = ["This is a short translation test."]

tests = [
    ("Known English -> Santali", "eng_Latn", "sat_Olck"),
    ("Unknown target-code probe", "eng_Latn", "xyz_Test"),
]

for label, src_lang, tgt_lang in tests:
    print(f"\n{'=' * 60}")
    print(label)
    print(f"Source code: {src_lang} | Target code: {tgt_lang}")

    try:
        result = ip.preprocess_batch(
            source_example,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
        )
        print("Preprocessing accepted.")
        print("Output:", result)
    except Exception as e:
        print("Preprocessing rejected.")
        print("Error type:", type(e).__name__)
        print("Error:", str(e)[:500])

## 5. Language Conditioning and Model Compatibility

### Objective

Before training the baseline LoRA model, we must establish how the pretrained IndicTrans2 model receives source- and target-language information and whether its existing language-conditioning mechanism is appropriate for our four translation tasks.

Our tasks are:

- Hindi → Bhili
- Hindi → Gondi
- Hindi → Mundari
- English → Santali

The previous experiments established that `IndicProcessor` accepts both known and arbitrary language-code strings. However, accepting a code during preprocessing does not demonstrate that the pretrained model supports the corresponding language.

We will therefore inspect the model's actual tokenization, embedding, and generation interfaces before choosing a conditioning strategy.

### Research principle

The baseline and vocabulary-extended experiments must use the same language-conditioning convention. Otherwise, any observed translation-quality difference could result from changing the conditioning strategy rather than extending the vocabulary.

We will preserve the existing dataset splits and original text. Any additional language codes or conditioning conventions must be documented explicitly and applied consistently during training, validation, and inference.

### Expected outcome

Identify a defensible conditioning strategy for each translation task, document its limitations, and verify that the model can process the resulting inputs before starting baseline LoRA training.

In [ ]:

# Section 5.2: Inspect language-conditioning interfaces
# This is a diagnostic only; no model training or dataset modification.

import inspect
from transformers import AutoConfig, AutoTokenizer

MODEL_TO_INSPECT = "ai4bharat/indictrans2-indic-indic-1B"

print("Model:", MODEL_TO_INSPECT)

# Inspect the configuration without loading the model weights.
config = AutoConfig.from_pretrained(
    MODEL_TO_INSPECT,
    trust_remote_code=True,
    token=hf_token
)

print("\n--- Model configuration ---")
for key in [
    "model_type",
    "architectures",
    "vocab_size",
    "decoder_start_token_id",
    "bos_token_id",
    "eos_token_id",
    "pad_token_id",
    "tie_word_embeddings",
]:
    print(f"{key}: {getattr(config, key, None)}")

# Reuse the existing tokenizer result if it is already loaded.
# Otherwise, load the tokenizer for this checkpoint.
print("\n--- Tokenizer class and special tokens ---")
tok = AutoTokenizer.from_pretrained(
    MODEL_TO_INSPECT,
    trust_remote_code=True,
    token=hf_token
)

print("Tokenizer class:", type(tok).__name__)
print("Tokenizer length:", len(tok))
print("Special tokens map:", getattr(tok, "special_tokens_map", None))
print("Added vocabulary size:", len(tok.get_added_vocab()))

# Check the model-facing tokenizer methods and available language-related fields.
print("\n--- Tokenizer attributes related to language ---")
for attr in dir(tok):
    if any(term in attr.lower() for term in [
        "lang", "src", "tgt", "prefix", "special", "spm"
    ]):
        try:
            value = getattr(tok, attr)
            if not callable(value):
                print(f"{attr}: {str(value)[:300]}")
        except Exception as exc:
            print(f"{attr}: <could not inspect: {type(exc).__name__}>")

print("\n--- Tokenizer call signature ---")
try:
    print(inspect.signature(tok.__call__))
except Exception as exc:
    print("Could not inspect:", repr(exc))

print("\n--- Tokenizer class methods mentioning language or prefix ---")
for name in dir(type(tok)):
    if any(term in name.lower() for term in ["lang", "prefix", "prepare", "encode"]):
        method = getattr(type(tok), name, None)
        if callable(method):
            try:
                print(f"{name}{inspect.signature(method)}")
            except Exception:
                print(name)


In [ ]:

# Section 5.3: Inspect the language-tag extension method
# Diagnostic only — no vocabulary or model weights are modified.

import inspect

tok_class = type(tok)

print("Tokenizer class:", tok_class)
print("\n--- add_new_language_tags source ---")

try:
    print(inspect.getsource(tok_class.add_new_language_tags))
except (OSError, TypeError) as exc:
    print("Source unavailable:", repr(exc))

print("\n--- Relevant tokenizer attributes ---")
for attr in [
    "src_vocab_size",
    "tgt_vocab_size",
    "src_encoder",
    "tgt_encoder",
    "src_decoder",
    "tgt_decoder",
    "src_spm",
    "tgt_spm",
]:
    value = getattr(tok, attr, None)

    if isinstance(value, dict):
        print(f"{attr}: dictionary with {len(value)} entries")
    elif value is not None:
        print(f"{attr}: {type(value).__name__} = {value}")
    else:
        print(f"{attr}: not available")

print("\n--- Language-tag examples already in the vocabularies ---")
for tag in [
    "eng_Latn",
    "hin_Deva",
    "sat_Olck",
    "bhil",
    "gon",
    "unrecognized_Test",
]:
    print(
        f"{tag!r}: "
        f"source_id={tok.src_encoder.get(tag)}, "
        f"target_id={tok.tgt_encoder.get(tag)}"
    )


### 5.5. Pilot Selection and Experimental Scope

We will first investigate two representative cases:

1. English → Santali, using the supported `eng_Latn` and `sat_Olck` language tags.
2. Hindi → Bhili, using `hin_Deva` and the documented `mar_Deva` proxy.

These pilots serve different purposes: Santali provides a supported-target control, while Bhili tests a proxy-based adaptation strategy.

Successful preprocessing or tokenization will not be treated as proof of translation capability. We will verify the actual model inputs and target-label construction before training.

After validating the baseline pipeline, we will extend the experiments to Gondi and Mundari. Proxy selection based on linguistic similarity and transfer learning will be treated as a separate, controlled investigation.

The primary vocabulary-extension experiment will keep the language-conditioning strategy fixed between baseline and modified models.

In [ ]:

# Section 5.6: Inspect the two pilot input formats
# This only preprocesses sample sentences; it does not train a model.

from IndicTransToolkit import IndicProcessor

pilot_processor = IndicProcessor(inference=True)

pilot_configs = {
    "Santali": {
        "source_lang": "eng_Latn",
        "target_lang": "sat_Olck",
        "data_key": "santali",
        "source_column": "English",
        "target_column": "Santali",
    },
    "Bhili": {
        "source_lang": "hin_Deva",
        "target_lang": "mar_Deva",
        "data_key": "bhili",
        "source_column": "Hindi",
        "target_column": "Bhili",
    },
}

for name, cfg in pilot_configs.items():
    print(f"\n{'=' * 20} {name} {'=' * 20}")

    row = all_data[cfg["data_key"]]["train"].iloc[0]
    source_text = str(row[cfg["source_column"]])
    target_text = str(row[cfg["target_column"]])

    processed_source = pilot_processor.preprocess_batch(
        [source_text],
        src_lang=cfg["source_lang"],
        tgt_lang=cfg["target_lang"],
    )

    print("Source language tag:", cfg["source_lang"])
    print("Target/proxy tag:", cfg["target_lang"])
    print("Original source:", repr(source_text))
    print("Original target:", repr(target_text))
    print("Processed source:", repr(processed_source[0]))


In [ ]:

# Section 5.7: Inspect source prefixes and target labels
# Diagnostic only — no model training or tokenizer modification.

from transformers import AutoTokenizer

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"

check_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=hf_token
)

for name, cfg in pilot_configs.items():
    print(f"\n{'=' * 18} {name} {'=' * 18}")

    row = all_data[cfg["data_key"]]["train"].iloc[0]

    source_text = str(row[cfg["source_column"]])
    target_text = str(row[cfg["target_column"]])

    processed_source = pilot_processor.preprocess_batch(
        [source_text],
        src_lang=cfg["source_lang"],
        tgt_lang=cfg["target_lang"],
    )[0]

    # Encode the processed source and the unmodified target independently.
    source_ids = check_tokenizer(
        processed_source,
        add_special_tokens=True
    )["input_ids"]

    target_ids = check_tokenizer(
        text_target=target_text,
        add_special_tokens=True
    )["input_ids"]

    print("Source prefix:", processed_source.split()[:2])
    print("Source first token IDs:", source_ids[:12])
    print("Source first tokens:", check_tokenizer.convert_ids_to_tokens(source_ids[:12]))

    print("\nTarget original text:", repr(target_text[:160]))
    print("Target first token IDs:", target_ids[:12])
    print("Target first tokens:", check_tokenizer.convert_ids_to_tokens(target_ids[:12]))

    print("\nSource ends with EOS:", source_ids[-1] == check_tokenizer.eos_token_id)
    print("Target ends with EOS:", target_ids[-1] == check_tokenizer.eos_token_id)


In [ ]:

# Section 5.8: Compare target-side tokenization across checkpoints
# Diagnostic only — does not train or modify model weights.

from transformers import AutoTokenizer

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

# Use the actual first Santali and Bhili training examples.
samples = {
    "Santali": str(all_data["santali"]["train"].iloc[0]["Santali"]),
    "Bhili": str(all_data["bhili"]["train"].iloc[0]["Bhili"]),
}

for model_label, model_id in TOKENIZER_MODELS.items():
    print(f"\n{'=' * 18} {model_label} {'=' * 18}")

    current_tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    print("Tokenizer:", type(current_tok).__name__)

    for language, text in samples.items():
        ids = current_tok(
            text_target=text,
            add_special_tokens=True,
        )["input_ids"]

        pieces = current_tok.convert_ids_to_tokens(ids[:20])
        decoded = current_tok.decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        print(f"\n{language}")
        print("Original:", repr(text[:140]))
        print("Token pieces:", pieces)
        print("Decoded:", repr(decoded[:180]))
        print("Exact round trip:", decoded == text)


In [ ]:

# Section 5.9: Measure target tokenization quality
# No model weights are loaded or changed.

from transformers import AutoTokenizer
import numpy as np

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

TARGET_COLUMNS = {
    "bhili": "Bhili",
    "gondi": "Gondi",
    "mundari": "Mundari",
    "santali": "Santali",
}

results = []

for model_label, model_id in TOKENIZER_MODELS.items():
    print(f"\nLoading {model_label}...")
    current_tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    unk_id = current_tok.unk_token_id

    for language, target_col in TARGET_COLUMNS.items():
        texts = (
            all_data[language]["validation"][target_col]
            .dropna()
            .astype(str)
            .head(100)
            .tolist()
        )

        unk_count = 0
        total_tokens = 0
        affected_sentences = 0
        roundtrip_exact = 0

        for text in texts:
            ids = current_tok(
                text_target=text,
                add_special_tokens=False,
            )["input_ids"]

            sentence_unk_count = sum(token_id == unk_id for token_id in ids)
            unk_count += sentence_unk_count
            total_tokens += len(ids)

            if sentence_unk_count:
                affected_sentences += 1

            decoded = current_tok.decode(
                ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )

            if decoded == text:
                roundtrip_exact += 1

        result = {
            "model": model_label,
            "language": language,
            "examples": len(texts),
            "unknown_tokens": unk_count,
            "total_tokens": total_tokens,
            "unknown_token_rate_pct": (
                100 * unk_count / total_tokens if total_tokens else 0
            ),
            "sentences_with_unknown_pct": (
                100 * affected_sentences / len(texts) if texts else 0
            ),
            "exact_roundtrip_pct": (
                100 * roundtrip_exact / len(texts) if texts else 0
            ),
        }
        results.append(result)

        print(
            f"{language:8} | "
            f"UNK={unk_count:6} | "
            f"tokens={total_tokens:7} | "
            f"UNK rate={result['unknown_token_rate_pct']:7.3f}% | "
            f"sentences affected={result['sentences_with_unknown_pct']:6.1f}% | "
            f"exact round-trip={result['exact_roundtrip_pct']:6.1f}%"
        )

import pandas as pd
tokenization_report = pd.DataFrame(results)
display(tokenization_report)


In [ ]:
from transformers import AutoTokenizer

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

for label, model_id in TOKENIZER_MODELS.items():
    print(f"\n{'=' * 65}")
    print(label)

    tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token
    )

    print("Tokenizer class:", type(tok).__name__)
    print("Tokenizer length:", len(tok))
    print("UNK token ID:", tok.unk_token_id)

    for attr in [
        "src_vocab_size",
        "tgt_vocab_size",
        "src_encoder",
        "tgt_encoder",
        "src_spm",
        "tgt_spm",
    ]:
        obj = getattr(tok, attr, None)

        if obj is None:
            print(f"{attr}: unavailable")
        elif isinstance(obj, dict):
            print(f"{attr}: {len(obj)} entries")
        elif isinstance(obj, (int, str)):
            print(f"{attr}: {obj}")
        else:
            print(f"{attr}: {type(obj).__name__}")

    print("Target tokenization example:")
    example = all_data["santali"]["validation"]["Santali"].iloc[0]
    encoded = tok(text_target=str(example), add_special_tokens=False)
    ids = encoded["input_ids"]

    print("Target text:", repr(str(example)[:150]))
    print("First 20 IDs:", ids[:20])
    print("First 20 token strings:", tok.convert_ids_to_tokens(ids[:20]))
    print("UNK IDs in target encoding:", sum(i == tok.unk_token_id for i in ids))

In [ ]:
from transformers import AutoTokenizer

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

example = str(
    all_data["santali"]["validation"]["Santali"].iloc[0]
)

for label, model_id in TOKENIZER_MODELS.items():
    tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    print(f"\n{'=' * 60}")
    print(label)

    # Inspect the target SentencePiece model directly.
    sp = tok.tgt_spm
    pieces = sp.encode(example, out_type=str)
    ids = sp.encode(example, out_type=int)
    decoded = sp.decode(ids)

    print("Target SentencePiece vocabulary:", sp.get_piece_size())
    print("Target piece count:", len(pieces))
    print("Target UNK piece count:", sum(p == "<unk>" for p in pieces))
    print("Target UNK ID count:", sum(i == sp.unk_id() for i in ids))
    print("First 20 target pieces:", pieces[:20])
    print("SentencePiece round-trip exact:", decoded == example)
    print("SentencePiece round-trip NFC:",
          __import__("unicodedata").normalize("NFC", decoded)
          == __import__("unicodedata").normalize("NFC", example))


## 5.10 Model Architecture and Embedding Compatibility Verification

### Objective

Before beginning baseline LoRA training or vocabulary extension, we need to understand how the pretrained IndicTrans2 model represents source and target tokens internally.

The tokenizer inspection in Section 5.9 established that the model has separate source and target vocabularies. For the Indic-to-Indic 1B checkpoint, the tokenizer reports 122,706 source vocabulary entries and 122,672 target vocabulary entries, while the model configuration reports a vocabulary size of 122,672.

These differences make it important to inspect the actual model architecture instead of assuming that the configuration's `vocab_size` describes every embedding matrix.

### What We Will Investigate

1. **Model configuration:** Identify the model architecture, configured vocabulary size, and weight-tying setting.
2. **Encoder embeddings:** Determine the number of embedding rows available for source token IDs.
3. **Decoder embeddings:** Determine the number of embedding rows available for target token IDs.
4. **Output embeddings:** Check the dimensions of the output projection and its compatibility with target token IDs.
5. **Vocabulary compatibility:** Verify that the token IDs produced by the tokenizer can be represented by the corresponding model components.

### Why This Matters for Our Research

Our primary experiment compares the original tokenizer and vocabulary against a vocabulary-extended configuration. To make this comparison meaningful, both experimental settings must preserve a valid mapping between token IDs and model parameters.

If vocabulary extension is implemented incorrectly, token IDs may exceed embedding dimensions, pretrained representations may become misaligned, or existing model parameters may be unintentionally changed.

Therefore, we will first establish the original model's embedding structure. We will not resize embeddings or modify vocabulary mappings until their compatibility has been verified.

### Expected Outcome

A clear record of the model's encoder, decoder, and output embedding dimensions, together with a verified mapping between source and target token IDs and their corresponding model components.

This will provide the architectural baseline needed before proceeding to LoRA training and the vocabulary-extension experiment.


In [ ]:

import torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("Model architecture:", config.architectures)
print("Model type:", config.model_type)
print("Configured vocabulary size:", config.vocab_size)
print("Tied word embeddings:", config.tie_word_embeddings)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

def print_embedding_shape(label, module):
    if module is None:
        print(f"{label}: unavailable")
        return

    weight = getattr(module, "weight", None)
    print(
        f"{label}:",
        tuple(weight.shape) if weight is not None else "no weight matrix"
    )

print("\n--- Embedding matrices ---")

print_embedding_shape(
    "Input embeddings",
    model.get_input_embeddings()
)

print_embedding_shape(
    "Output embeddings",
    model.get_output_embeddings()
)

for component_name in ("encoder", "decoder"):
    component = getattr(model, component_name, None)

    if component is None and hasattr(model, "model"):
        component = getattr(model.model, component_name, None)

    print(f"\n{component_name.capitalize()} component:")

    if component is not None:
        print_embedding_shape(
            f"{component_name} token embeddings",
            getattr(component, "embed_tokens", None)
        )
    else:
        print("Component not found at the expected attribute path.")

print("\n--- Memory cleanup ---")
del model
torch.cuda.empty_cache()



### 5.10.2 Results and Interpretation

The model architecture and embedding matrices were inspected using the `ai4bharat/indictrans2-indic-indic-1B` checkpoint.

#### Observed Results

| Component | Embedding shape | Vocabulary entries |
|---|---:|---:|
| Encoder input embeddings | (122706, 1024) | 122,706 |
| Decoder token embeddings | (122672, 1024) | 122,672 |
| Output embeddings | (122672, 1024) | 122,672 |

Additional configuration details:

- Model architecture: `IndicTransForConditionalGeneration`
- Model type: `IndicTrans`
- Configured vocabulary size: `122672`
- `tie_word_embeddings`: `True`

#### Key Observations

**1. Source and target embedding dimensions differ in vocabulary size.**

The encoder has 122,706 embedding rows, whereas the decoder has 122,672. Both use an embedding dimension of 1,024.

The encoder therefore accommodates 34 more token IDs than the decoder. This is consistent with the tokenizer's previously observed difference between source and target vocabulary sizes.

**2. The configured vocabulary size does not describe both embedding matrices.**

Although the model configuration reports a vocabulary size of 122,672, the encoder embedding matrix contains 122,706 rows. Therefore, using `config.vocab_size` alone would not accurately describe the encoder's embedding capacity.

**3. Output embeddings match the decoder vocabulary size.**

The output embedding matrix has shape `(122672, 1024)`, matching the decoder token embedding dimensions. This is consistent with the model's configured vocabulary size.

**4. Weight tying requires further verification.**

The configuration reports `tie_word_embeddings=True`. This flag alone does not establish which parameters share weights in the instantiated architecture. The relationship between the decoder embeddings, output projection, and encoder embeddings should be checked directly if it affects the vocabulary-extension implementation.

#### Implications for Vocabulary Extension

These results establish the original embedding dimensions and explain why source and target vocabularies must be handled separately.

For our experiments, vocabulary extension must preserve the correspondence between source token IDs and encoder embeddings, as well as between target token IDs, decoder embeddings, and output logits.

We should not resize the model's embeddings yet. First, we must verify the actual token-ID ranges, special-token mappings, and parameter-sharing behavior. This will help prevent invalid IDs, incorrectly initialized embeddings, or unintended changes to pretrained parameters.

#### Conclusion

The inspected checkpoint has separate encoder and decoder embedding matrices with a common embedding dimension of 1,024 but different vocabulary sizes. This provides the architectural baseline for investigating vocabulary extension. The precise resizing and initialization strategy remains to be determined after the remaining compatibility checks.



### 5.10.3 Verification of Weight Tying and Parameter Sharing

#### Objective

The model configuration reports `tie_word_embeddings=True`. However, this setting alone does not tell us whether the encoder embeddings, decoder embeddings, and output projection actually share the same parameters.

We will inspect the instantiated model to determine:

1. Whether the encoder and decoder embedding matrices share the same underlying parameters.
2. Whether the decoder token embeddings and output projection share weights.
3. Whether the model's parameter shapes and storage relationships agree with the observed vocabulary dimensions.

#### Why This Matters

Vocabulary extension changes the number of token representations available to the model. If parameters are shared, resizing or replacing one embedding matrix may affect other components. If they are separate, each affected component may require its own compatible update.

Understanding these relationships is necessary to design a controlled vocabulary-extension experiment without unintentionally changing the pretrained model's architecture.

#### Method

We will compare parameter object identity, storage pointers, and tensor shapes. These checks help distinguish genuinely shared parameters from matrices that merely have matching dimensions.

No model weights will be modified in this section.


In [ ]:

import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

encoder_embeddings = model.get_input_embeddings()
decoder_embeddings = model.get_decoder().embed_tokens
output_embeddings = model.get_output_embeddings()

def inspect_weight(label, module):
    weight = module.weight
    print(f"{label}:")
    print("  Shape:", tuple(weight.shape))
    print("  Parameter object ID:", id(weight))
    print("  Storage pointer:", weight.data_ptr())
    print("  Storage offset:", weight.storage_offset())

print("=== Embedding parameter inspection ===\n")

inspect_weight("Encoder embeddings", encoder_embeddings)
inspect_weight("Decoder embeddings", decoder_embeddings)
inspect_weight("Output embeddings", output_embeddings)

print("\n=== Parameter-sharing checks ===")

print(
    "Encoder and decoder share parameter object:",
    encoder_embeddings.weight is decoder_embeddings.weight
)

print(
    "Decoder and output share parameter object:",
    decoder_embeddings.weight is output_embeddings.weight
)

print(
    "Encoder and output share parameter object:",
    encoder_embeddings.weight is output_embeddings.weight
)

print("\n=== Shape compatibility ===")

print(
    "Decoder and output shapes match:",
    decoder_embeddings.weight.shape == output_embeddings.weight.shape
)

print(
    "Encoder and decoder shapes match:",
    encoder_embeddings.weight.shape == decoder_embeddings.weight.shape
)

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:

import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print("Configuration tie_word_embeddings:",
      model.config.tie_word_embeddings)

print("\nDeclared tied-weight keys:")
print(getattr(model, "_tied_weights_keys", None))

print("\nDeclared keys to ignore on load:")
print(getattr(model, "_keys_to_ignore_on_load_missing", None))

def report_sharing(stage):
    encoder = model.get_input_embeddings()
    decoder = model.get_decoder().embed_tokens
    output = model.get_output_embeddings()

    print(f"\n--- {stage} ---")
    print("Encoder is decoder:", encoder.weight is decoder.weight)
    print("Decoder is output:", decoder.weight is output.weight)
    print("Encoder is output:", encoder.weight is output.weight)

report_sharing("Before built-in tie_weights()")

print("\nCalling model.tie_weights()...")
model.tie_weights()

report_sharing("After built-in tie_weights()")

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:

import inspect
import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

decoder_weight = model.get_decoder().embed_tokens.weight
output_weight = model.get_output_embeddings().weight

print("=== Parameter comparison ===")
print("Decoder shape:", tuple(decoder_weight.shape))
print("Output shape:", tuple(output_weight.shape))
print("Same object:", decoder_weight is output_weight)
print(
    "Same underlying storage:",
    decoder_weight.untyped_storage().data_ptr()
    == output_weight.untyped_storage().data_ptr()
)
print(
    "Exactly equal values:",
    torch.equal(decoder_weight, output_weight)
)

print("\n=== Model implementation ===")

for method_name in ("tie_weights", "get_output_embeddings"):
    method = getattr(type(model), method_name, None)
    print(f"\n{method_name}:")

    if method is None:
        print("Method not found on the model class.")
        continue

    try:
        print(inspect.getsource(method))
    except (OSError, TypeError) as exc:
        print("Source unavailable:", repr(exc))

print("\n=== Model class ===")
print(type(model))
print("Model source file:", inspect.getfile(type(model)))

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
from transformers import AutoConfig

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("tie_word_embeddings:",
      getattr(config, "tie_word_embeddings", None))

print("share_decoder_input_output_embed:",
      getattr(config, "share_decoder_input_output_embed", None))

print("\nAll config attributes related to embedding/tied weights:")

for key, value in config.to_dict().items():
    key_lower = key.lower()

    if (
        "embed" in key_lower
        or "tie" in key_lower
        or "share" in key_lower
        or "vocab" in key_lower
    ):
        print(f"{key}: {value}")

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("=== Vocabulary dimensions ===")
print("Tokenizer length:", len(tok))
print("Source vocabulary size:", tok.src_vocab_size)
print("Target vocabulary size:", tok.tgt_vocab_size)

print("\n=== Token ID range checks ===")

for name, vocab in [
    ("Source", tok.src_encoder),
    ("Target", tok.tgt_encoder),
]:
    ids = list(vocab.values())

    print(f"\n{name}:")
    print("Number of entries:", len(vocab))
    print("Minimum token ID:", min(ids))
    print("Maximum token ID:", max(ids))
    print("Unique token IDs:", len(set(ids)))
    print("Contiguous IDs:", set(ids) == set(range(max(ids) + 1)))

print("\n=== Special and language token IDs ===")

for token in [
    "<s>", "<pad>", "</s>", "<unk>",
    "eng_Latn", "hin_Deva", "mar_Deva", "sat_Olck",
]:
    print(
        repr(token),
        "| source ID:", tok.src_encoder.get(token),
        "| target ID:", tok.tgt_encoder.get(token),
    )

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Use one real example from the prepared dataset.
src_col, tgt_col = language_columns["santali"]
row = all_data["santali"]["train"].iloc[0]

source_text = str(row[src_col])
target_text = str(row[tgt_col])

# Apply the same language-conditioning convention used earlier.
processed_source = processor.preprocess_batch(
    [source_text],
    src_lang="eng_Latn",
    tgt_lang="sat_Olck",
)[0]

source_encoding = tok(
    processed_source,
    return_tensors="pt",
    add_special_tokens=True,
)

target_encoding = tok(
    text_target=target_text,
    return_tensors="pt",
    add_special_tokens=True,
)

print("=== Model embedding sizes ===")
print("Encoder embedding rows:", model.get_input_embeddings().num_embeddings)
print("Decoder embedding rows:", model.get_decoder().embed_tokens.num_embeddings)
print("Output projection rows:", model.get_output_embeddings().out_features)

print("\n=== Conditioned source ===")
print(processed_source[:300])
print("Source IDs:", source_encoding["input_ids"][0, :12].tolist())
print("Source length:", source_encoding["input_ids"].shape[1])

print("\n=== Target labels ===")
print("Target text:", target_text[:200])
print("Target IDs:", target_encoding["input_ids"][0, :12].tolist())
print("Target length:", target_encoding["input_ids"].shape[1])

print("\n=== ID range checks ===")
print(
    "All source IDs valid:",
    source_encoding["input_ids"].min().item() >= 0
    and source_encoding["input_ids"].max().item()
    < model.get_input_embeddings().num_embeddings
)
print(
    "All target IDs valid:",
    target_encoding["input_ids"].min().item() >= 0
    and target_encoding["input_ids"].max().item()
    < model.get_output_embeddings().out_features
)

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import inspect
from transformers import AutoConfig, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print("=== Forward method signature ===")
print(inspect.signature(model.forward))

print("\n=== Generation configuration ===")
print(model.generation_config)

print("\n=== Forward method source ===")
try:
    print(inspect.getsource(type(model).forward))
except (OSError, TypeError) as exc:
    print("Could not retrieve source:", repr(exc))

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# 6. Training Pipeline Validation

## 6.1 Forward Pass and Gradient Verification

### Objective

Before implementing LoRA fine-tuning, we need to verify that the pretrained IndicTrans2 model can process our prepared parallel data, calculate a valid translation loss, and propagate gradients successfully.

### Procedure

Using one real Santali training example, we will:

1. Prepare the English source sentence using the established `eng_Latn` → `sat_Olck` conditioning convention.
2. Tokenize the source and target sentences.
3. Verify that target token IDs are within the decoder vocabulary range.
4. Run a forward pass and check that the loss is finite.
5. Run a backward pass and confirm that gradients are computed.

### Expected Outcome

The model should produce a finite loss, correctly shaped logits, and a successful backward pass.

This is a diagnostic test only. We will not perform an optimizer step or update the pretrained weights.

Once these checks pass, we can inspect the model's actual attention-module names and determine suitable LoRA target modules before beginning baseline fine-tuning.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

model.train()

src_col, tgt_col = language_columns["santali"]
row = all_data["santali"]["train"].iloc[0]

source_text = str(row[src_col])
target_text = str(row[tgt_col])

processed_source = processor.preprocess_batch(
    [source_text],
    src_lang="eng_Latn",
    tgt_lang="sat_Olck",
)[0]

source_batch = tok(
    processed_source,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

target_batch = tok(
    text_target=target_text,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

input_ids = source_batch["input_ids"].to(DEVICE)
attention_mask = source_batch["attention_mask"].to(DEVICE)
labels = target_batch["input_ids"].to(DEVICE)

print("Input shape:", tuple(input_ids.shape))
print("Labels shape:", tuple(labels.shape))
print("Labels min/max:", labels.min().item(), labels.max().item())
print("Decoder vocabulary size:", model.config.decoder_vocab_size)

assert labels.min().item() >= 0
assert labels.max().item() < model.config.decoder_vocab_size

model.zero_grad(set_to_none=True)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
)

print("\nForward-pass loss:", outputs.loss.item())
print("Loss finite:", torch.isfinite(outputs.loss).item())
print("Logits shape:", tuple(outputs.logits.shape))

assert torch.isfinite(outputs.loss).item(), "Loss is not finite!"

outputs.loss.backward()

gradient_count = sum(
    parameter.grad is not None
    for parameter in model.parameters()
)

print("Parameters receiving gradients:", gradient_count)
print("Backward pass completed successfully.")

del outputs, model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

attention_modules = []

for name, module in model.named_modules():
    class_name = module.__class__.__name__.lower()

    if "attention" in class_name or "attn" in class_name:
        attention_modules.append((name, module.__class__.__name__))

print("Attention-related modules:")
for name, class_name in attention_modules:
    print(f"{name}: {class_name}")

print("\nTotal attention-related modules:", len(attention_modules))

In [ ]:
# Inspect the actual linear layers in one encoder and one decoder attention module.

for module_name in [
    "model.encoder.layers.0.self_attn",
    "model.decoder.layers.0.self_attn",
    "model.decoder.layers.0.encoder_attn",
]:
    module = model.get_submodule(module_name)

    print(f"\n{module_name}")
    print("-" * len(module_name))

    for name, submodule in module.named_modules():
        if isinstance(submodule, torch.nn.Linear):
            print(
                f"{name or '<self>'}: "
                f"{submodule.in_features} -> {submodule.out_features}"
            )

In [ ]:
import torch
import transformers
import peft
import importlib.metadata as metadata

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

for package in ["torchao", "accelerate", "tokenizers"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

In [ ]:
%pip uninstall -y torchao

In [ ]:
from peft import LoraConfig, get_peft_model

# Apply LoRA to query and value projections across encoder and decoder attention.
LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

lora_model = get_peft_model(model, LORA_CONFIG)

lora_model.print_trainable_parameters()

trainable_params = sum(
    p.numel() for p in lora_model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in lora_model.parameters())

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

# Verify exactly which modules received LoRA adapters.
print("\nLoRA-wrapped modules:")
for name, module in lora_model.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        print(name)

In [ ]:
# Verify trainable parameter names and confirm the base model is frozen.

trainable_names = [
    name
    for name, param in lora_model.named_parameters()
    if param.requires_grad
]

print("Number of trainable parameter tensors:", len(trainable_names))
print("\nFirst 20 trainable parameter names:")

for name in trainable_names[:20]:
    print(name)

unexpected_trainable = [
    name for name in trainable_names
    if "lora_A" not in name and "lora_B" not in name
]

print("\nUnexpected trainable parameters:", len(unexpected_trainable))

assert len(trainable_names) > 0
assert len(unexpected_trainable) == 0, (
    "Some trainable parameters are not LoRA A/B adapters."
)

print("\nPASS: Only LoRA adapter parameters are trainable.")

In [ ]:
# Prepare Santali training and validation examples for LoRA fine-tuning.

from torch.utils.data import Dataset, DataLoader
import pandas as pd

LANGUAGE = "santali"
SRC_LANG = "eng_Latn"
TGT_LANG = "sat_Olck"
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

src_col, tgt_col = language_columns[LANGUAGE]

train_df = all_data[LANGUAGE]["train"].copy()
val_df = all_data[LANGUAGE]["validation"].copy()

def prepare_examples(df):
    examples = []

    for _, row in df.iterrows():
        source_text = str(row[src_col]).strip()
        target_text = str(row[tgt_col]).strip()

        if not source_text or not target_text:
            continue

        processed_source = processor.preprocess_batch(
            [source_text],
            src_lang=SRC_LANG,
            tgt_lang=TGT_LANG,
        )[0]

        examples.append({
            "source": processed_source,
            "target": target_text,
        })

    return examples

train_examples = prepare_examples(train_df)
val_examples = prepare_examples(val_df)

print("Language:", LANGUAGE)
print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("\nPrepared example:")
print("Source:", train_examples[0]["source"])
print("Target:", train_examples[0]["target"])

assert len(train_examples) > 0
assert len(val_examples) > 0

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 4

class TranslationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_translation_batch(examples):
    sources = [example["source"] for example in examples]
    targets = [example["target"] for example in examples]

    source_batch = tok(
        sources,
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt",
    )

    target_batch = tok(
        text_target=targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt",
    )

    labels = target_batch["input_ids"].clone()
    labels[target_batch["attention_mask"] == 0] = -100

    return {
        "input_ids": source_batch["input_ids"],
        "attention_mask": source_batch["attention_mask"],
        "labels": labels,
    }


train_dataset = TranslationDataset(train_examples)
val_dataset = TranslationDataset(val_examples)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_translation_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_translation_batch,
)

batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")

print("\nLabels contain -100:", (batch["labels"] == -100).any().item())
print("Input IDs within source vocabulary:",
      batch["input_ids"].max().item() < model.config.encoder_vocab_size)

valid_labels = batch["labels"][batch["labels"] != -100]
print("Target IDs within decoder vocabulary:",
      valid_labels.max().item() < model.config.decoder_vocab_size)

assert batch["input_ids"].shape[0] == BATCH_SIZE
assert (batch["labels"] == -100).any().item()
assert batch["input_ids"].max().item() < model.config.encoder_vocab_size
assert valid_labels.max().item() < model.config.decoder_vocab_size

print("\nPASS: Batch construction and vocabulary checks succeeded.")

In [ ]:
from torch.optim import AdamW

# Keep the first test small; this is not the full training loop.
LEARNING_RATE = 1e-4

optimizer = AdamW(
    (p for p in lora_model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
)

lora_model.train()
lora_model.zero_grad(set_to_none=True)

batch = {
    key: value.to(DEVICE)
    for key, value in batch.items()
}

# Snapshot one trainable parameter before the update.
tracked_name, tracked_param = next(
    (name, param)
    for name, param in lora_model.named_parameters()
    if param.requires_grad and "lora_B" in name
)
parameter_before = tracked_param.detach().clone()

outputs = lora_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],
    use_cache=False,
)

loss = outputs.loss
print("Loss before update:", loss.item())
assert torch.isfinite(loss).item(), "Loss is not finite."

loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in lora_model.parameters() if p.requires_grad],
    max_norm=1.0,
)
print("Gradient norm:", grad_norm.item())
assert torch.isfinite(grad_norm).item(), "Gradient norm is not finite."

optimizer.step()

parameter_after = dict(lora_model.named_parameters())[tracked_name].detach()
changed = not torch.equal(parameter_before, parameter_after)

print("Tracked parameter:", tracked_name)
print("Parameter changed after optimizer step:", changed)

assert changed, "The tracked LoRA parameter did not change."
print("\nPASS: One optimizer step completed successfully.")


# 7. Baseline LoRA Fine-Tuning

## 7.1 Experimental Setup

We will establish the baseline by fine-tuning the pretrained IndicTrans2 model using LoRA while keeping its original vocabulary unchanged.

The first experiment will use English-to-Santali translation, with the established language-tag convention and the prepared training and validation splits.

### Training Configuration

- **Maximum epochs:** 10
- **Early stopping:** Patience of 2 epochs
- **Early-stopping minimum improvement:** 0.001 in validation loss
- **Micro-batch size:** 4
- **Gradient accumulation steps:** 4
- **Effective batch size:** 16
- **Learning rate:** 0.0001
- **Optimizer:** AdamW
- **LoRA rank:** 16
- **LoRA alpha:** 32
- **LoRA dropout:** 0.05
- **Vocabulary:** Original, unchanged

### Experimental Procedure

1. Reload the original pretrained checkpoint and initialize fresh LoRA adapters.
2. Train on the Santali training split.
3. Evaluate validation loss after each epoch.
4. Save the best checkpoint and stop early if validation loss stops improving.
5. Record training loss, validation loss, runtime, and the best epoch.
6. Evaluate the selected checkpoint on the held-out test split only after training and model selection are complete.

This trained baseline will serve as the reference for the subsequent vocabulary-extension experiment. The same data splits, language tags, LoRA configuration, and evaluation procedure must be used in both experiments.


In [ ]:
# Santali baseline LoRA training configuration

import random
import numpy as np
import torch

SEED = 42
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 10                 # Maximum epochs
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 0.001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Experiment: Santali baseline — original vocabulary + LoRA")
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Minimum validation-loss improvement:", EARLY_STOPPING_MIN_DELTA)
print("Micro-batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Warmup ratio:", WARMUP_RATIO)
print("Device:", DEVICE)

In [ ]:
import gc
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

# Release previous model instances and GPU memory.
if "lora_model" in globals():
    del lora_model
if "model" in globals():
    del model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load a fresh copy of the original pretrained checkpoint.
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

# Create new LoRA adapters with the established configuration.
LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

lora_model = get_peft_model(model, LORA_CONFIG)

# Verify the fresh training setup.
lora_model.print_trainable_parameters()

print("\nModel:", MODEL_ID)
print("Device:", DEVICE)
print("Maximum epochs:", NUM_EPOCHS)
print("Early-stopping patience:", EARLY_STOPPING_PATIENCE)
print("Original vocabulary retained:", True)
print("Fresh LoRA adapters initialized:", True)

In [ ]:
import math
from pathlib import Path
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Reproducible baseline output directory
BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

# Number of optimizer updates in one epoch
updates_per_epoch = math.ceil(
    len(train_loader) / GRADIENT_ACCUMULATION_STEPS
)
total_training_steps = updates_per_epoch * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

optimizer = AdamW(
    (p for p in lora_model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Training batches per epoch:", len(train_loader))
print("Optimizer updates per epoch:", updates_per_epoch)
print("Maximum optimizer updates:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Initial learning rate:", optimizer.param_groups[0]["lr"])
print("Checkpoint directory:", BASELINE_DIR)

assert len(train_loader) > 0
assert len(val_loader) > 0
assert total_training_steps > 0

print("\nPASS: Optimizer and scheduler are ready.")


## 7.3 Baseline Training Loop

Train the fresh LoRA adapters on the Santali training split and evaluate validation loss after every epoch.

The loop will:
- Accumulate gradients over four micro-batches.
- Clip gradients and update the optimizer and learning-rate scheduler.
- Save the adapter checkpoint whenever validation loss improves by at least the configured minimum delta.
- Stop after two consecutive epochs without sufficient improvement, or after ten epochs.
- Restore the best checkpoint at the end.

The test split will remain untouched until final evaluation.


In [ ]:
import copy
import json
import time
import numpy as np
import torch
from pathlib import Path

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []
global_step = 0

best_checkpoint_dir = BASELINE_DIR / "best_adapter"
best_checkpoint_dir.mkdir(parents=True, exist_ok=True)

optimizer.zero_grad(set_to_none=True)
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    lora_model.train()
    train_losses = []
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = lora_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
            use_cache=False,
        )

        loss = outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}, batch {batch_idx}"
            )

        train_losses.append(loss.item())
        (loss / GRADIENT_ACCUMULATION_STEPS).backward()

        should_update = (
            batch_idx % GRADIENT_ACCUMULATION_STEPS == 0
            or batch_idx == len(train_loader)
        )

        if should_update:
            torch.nn.utils.clip_grad_norm_(
                [p for p in lora_model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        if batch_idx % 500 == 0:
            print(
                f"Epoch {epoch}/{NUM_EPOCHS} | "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    # Validation: no gradients or optimizer updates.
    lora_model.eval()
    val_losses = []

    with torch.no_grad():
        for val_batch in val_loader:
            val_batch = {k: v.to(DEVICE) for k, v in val_batch.items()}

            val_outputs = lora_model(
                input_ids=val_batch["input_ids"],
                attention_mask=val_batch["attention_mask"],
                labels=val_batch["labels"],
                use_cache=False,
            )

            val_loss = val_outputs.loss
            if not torch.isfinite(val_loss):
                raise RuntimeError(
                    f"Non-finite validation loss at epoch {epoch}"
                )

            val_losses.append(val_loss.item())

    mean_train_loss = float(np.mean(train_losses))
    mean_val_loss = float(np.mean(val_losses))
    elapsed = time.time() - epoch_start

    record = {
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "validation_loss": mean_val_loss,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "optimizer_steps": global_step,
        "epoch_seconds": elapsed,
    }
    history.append(record)

    print(
        f"\nEpoch {epoch}: "
        f"train_loss={mean_train_loss:.4f}, "
        f"validation_loss={mean_val_loss:.4f}, "
        f"time={elapsed / 60:.1f} min"
    )

    # A new best requires the configured minimum improvement.
    if mean_val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
        best_val_loss = mean_val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        lora_model.save_pretrained(best_checkpoint_dir)

        with open(BASELINE_DIR / "training_history.json", "w") as f:
            json.dump(history, f, indent=2)

        print(f"Saved new best adapter at epoch {epoch}.")
    else:
        epochs_without_improvement += 1
        print(
            "No sufficient validation improvement: "
            f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

# Restore the best adapter weights before evaluation.
if best_epoch == 0:
    raise RuntimeError("No best checkpoint was saved.")

from peft import PeftModel

lora_model.load_adapter(str(best_checkpoint_dir), adapter_name="best")
lora_model.set_adapter("best")

with open(BASELINE_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nTraining finished.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Optimizer steps completed:", global_step)
print("Total training time (minutes):", round((time.time() - training_start) / 60, 2))
print("Best adapter directory:", best_checkpoint_dir)

In [67]:
from pathlib import Path
import json

BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
BEST_DIR = BASELINE_DIR / "best_adapter"

print("Baseline directory exists:", BASELINE_DIR.exists())
print("Best adapter directory exists:", BEST_DIR.exists())

if BASELINE_DIR.exists():
    print("\nFiles in baseline directory:")
    for path in sorted(BASELINE_DIR.rglob("*")):
        if path.is_file():
            print(f"  {path.relative_to(BASELINE_DIR)} ({path.stat().st_size / 1024:.1f} KB)")

history_path = BASELINE_DIR / "training_history.json"

if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)

    print("\nRecorded training history:")
    for record in history:
        print(record)
else:
    print("\nNo training_history.json found.")

Baseline directory exists: True
Best adapter directory exists: True

Files in baseline directory:
  best_adapter/README.md (5.1 KB)
  best_adapter/adapter_config.json (1.1 KB)
  best_adapter/adapter_model.safetensors (13854.0 KB)
  training_history.json (1.5 KB)

Recorded training history:
{'epoch': 1, 'train_loss': 3.4527233228723997, 'validation_loss': 3.1480962914156625, 'learning_rate': 9.278008298755188e-05, 'optimizer_steps': 1118, 'epoch_seconds': 730.6976745128632}
{'epoch': 2, 'train_loss': 3.474387677183516, 'validation_loss': 3.1195955697791167, 'learning_rate': 8.247118487782388e-05, 'optimizer_steps': 2236, 'epoch_seconds': 733.1934769153595}
{'epoch': 3, 'train_loss': 3.4220167555496532, 'validation_loss': 3.0892319277108435, 'learning_rate': 7.21622867680959e-05, 'optimizer_steps': 3354, 'epoch_seconds': 731.6470923423767}
{'epoch': 4, 'train_loss': 3.384134605303623, 'validation_loss': 3.0466161521084336, 'learning_rate': 6.185338865836792e-05, 'optimizer_steps': 4472, 

In [68]:
import json
from pathlib import Path

BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
BEST_DIR = BASELINE_DIR / "best_adapter"

with open(BASELINE_DIR / "training_history.json", "r") as f:
    history = json.load(f)

best_record = min(history, key=lambda x: x["validation_loss"])

print("Recorded epochs:", len(history))
print("Last recorded epoch:", history[-1]["epoch"])
print("Best recorded epoch:", best_record["epoch"])
print("Best recorded validation loss:", best_record["validation_loss"])
print("Adapter weights exist:", (BEST_DIR / "adapter_model.safetensors").exists())
print("Adapter config exists:", (BEST_DIR / "adapter_config.json").exists())
print("History file size:", (BASELINE_DIR / "training_history.json").stat().st_size, "bytes")

Recorded epochs: 7
Last recorded epoch: 7
Best recorded epoch: 7
Best recorded validation loss: 3.0151935868473894
Adapter weights exist: True
Adapter config exists: True
History file size: 1520 bytes


In [69]:
import gc
import json
import torch
from pathlib import Path
from transformers import AutoModelForSeq2SeqLM
from peft import PeftModel

# Paths
MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
BEST_DIR = BASELINE_DIR / "best_adapter"

# Verify checkpoint files before loading
assert (BEST_DIR / "adapter_config.json").exists(), "Adapter config missing"
assert (BEST_DIR / "adapter_model.safetensors").exists(), "Adapter weights missing"

with open(BASELINE_DIR / "training_history.json", "r") as f:
    history = json.load(f)

best_record = min(history, key=lambda x: x["validation_loss"])
print("Best recorded epoch:", best_record["epoch"])
print("Best validation loss:", round(best_record["validation_loss"], 4))

# Release existing model objects, if any
for name in ("lora_model", "model"):
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load the original base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

# Attach the saved, trained LoRA adapter
baseline_model = PeftModel.from_pretrained(
    base_model,
    str(BEST_DIR),
    is_trainable=False,
)

baseline_model.eval()

print("\nSaved baseline adapter loaded successfully.")
print("Evaluation mode:", not baseline_model.training)
print("Device:", next(baseline_model.parameters()).device)
print("Trainable parameters:", sum(
    p.numel() for p in baseline_model.parameters() if p.requires_grad
))

Best recorded epoch: 7
Best validation loss: 3.0152

Saved baseline adapter loaded successfully.
Evaluation mode: True
Device: cuda:0
Trainable parameters: 0


In [70]:
# Verify the held-out Santali test split and evaluation objects

LANGUAGE = "santali"
SRC_LANG = "eng_Latn"
TGT_LANG = "sat_Olck"

test_df = all_data[LANGUAGE]["test"].copy()
src_col, tgt_col = language_columns[LANGUAGE]

assert len(test_df) == 994, f"Unexpected test size: {len(test_df)}"
assert "baseline_model" in globals()
assert "tok" in globals()
assert "processor" in globals()

print("Language:", LANGUAGE)
print("Direction: English -> Santali")
print("Test examples:", len(test_df))
print("Source column:", src_col)
print("Target column:", tgt_col)
print("Missing source values:", test_df[src_col].isna().sum())
print("Missing target values:", test_df[tgt_col].isna().sum())
print("Model is in evaluation mode:", not baseline_model.training)
print("Test split loaded; no training performed.")

Language: santali
Direction: English -> Santali
Test examples: 994
Source column: English
Target column: Santali
Missing source values: 0
Missing target values: 0
Model is in evaluation mode: True
Test split loaded; no training performed.


In [71]:
import torch
import pandas as pd
from tqdm.auto import tqdm

baseline_model.eval()

test_sources = test_df[src_col].astype(str).str.strip().tolist()
test_references = test_df[tgt_col].astype(str).str.strip().tolist()

# Apply the same language-tag preprocessing used during training.
processed_sources = [
    processor.preprocess_batch(
        [source],
        src_lang=SRC_LANG,
        tgt_lang=TGT_LANG,
    )[0]
    for source in test_sources
]

predictions = []
batch_size = 8

for start in tqdm(
    range(0, len(processed_sources), batch_size),
    desc="Generating test translations",
):
    batch_texts = processed_sources[start:start + batch_size]

    inputs = tok(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.inference_mode():
        generated_ids = baseline_model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            do_sample=False,
        )

    batch_predictions = tok.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )
    predictions.extend(batch_predictions)

assert len(predictions) == len(test_references)

test_results = pd.DataFrame({
    "source_english": test_sources,
    "reference_santali": test_references,
    "prediction_santali": predictions,
})

print("Test examples translated:", len(test_results))
print("\nSample predictions:")
display(test_results.head(5))

Generating test translations:   0%|          | 0/125 [00:00<?, ?it/s]

Test examples translated: 994

Sample predictions:


,source_english,reference_santali,prediction_santali
0,Puducherry is one of the most popular tourist ...,ᱡᱟᱹᱛᱤᱭᱟᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱫ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ ᱠᱚᱴᱷᱮᱱ ᱯᱩ...,ᱯᱩᱫᱩᱪᱮᱨᱤ ᱫᱚ ᱵᱷᱟᱨᱚᱛ ᱨᱮᱭᱟᱜ ᱡᱟᱹᱛᱤᱭᱟᱹᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡ...
1,She is widely regarded in modern Indian histor...,ᱱᱟᱦᱟᱜ ᱵᱷᱟᱨᱚᱛᱤᱭᱚ ᱱᱟᱜᱟᱢ ᱚᱞ ᱨᱮ ᱩᱱᱤ ᱫᱚ ᱟᱹᱰᱤ ᱛᱮᱫ ᱜᱮ...,ᱩᱱᱤ ᱫᱚ ᱱᱟᱦᱟᱜ ᱵᱷᱟᱨᱚᱛ ᱨᱮᱭᱟᱜ ᱱᱟᱜᱟᱢᱤᱭᱟᱹ ᱨᱮ ᱟᱠᱵᱚᱨ ᱟ...
2,"This way, the Robo advisor can go from giving ...","ᱱᱚᱶᱟ ᱞᱮᱠᱟᱛᱮ, ᱯᱚᱨᱟᱢᱮᱥ ᱮᱢ ᱠᱷᱚᱱ ᱮᱦᱚᱵ ᱠᱟᱛᱮᱜ ᱯᱨᱚᱠᱤᱛ...","ᱱᱚᱶᱟ ᱞᱮᱠᱟᱛᱮ,19 ᱨᱳᱵᱳ ᱯᱚᱨᱟᱢᱚᱥᱚᱠ ᱟᱢᱟᱜ ᱠᱟᱹᱣᱰᱤ ᱵᱚᱱᱫ..."
3,He hung the portraits of all these three perso...,ᱩᱱᱤ ᱫᱚ ᱟᱭᱟᱜ ᱜᱳᱣᱨᱤ ᱠᱩᱧᱡᱚ ᱨᱮᱱᱟᱜ ᱛᱟᱦᱮᱱ ᱚᱲᱟᱜ ᱨᱮ ᱱᱩ...,ᱩᱱᱤ ᱫᱚ ᱟᱭᱟᱜ ᱜᱳᱨᱤ ᱠᱩᱱ ᱟᱜ ᱚᱲᱟᱜ ᱨᱮ ᱱᱚᱣᱟ ᱯᱮᱭᱟ ᱦᱚᱲᱟ...
4,I am unsure about driving in the US after driv...,ᱤᱧ ᱵᱷᱟᱨᱚᱛ ᱨᱮ ᱜᱟᱹᱰᱤ ᱪᱟᱞᱟᱣ ᱛᱟᱭᱚᱢ ᱢᱟᱨᱠᱤᱱ ᱡᱩᱠᱛᱚ ᱨᱟ...,ᱵᱷᱟᱨᱚᱛ ᱨᱮ ᱰᱨᱟᱭᱵᱷᱤᱝ ᱛᱟᱭᱚᱢ ᱤᱧ ᱢᱟᱨᱠᱤᱱ ᱡᱩᱠᱛᱚᱨᱟᱥᱴᱨᱚ...


In [72]:
import json
from pathlib import Path

BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

# Save source, reference, and generated translation for every test example.
predictions_path = BASELINE_DIR / "baseline_test_predictions.csv"
test_results.to_csv(predictions_path, index=False, encoding="utf-8-sig")

# Save metadata about the evaluated checkpoint.
evaluation_metadata = {
    "model_id": MODEL_ID,
    "language": "Santali",
    "direction": "English to Santali",
    "checkpoint_epoch": 7,
    "best_validation_loss": 3.0151935868473894,
    "test_examples": len(test_results),
    "generation": {
        "batch_size": 8,
        "num_beams": 4,
        "do_sample": False,
        "max_new_tokens": MAX_TARGET_LENGTH,
    },
}

metadata_path = BASELINE_DIR / "baseline_evaluation_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_metadata, f, ensure_ascii=False, indent=2)

print("Predictions saved:", predictions_path)
print("Rows saved:", len(test_results))
print("Metadata saved:", metadata_path)
print("CSV size (KB):", round(predictions_path.stat().st_size / 1024, 2))

Predictions saved: /kaggle/working/santali_baseline_lora/baseline_test_predictions.csv
Rows saved: 994
Metadata saved: /kaggle/working/santali_baseline_lora/baseline_evaluation_metadata.json
CSV size (KB): 717.26


In [73]:
# Install SacreBLEU only if it is not already available.
try:
    import sacrebleu
except ImportError:
    %pip install -q sacrebleu
    import sacrebleu

import json
from pathlib import Path

BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
PREDICTIONS_PATH = BASELINE_DIR / "baseline_test_predictions.csv"

# Read the saved predictions to ensure evaluation is reproducible.
eval_df = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")

preds = eval_df["prediction_santali"].fillna("").astype(str).tolist()
refs = eval_df["reference_santali"].fillna("").astype(str).tolist()

assert len(preds) == len(refs) == 994

# Corpus-level translation metrics.
bleu = sacrebleu.corpus_bleu(preds, [refs])
chrf = sacrebleu.corpus_chrf(preds, [refs])

# Exact string match is a simple additional diagnostic, not a translation metric.
exact_match = sum(
    p.strip() == r.strip() for p, r in zip(preds, refs)
) / len(refs) * 100

metrics = {
    "model": "IndicTrans2 Indic-Indic 1B + LoRA",
    "direction": "English to Santali",
    "checkpoint_epoch": 7,
    "validation_loss": 3.0151935868473894,
    "test_examples": len(refs),
    "sacrebleu": bleu.score,
    "chrf": chrf.score,
    "exact_match_percent": exact_match,
    "bleu_signature": str(bleu),
    "chrf_signature": str(chrf),
}

metrics_path = BASELINE_DIR / "baseline_test_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("BASELINE TEST METRICS")
print("-" * 40)
print(f"Test examples:       {len(refs)}")
print(f"SacreBLEU:           {bleu.score:.4f}")
print(f"chrF:                {chrf.score:.4f}")
print(f"Exact match:         {exact_match:.2f}%")
print(f"\nMetrics saved to: {metrics_path}")
print("\nBLEU configuration:", bleu)
print("chrF configuration:", chrf)

BASELINE TEST METRICS
----------------------------------------
Test examples:       994
SacreBLEU:           8.3859
chrF:                41.0353
Exact match:         0.00%

Metrics saved to: /kaggle/working/santali_baseline_lora/baseline_test_metrics.json

BLEU configuration: BLEU = 8.39 34.6/12.8/5.2/2.2 (BP = 1.000 ratio = 1.235 hyp_len = 24614 ref_len = 19923)
chrF configuration: chrF2 = 41.04


# Santali Baseline LoRA Experiment — Results and Progress

## 1. Objective

This experiment establishes the original-vocabulary baseline for our low-resource translation research before investigating vocabulary extension and embedding initialization.

- **Translation direction:** English → Santali
- **Base model:** `ai4bharat/indictrans2-indic-indic-1B`
- **Fine-tuning method:** LoRA
- **Vocabulary:** Original pretrained vocabulary, without extension
- **Evaluation set:** 994 test examples

The objective is to determine the baseline performance against which the vocabulary-extension experiment will be compared.

## 2. Dataset Preparation

The preprocessed dataset was loaded from the existing train, validation, and test splits.

| Split | Number of examples | Purpose |
|---|---:|---|
| Training | 17,883 | Fine-tuning LoRA parameters |
| Validation | 993 | Monitoring validation loss and selecting the checkpoint |
| Test | 994 | Final translation-quality evaluation |

The source column is `English`, and the target column is `Santali`.

The test split contained no missing source or target values. Earlier data checks found no exact source-target pair overlap across splits.

The test set was kept separate from training and checkpoint selection.

## 3. Model Architecture and Configuration

### Base model

- Model checkpoint: `ai4bharat/indictrans2-indic-indic-1B`
- Architecture: IndicTrans encoder-decoder sequence-to-sequence model
- Source language tag: `eng_Latn`
- Target language tag: `sat_Olck`

### LoRA configuration

| Parameter | Value |
|---|---:|
| Rank (`r`) | 16 |
| LoRA alpha | 32 |
| LoRA dropout | 0.05 |
| Target modules | `q_proj`, `v_proj` |
| Trainable parameters | 3,538,944 |
| Total parameters reported | 1,211,635,712 |
| Trainable parameter percentage | Approximately 0.2921% |

The setup checks confirmed that only LoRA parameters were trainable.

### Training configuration

| Parameter | Value |
|---|---:|
| Maximum epochs | 10 |
| Micro-batch size | 4 |
| Gradient accumulation steps | 4 |
| Nominal effective batch size | 16 |
| Learning rate | 0.0001 |
| Weight decay | 0.01 |
| Warmup ratio | 0.03 |
| Maximum source length | 256 tokens |
| Maximum target length | 256 tokens |
| Early-stopping patience | 2 epochs |
| Minimum validation-loss improvement | 0.001 |
| Hardware | NVIDIA Tesla T4 GPU |

Validation loss was calculated after each epoch. The adapter was saved when validation loss improved by at least the configured minimum delta.

## 4. Training Results

The recorded training history contains seven completed epochs.

| Epoch | Training Loss | Validation Loss | Learning Rate at Epoch End | Optimizer Steps | Time |
|---:|---:|---:|---:|---:|---:|
| 1 | 3.4527 | 3.1481 | 9.2780e-05 | 1,118 | 12.2 min |
| 2 | 3.4744 | 3.1196 | 8.2471e-05 | 2,236 | 12.2 min |
| 3 | 3.4220 | 3.0892 | 7.2162e-05 | 3,354 | 12.2 min |
| 4 | 3.3841 | 3.0466 | 6.1853e-05 | 4,472 | 12.2 min |
| 5 | 3.3567 | 3.0342 | 5.1544e-05 | 5,590 | 12.2 min |
| 6 | 3.3324 | 3.0255 | 4.1236e-05 | 6,708 | 12.2 min |
| 7 | 3.3140 | **3.0152** | 3.0927e-05 | 7,826 | 12.2 min |

### Observations

1. Validation loss decreased consistently from 3.1481 in epoch 1 to 3.0152 in epoch 7.
2. Training loss generally decreased, with a small increase during epoch 2.
3. Epoch 7 was the best recorded epoch.
4. Each epoch took approximately 12.2 minutes.
5. The available history does not establish that epochs 8–10 completed.

Validation loss measures the model's predictive loss on the validation data. It does not directly establish the quality or correctness of generated translations.

## 5. Checkpoint Recovery

The notebook session stopped after the laptop was left unattended. However, the saved adapter and training history were available in:

`/kaggle/working/santali_baseline_lora/`

The following files were verified:

- `best_adapter/README.md`
- `best_adapter/adapter_config.json`
- `best_adapter/adapter_model.safetensors`
- `training_history.json`

The best recorded validation loss was **3.0151935868 at epoch 7**.

The adapter was successfully loaded again.

- Model evaluation mode: Enabled
- Device: `cuda:0`
- Trainable parameters: 0

This confirmed that the saved adapter could be used for inference.

### Checkpointing limitation

The original training loop saved adapter weights and epoch-level history, but did not preserve the complete optimizer state, scheduler state, exact batch position, or random-number-generator states.

Therefore, the saved adapter supports inference and can serve as the starting point for further fine-tuning, but the interrupted training process cannot be resumed exactly from its previous optimizer and batch state.

Future experiments will use resumable checkpoints that preserve model or adapter weights, optimizer and scheduler states, epoch and step positions, early-stopping information, and random states.

## 6. Test Translation Generation

The saved epoch 7 adapter was used to generate translations for all 994 held-out English sentences.

### Generation settings

| Setting | Value |
|---|---|
| Batch size | 8 |
| Beam search | 4 beams |
| Sampling | Disabled |
| Maximum new tokens | 256 |
| Source language tag | `eng_Latn` |
| Target language tag | `sat_Olck` |

Generation completed successfully:

- Test examples translated: 994
- Batches processed: 125
- Approximate generation time: 5 minutes 58 seconds

### Saved predictions

The predictions were saved to:

`/kaggle/working/santali_baseline_lora/baseline_test_predictions.csv`

The CSV contains:

- `source_english`
- `reference_santali`
- `prediction_santali`

The recorded file size was approximately 717.26 KB.

Evaluation metadata was saved to:

`/kaggle/working/santali_baseline_lora/baseline_evaluation_metadata.json`

## 7. Test Evaluation Metrics

The metrics were calculated using the saved predictions and references for all 994 test examples.

| Metric | Baseline Result |
|---|---:|
| SacreBLEU | **8.3859** |
| chrF (chrF2) | **41.0353** |
| Exact string match | **0.00%** |

The metrics were saved to:

`/kaggle/working/santali_baseline_lora/baseline_test_metrics.json`

### 7.1 SacreBLEU Details

The reported SacreBLEU configuration produced:

- BLEU score: 8.39
- N-gram precisions: 34.6 / 12.8 / 5.2 / 2.2
- Brevity penalty: 1.000
- Hypothesis/reference length ratio: 1.235
- Hypothesis length: 24,614
- Reference length: 19,923

### 7.2 Interpretation

**SacreBLEU = 8.3859**

This indicates limited word-level n-gram overlap between generated translations and references under the reported evaluation configuration. BLEU is an overlap-based metric rather than a direct measure of semantic correctness.

**chrF = 41.0353**

chrF measures character n-gram overlap and provides a complementary signal for morphologically rich languages such as Santali. It should be interpreted alongside BLEU and manual inspection.

**Exact match = 0.00%**

None of the complete predictions exactly matched its reference after trimming leading and trailing whitespace. Exact match is a strict diagnostic and does not imply that every translation is incorrect.

**Hypothesis/reference length ratio = 1.235**

The generated translations were longer overall than the reference corpus. This warrants further investigation for verbosity, extra content, repetition, or segmentation differences. The ratio alone does not establish the cause.

A small sample of translations was inspected. Although the predictions were in Santali script, some differed substantially from their references. Systematic error analysis is needed to identify the main failure patterns.

## 8. What We Have Established

This experiment establishes an original-vocabulary LoRA baseline for English-to-Santali translation.

The results confirm that:

1. The IndicTrans2 model can be fine-tuned using the selected LoRA configuration.
2. The adapter checkpoint was saved and successfully reloaded.
3. The recovered adapter generated translations for all 994 test examples.
4. SacreBLEU and chrF were calculated using the saved predictions.
5. Predictions, metrics, metadata, adapter weights, and training history were saved as separate artifacts.

However, these results do not yet establish that vocabulary extension will improve translation quality. They also do not prove that the observed translation errors are caused by tokenization or missing vocabulary entries.

## 9. Limitations

- The current baseline covers only English → Santali.
- Seven completed epochs are recorded; epochs 8–10 are not confirmed.
- The saved adapter is not a complete resumable training checkpoint.
- The test set has been used for baseline evaluation and must remain unchanged for controlled comparisons.
- BLEU and chrF can be affected by Unicode normalization, punctuation, whitespace, and reference wording.
- The cause of the observed translation errors has not yet been established.

Any normalization applied during later evaluation must be documented and applied consistently across experimental conditions.

## 10. Next Steps

1. Preserve the baseline adapter, predictions, metrics, and training history.
2. Inspect representative translation errors and verify evaluation and tokenization behavior.
3. Implement and test a robust resumable-checkpoint mechanism before starting the next long training run.
4. Conduct the vocabulary-extension experiment using the same data splits, language tags, LoRA configuration, generation settings, and evaluation implementation wherever applicable.
5. Compare the original-vocabulary baseline against vocabulary extension using SacreBLEU, chrF, validation loss, tokenization diagnostics, and targeted error analysis.
6. Investigate embedding initialization as a separate experimental variable so that the effect of vocabulary extension can be distinguished from the effect of initialization.

## 11. Artifact Directory

The current Kaggle working directory contains the following artifacts:

```text
/kaggle/working/santali_baseline_lora/
├── best_adapter/
│   ├── README.md
│   ├── adapter_config.json
│   └── adapter_model.safetensors
├── training_history.json
├── baseline_test_predictions.csv
├── baseline_evaluation_metadata.json
└── baseline_test_metrics.json
```

These paths refer to the current Kaggle session's working directory and are not guaranteed to persist after a session reset unless the artifacts are preserved or downloaded.

## 12. Final Summary

The original-vocabulary IndicTrans2 1B + LoRA baseline for English → Santali achieved a best recorded validation loss of **3.0152 at epoch 7**.

On the held-out test set of **994 examples**, it achieved:

- **SacreBLEU:** 8.3859
- **chrF:** 41.0353
- **Exact match:** 0.00%

The adapter was recovered successfully, and all test predictions and evaluation metrics were saved.

These results form the reference baseline for the next controlled experiment: vocabulary extension with a robust resumable-checkpoint mechanism.


**MODEL SAVING**

In [74]:
import json
import zipfile
import shutil
from pathlib import Path
from datetime import datetime

# --------------------------------------------------
# 1. Define paths
# --------------------------------------------------
BASELINE_DIR = Path("/kaggle/working/santali_baseline_lora")
BEST_ADAPTER_DIR = BASELINE_DIR / "best_adapter"
BUNDLE_DIR = Path("/kaggle/working/santali_baseline_backup")

ZIP_PATH = Path("/kaggle/working/santali_baseline_backup.zip")

# --------------------------------------------------
# 2. Verify essential artifacts before packaging
# --------------------------------------------------
required_files = {
    "Best adapter weights":
        BEST_ADAPTER_DIR / "adapter_model.safetensors",
    "Adapter configuration":
        BEST_ADAPTER_DIR / "adapter_config.json",
    "Training history":
        BASELINE_DIR / "training_history.json",
    "Test predictions":
        BASELINE_DIR / "baseline_test_predictions.csv",
    "Evaluation metrics":
        BASELINE_DIR / "baseline_test_metrics.json",
    "Evaluation metadata":
        BASELINE_DIR / "baseline_evaluation_metadata.json",
}

missing = [
    name for name, path in required_files.items()
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        "Cannot create a complete baseline backup. Missing files:\n"
        + "\n".join(missing)
    )

# --------------------------------------------------
# 3. Prepare a clean backup directory
# --------------------------------------------------
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)

BUNDLE_DIR.mkdir(parents=True)

# Copy adapter, history, predictions, and evaluation files.
shutil.copytree(
    BEST_ADAPTER_DIR,
    BUNDLE_DIR / "best_adapter"
)

for name, path in required_files.items():
    if path.parent == BASELINE_DIR:
        shutil.copy2(path, BUNDLE_DIR / path.name)

# --------------------------------------------------
# 4. Save model and experiment configuration
# --------------------------------------------------
training_history_path = BUNDLE_DIR / "training_history.json"

with open(training_history_path, encoding="utf-8") as f:
    training_history = json.load(f)

baseline_info = {
    "experiment": "Original-vocabulary Santali LoRA baseline",
    "base_model": "ai4bharat/indictrans2-indic-indic-1B",
    "adapter_path": "best_adapter/",
    "best_recorded_epoch": 7,
    "best_validation_loss": 3.0151935868473894,
    "language": "Santali",
    "direction": "English -> Santali",
    "source_language_tag": "eng_Latn",
    "target_language_tag": "sat_Olck",
    "vocabulary_extension": False,
    "training_configuration": {
        "lora_rank": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "v_proj"],
        "learning_rate": 0.0001,
        "weight_decay": 0.01,
        "micro_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "maximum_epochs": 10,
        "maximum_source_length": 256,
        "maximum_target_length": 256,
        "early_stopping_patience": 2,
        "early_stopping_min_delta": 0.001,
    },
    "generation_configuration": {
        "batch_size": 8,
        "num_beams": 4,
        "do_sample": False,
        "max_new_tokens": 256,
    },
    "recorded_completed_epochs": len(training_history),
    "checkpointing_note": (
        "This is an inference-ready adapter backup, not a full resumable "
        "training checkpoint. Optimizer, scheduler, exact sampler position, "
        "and random states were not saved by the original training loop."
    ),
    "created_utc": datetime.utcnow().isoformat() + "Z",
}

with open(
    BUNDLE_DIR / "baseline_model_info.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(baseline_info, f, ensure_ascii=False, indent=2)

# --------------------------------------------------
# 5. Add instructions for future reuse
# --------------------------------------------------
readme = """# Santali Baseline Model Backup

## Model
Base checkpoint: ai4bharat/indictrans2-indic-indic-1B
Fine-tuning: LoRA
Direction: English -> Santali
Source tag: eng_Latn
Target tag: sat_Olck
Vocabulary: original pretrained vocabulary
Best recorded epoch: 7
Best validation loss: 3.0151935868

## Contents
- best_adapter/: trained LoRA adapter weights and configuration
- training_history.json: recorded epoch-level history
- baseline_test_predictions.csv: all 994 test predictions and references
- baseline_test_metrics.json: SacreBLEU, chrF, and exact-match results
- baseline_evaluation_metadata.json: evaluation metadata
- baseline_model_info.json: model, training, and generation configuration

## Baseline test results
- SacreBLEU: 8.3859
- chrF: 41.0353
- Exact match: 0.00%
- Test examples: 994

## Reloading the adapter
Use a compatible Transformers/PEFT environment and the base model checkpoint.
Load the base model with trust_remote_code=True, then load the adapter
from the best_adapter directory using PEFT.

The original base model weights are NOT included in this ZIP. They must be
downloaded from Hugging Face, so internet access and any required access
permissions are needed.

## Important limitation
This backup supports inference and further fine-tuning from the saved adapter.
It does not resume the interrupted training run exactly because the original
training loop did not save optimizer, scheduler, sampler, and RNG states.

Kaggle working-directory paths are not permanent. Keep this ZIP downloaded
on your own computer or in another persistent storage location.
"""

(BUNDLE_DIR / "README.md").write_text(readme, encoding="utf-8")

# --------------------------------------------------
# 6. Create ZIP
# --------------------------------------------------
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for file_path in sorted(BUNDLE_DIR.rglob("*")):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=file_path.relative_to(BUNDLE_DIR.parent),
            )

# --------------------------------------------------
# 7. Verify the ZIP
# --------------------------------------------------
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_file = archive.testzip()
    archived_files = archive.namelist()

if bad_file:
    raise RuntimeError(f"ZIP verification failed: {bad_file}")

print("BASELINE BACKUP CREATED SUCCESSFULLY")
print("-" * 45)
print("ZIP path:", ZIP_PATH)
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))
print("Files included:", len(archived_files))
print("ZIP integrity: passed")
print("\nIncluded files:")

for file_name in archived_files:
    print(" -", file_name)

/tmp/ipykernel_144/4287085878.py:109: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_utc": datetime.utcnow().isoformat() + "Z",


BASELINE BACKUP CREATED SUCCESSFULLY
---------------------------------------------
ZIP path: /kaggle/working/santali_baseline_backup.zip
ZIP size (MB): 12.72
Files included: 9
ZIP integrity: passed

Included files:
 - santali_baseline_backup/README.md
 - santali_baseline_backup/baseline_evaluation_metadata.json
 - santali_baseline_backup/baseline_model_info.json
 - santali_baseline_backup/baseline_test_metrics.json
 - santali_baseline_backup/baseline_test_predictions.csv
 - santali_baseline_backup/best_adapter/README.md
 - santali_baseline_backup/best_adapter/adapter_config.json
 - santali_baseline_backup/best_adapter/adapter_model.safetensors
 - santali_baseline_backup/training_history.json


# 8. Vocabulary Extension + LoRA Fine-Tuning

## 8.1 Objective

The objective of this experiment is to investigate whether extending the vocabulary of a pretrained multilingual neural machine translation model with useful Santali tokens or subwords can improve translation quality for a low-resource language.

In the baseline experiment, the original IndicTrans2 vocabulary was retained, and only LoRA parameters were fine-tuned. In this experiment, we will extend the vocabulary and investigate how the new token embeddings should be initialized before fine-tuning.

The primary research question is:

**Does vocabulary extension improve English-to-Santali translation quality compared with fine-tuning the original vocabulary using LoRA?**

## 8.2 Experimental Comparison

We will compare two experimental conditions.

| Component | Experiment A: Baseline | Experiment B: Vocabulary Extension |
|---|---|---|
| Base model | IndicTrans2 1B | Same checkpoint |
| Vocabulary | Original | Extended with selected Santali tokens/subwords |
| Embedding initialization | Original pretrained embeddings | Original embeddings retained; new embeddings initialized using a defined strategy |
| Fine-tuning | LoRA | LoRA |
| Training dataset | Santali training split | Same training split |
| Validation dataset | Santali validation split | Same validation split |
| Test dataset | 994 held-out examples | Same 994 held-out examples |
| Evaluation metrics | SacreBLEU, chrF | Same metrics and evaluation configuration |

The baseline results will remain unchanged and serve as the reference for the comparison.

## 8.3 Experimental Controls

To make the comparison meaningful, we will keep the following settings consistent wherever possible:

- Base model checkpoint and pretrained parameters.
- Training, validation, and test splits.
- Source and target language tags.
- LoRA rank, alpha, dropout, and target modules.
- Batch size, gradient accumulation, learning rate, and other training settings.
- Generation parameters and evaluation metric configuration.
- Test examples and reference translations.

The vocabulary modification and the associated new embedding parameters will be the principal differences between the experiments.

## 8.4 Vocabulary Extension Strategy

The model uses separate source-side and target-side SentencePiece vocabularies. Since the current task is English-to-Santali translation, we will first investigate the target-side Santali vocabulary and the corresponding decoder and output-projection parameters.

The extension procedure will be designed to:

1. Identify useful Santali tokens or subwords from the training corpus.
2. Avoid adding unnecessary or redundant vocabulary entries.
3. Preserve the original vocabulary IDs and pretrained embeddings.
4. Assign new IDs to selected tokens.
5. Initialize new embedding parameters using a clearly defined strategy.
6. Verify tokenizer, embedding, decoder, and output-projection compatibility.
7. Confirm that the model can complete a forward and backward pass after modification.

We will not assume that the standard tokenizer-extension procedure for other Transformers models can be applied directly to IndicTrans2 without checking its custom tokenizer and architecture.

## 8.5 Training and Checkpointing

The extended-vocabulary model will be fine-tuned using LoRA. Validation loss will be monitored after each epoch, and the best checkpoint will be selected according to the predefined validation criterion.

Unlike the original baseline training run, this experiment will use a resumable-checkpoint mechanism designed to preserve:

- Adapter and relevant model parameters.
- Optimizer and scheduler states.
- Epoch, batch position, and global optimizer step.
- Best validation loss and early-stopping state.
- Python, NumPy, PyTorch CPU, and CUDA random-number-generator states.
- The data-order and gradient-accumulation information required for accurate recovery.

Periodic recovery checkpoints and a separate best-model checkpoint will be maintained.

## 8.6 Evaluation

After training and checkpoint selection, the extended-vocabulary model will generate translations for the same 994 held-out English test examples used in the baseline evaluation.

We will evaluate using:

- **SacreBLEU:** Word-level n-gram overlap.
- **chrF:** Character n-gram overlap.
- **Validation loss:** For monitoring model learning and checkpoint selection.
- **Tokenization diagnostics:** To measure changes in how Santali words are segmented.
- **Error analysis:** To inspect omissions, extra content, repetition, and translation differences.

The baseline and extended-vocabulary predictions will be evaluated using the same metric configuration. The test set will not be used to select vocabulary entries, tune training settings, or choose checkpoints.

## 8.7 Baseline Reference Results

The original-vocabulary baseline used `ai4bharat/indictrans2-indic-indic-1B` with LoRA for English-to-Santali translation.

| Metric | Baseline result |
|---|---:|
| Best recorded epoch | 7 |
| Best validation loss | 3.0152 |
| Test examples | 994 |
| SacreBLEU | 8.3859 |
| chrF | 41.0353 |
| Exact string match | 0.00% |

The baseline adapter, training history, predictions, evaluation metadata, and metrics have been saved in the `santali_baseline_backup.zip` archive.

## 8.8 Expected Research Contribution

This experiment will test whether explicit vocabulary adaptation can improve low-resource translation beyond ordinary LoRA fine-tuning with a fixed pretrained vocabulary.

The results will help us investigate:

- Whether vocabulary extension changes Santali tokenization efficiency.
- Whether newly added tokens improve translation quality.
- Whether embedding initialization affects the usefulness of the new vocabulary.
- Whether improvements in tokenization correspond to improvements in SacreBLEU and chrF.

No improvement is assumed in advance. The contribution will depend on the experimental results and on whether

In [79]:
# Section 8.9 (Recovery): Load the base model and tokenizer

import gc
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

# Release any existing model objects, if present
for name in ["model", "lora_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load the tokenizer
tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

# Load the original pretrained base model
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    low_cpu_mem_usage=True,
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = model.to(DEVICE)
model.eval()

print("Base model and tokenizer loaded successfully.")
print("Model class:", type(model).__name__)
print("Device:", DEVICE)
print("Model in evaluation mode:", not model.training)
print("Tokenizer class:", type(tok).__name__)
print("Tokenizer length:", len(tok))
print("Source vocabulary size:", tok.src_vocab_size)
print("Target vocabulary size:", tok.tgt_vocab_size)

print("\nBaseline adapter still exists:",
      __import__("pathlib").Path(
          "/kaggle/working/santali_baseline_lora/best_adapter/adapter_model.safetensors"
      ).is_file())

Base model and tokenizer loaded successfully.
Model class: IndicTransForConditionalGeneration
Device: cuda
Model in evaluation mode: True
Tokenizer class: IndicTransTokenizer
Tokenizer length: 122706
Source vocabulary size: 122706
Target vocabulary size: 122672

Baseline adapter still exists: True


In [80]:
# Section 8.9: Inspect vocabulary and embedding compatibility

print("=" * 60)
print("VOCABULARY AND EMBEDDING INSPECTION")
print("=" * 60)

# 1. Tokenizer sizes
print("\n--- Tokenizer ---")
print("Full tokenizer length:", len(tok))
print("Source vocabulary size:", tok.src_vocab_size)
print("Target vocabulary size:", tok.tgt_vocab_size)

# 2. Model configuration
print("\n--- Model configuration ---")
for attr in [
    "vocab_size",
    "encoder_vocab_size",
    "decoder_vocab_size",
    "encoder_embed_dim",
    "decoder_embed_dim",
    "share_decoder_input_output_embed",
]:
    print(f"{attr}: {getattr(model.config, attr, 'Not defined')}")

# 3. Embedding matrix dimensions
encoder_emb = model.model.encoder.embed_tokens.weight
decoder_emb = model.model.decoder.embed_tokens.weight
output_emb = model.lm_head.weight

print("\n--- Embedding matrices ---")
print("Encoder embedding shape:", tuple(encoder_emb.shape))
print("Decoder embedding shape:", tuple(decoder_emb.shape))
print("Output projection shape:", tuple(output_emb.shape))

# 4. Token ID bounds
print("\n--- Vocabulary ID bounds ---")
print("Source IDs expected:", f"0 to {tok.src_vocab_size - 1}")
print("Target IDs expected:", f"0 to {tok.tgt_vocab_size - 1}")
print("Encoder rows:", encoder_emb.shape[0])
print("Decoder rows:", decoder_emb.shape[0])
print("Output rows:", output_emb.shape[0])

# 5. Language tags
print("\n--- Language tags ---")
for tag in ["eng_Latn", "sat_Olck"]:
    print(
        f"{tag}: source ID={tok.src_encoder.get(tag)}, "
        f"target ID={tok.tgt_encoder.get(tag)}"
    )

# 6. Basic compatibility checks
assert encoder_emb.shape[0] == tok.src_vocab_size
assert decoder_emb.shape[0] == tok.tgt_vocab_size
assert output_emb.shape[0] == tok.tgt_vocab_size

print("\nPASS: tokenizer and embedding dimensions are compatible.")
print("No model parameters or vocabulary entries were changed.")

VOCABULARY AND EMBEDDING INSPECTION

--- Tokenizer ---
Full tokenizer length: 122706
Source vocabulary size: 122706
Target vocabulary size: 122672

--- Model configuration ---
vocab_size: 122672
encoder_vocab_size: 122706
decoder_vocab_size: 122672
encoder_embed_dim: 1024
decoder_embed_dim: 1024
share_decoder_input_output_embed: False

--- Embedding matrices ---
Encoder embedding shape: (122706, 1024)
Decoder embedding shape: (122672, 1024)
Output projection shape: (122672, 1024)

--- Vocabulary ID bounds ---
Source IDs expected: 0 to 122705
Target IDs expected: 0 to 122671
Encoder rows: 122706
Decoder rows: 122672
Output rows: 122672

--- Language tags ---
eng_Latn: source ID=4, target ID=None
sat_Olck: source ID=29925, target ID=None

PASS: tokenizer and embedding dimensions are compatible.
No model parameters or vocabulary entries were changed.


In [81]:

import inspect

print("=" * 65)
print("TOKENIZER IMPLEMENTATION INSPECTION")
print("=" * 65)

print("\n--- Tokenizer class ---")
print("Class:", type(tok))
print("Module:", type(tok).__module__)

print("\n--- Vocabulary-related methods ---")
for name in dir(tok):
    if any(term in name.lower() for term in [
        "vocab", "token", "sentencepiece", "spm",
        "encode", "decode", "add"
    ]):
        attr = getattr(tok, name, None)
        if callable(attr):
            try:
                print(f"{name}{inspect.signature(attr)}")
            except (TypeError, ValueError):
                print(name)

print("\n--- Source and target SentencePiece objects ---")
for name in ["src_spm", "tgt_spm"]:
    sp = getattr(tok, name, None)
    print(f"\n{name}: {type(sp).__name__ if sp is not None else 'Not found'}")
    if sp is not None:
        for method in ["get_piece_size", "get_piece", "piece_to_id", "encode", "decode"]:
            if hasattr(sp, method):
                print(f"  {method}: available")

print("\n--- Vocabulary encoder objects ---")
for name in ["src_encoder", "tgt_encoder"]:
    encoder = getattr(tok, name, None)
    print(f"{name}: {type(encoder).__name__ if encoder is not None else 'Not found'}")
    if isinstance(encoder, dict):
        print("  Number of entries:", len(encoder))
        print("  First five entries:", list(encoder.items())[:5])

print("\nInspection only: no tokenizer or model changes made.")


TOKENIZER IMPLEMENTATION INSPECTION

--- Tokenizer class ---
Class: <class 'transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.tokenization_indictrans.IndicTransTokenizer'>
Module: transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.tokenization_indictrans

--- Vocabulary-related methods ---
_add_tokens(new_tokens: Union[List[str], List[tokenizers.AddedToken]], special_tokens: bool = False) -> int
_batch_encode_plus(batch_text_or_text_pairs: Union[List[str], List[Tuple[str, str]], List[List[str]], List[Tuple[List[str], List[str]]], List[List[int]], List[Tuple[List[int], List[int]]]], add_special_tokens: bool = True, padding_strategy: transformers.utils.generic.PaddingStrategy = <PaddingStrategy.DO_NOT_PAD: 'do_not_pad'>, truncation_strategy: transformers.tokenization_utils_base.TruncationStrategy = <TruncationStrategy.DO_NOT_TRUNCATE: 'do_not_truncate'>, max_length: Optional[int] = None, stri

In [82]:

from collections import Counter
import pandas as pd

print("=" * 65)
print("SANTALI TARGET TOKENIZATION BASELINE")
print("=" * 65)

# Retrieve the Santali training data.
# Reuse all_data if available; otherwise load the saved split.
if "all_data" not in globals() or "santali" not in all_data:
    from pathlib import Path

    SPLIT_DIR = Path(
        "/kaggle/input/datasets/sanjeevanijaiswal/"
        "common-tribal-preprocessing/splits"
    )
    santali_train = pd.read_parquet(SPLIT_DIR / "santali_train.parquet")
else:
    santali_train = all_data["santali"]["train"]

print("\nTraining rows:", len(santali_train))
print("Training columns:", santali_train.columns.tolist())

# Confirm the target column rather than assuming its name.
target_candidates = [
    c for c in santali_train.columns
    if "santali" in c.lower()
]
print("Santali-related columns:", target_candidates)

assert len(target_candidates) == 1, (
    "Please inspect the columns above and select the Santali target column."
)
TARGET_COL = target_candidates[0]

# Inspect the actual SentencePiece target vocabulary.
sp = tok.tgt_spm
print("\n--- Existing target SentencePiece vocabulary ---")
print("SentencePiece pieces:", sp.get_piece_size())
print("Tokenizer target vocabulary entries:", len(tok.tgt_encoder))

# Analyze tokenization on a sample first to keep this diagnostic lightweight.
sample_size = min(2000, len(santali_train))
sample_texts = (
    santali_train[TARGET_COL]
    .dropna()
    .astype(str)
    .head(sample_size)
    .tolist()
)

piece_counts = Counter()
word_counts = Counter()
total_pieces = 0
total_words = 0
unk_piece_id = sp.unk_id()
unk_count = 0

for text in sample_texts:
    pieces = sp.encode(text, out_type=str)
    piece_counts.update(pieces)
    total_pieces += len(pieces)

    # Whitespace-delimited words are a simple diagnostic unit here.
    words = text.split()
    word_counts.update(words)
    total_words += len(words)

    unk_count += sum(
        1 for piece in pieces
        if sp.piece_to_id(piece) == unk_piece_id
    )

print("\n--- Sample tokenization statistics ---")
print("Sentences analyzed:", len(sample_texts))
print("Whitespace-delimited words:", total_words)
print("SentencePiece tokens:", total_pieces)
print(
    "Tokens per whitespace-delimited word:",
    round(total_pieces / max(total_words, 1), 3)
)
print("UNK pieces:", unk_count)
print(
    "UNK percentage of SentencePiece tokens:",
    round(100 * unk_count / max(total_pieces, 1), 4)
)

print("\n--- Most frequent existing SentencePiece pieces ---")
for piece, count in piece_counts.most_common(30):
    print(f"{piece!r}: {count}")

print("\n--- Most frequent whitespace-delimited target words ---")
for word, count in word_counts.most_common(30):
    print(f"{word!r}: {count}")

print("\nDiagnostic complete. No tokenizer or model changes made.")


SANTALI TARGET TOKENIZATION BASELINE

Training rows: 17883
Training columns: ['English', 'Santali']
Santali-related columns: ['Santali']

--- Existing target SentencePiece vocabulary ---
SentencePiece pieces: 128000
Tokenizer target vocabulary entries: 122672

--- Sample tokenization statistics ---
Sentences analyzed: 2000
Whitespace-delimited words: 38974
SentencePiece tokens: 56205
Tokens per whitespace-delimited word: 1.442
UNK pieces: 0
UNK percentage of SentencePiece tokens: 0.0

--- Most frequent existing SentencePiece pieces ---
'▁᱾': 1937
'▁ᱨᱮ': 1070
',': 993
'▁ᱫᱚ': 992
'▁ᱨᱮᱭᱟᱜ': 874
'▁ᱟᱨ': 855
'▁ᱠᱚ': 690
'▁ᱤᱧ': 509
'▁ᱞᱟᱹᱜᱤᱫ': 389
'▁ᱠᱟᱱᱟ': 368
'▁ᱢᱤᱫᱴᱟᱝ': 305
'ᱛᱮ': 289
'▁ᱟᱢ': 284
'▁ᱱᱚᱶᱟ': 282
'▁ᱛᱮ': 263
'▁ᱞᱮᱱᱟ': 260
'▁ᱠᱷᱚᱱ': 250
'▁ᱮ': 245
'▁ᱩᱱᱤ': 240
'▁ᱞᱮᱫᱟ': 216
'▁ᱨᱮᱱ': 205
'▁ᱢᱤᱫ': 195
'ᱼ': 193
'ᱨᱮ': 188
'▁ᱡᱮ': 188
'ᱮ': 182
'▁ᱵᱟᱝ': 179
'▁ᱢᱮ': 171
'ᱧ': 170
'▁ᱠᱟᱛᱮ': 164

--- Most frequent whitespace-delimited target words ---
'᱾': 1932
'ᱨᱮ': 986
'ᱫᱚ': 942
'ᱨᱮᱭᱟᱜ': 867
'ᱟᱨ': 827
'

In [83]:

from collections import Counter

# Reuse the word frequencies from the previous diagnostic.
# If the variable is unavailable, calculate them again.
if "word_counts" not in globals():
    from collections import Counter
    word_counts = Counter()
    for text in santali_train["Santali"].dropna().astype(str):
        word_counts.update(text.split())

sp = tok.tgt_spm
candidate_rows = []

for word, frequency in word_counts.most_common(300):
    pieces = sp.encode(word, out_type=str)
    candidate_rows.append({
        "word": word,
        "frequency": frequency,
        "piece_count": len(pieces),
        "existing_pieces": " | ".join(pieces),
        "already_single_piece": len(pieces) == 1,
    })

candidate_df = pd.DataFrame(candidate_rows)

print("Top 300 frequent Santali words: segmentation analysis")
print("Words analyzed:", len(candidate_df))
print(
    "Words represented by one piece:",
    int(candidate_df["already_single_piece"].sum())
)
print(
    "Words split into multiple pieces:",
    int((candidate_df["piece_count"] > 1).sum())
)
print(
    "Mean pieces per word:",
    round(candidate_df["piece_count"].mean(), 3)
)

print("\n--- Most frequent words split into multiple pieces ---")
display(
    candidate_df[candidate_df["piece_count"] > 1]
    .sort_values(["frequency", "piece_count"], ascending=[False, False])
    .head(40)
    .reset_index(drop=True)
)

print("\n--- Most frequent words already represented by one piece ---")
display(
    candidate_df[candidate_df["already_single_piece"]]
    .head(20)
    .reset_index(drop=True)
)

print("\nAnalysis only: no vocabulary or model changes made.")


Top 300 frequent Santali words: segmentation analysis
Words analyzed: 300
Words represented by one piece: 286
Words split into multiple pieces: 14
Mean pieces per word: 1.047

--- Most frequent words split into multiple pieces ---


,word,frequency,piece_count,existing_pieces,already_single_piece
0,ᱤᱧᱟᱹᱜ,91,2,▁ᱤᱧ | ᱟᱹᱜ,False
1,"ᱨᱮ,",41,2,"▁ᱨᱮ | ,",False
2,"ᱡᱮ,",29,2,"▁ᱡᱮ | ,",False
3,"ᱞᱮᱱᱟ,",28,2,"▁ᱞᱮᱱᱟ | ,",False
4,ᱤᱧᱤᱧ,28,2,▁ᱤᱧ | ᱤᱧ,False
5,"ᱠᱟᱱᱟ,",26,2,"▁ᱠᱟᱱᱟ | ,",False
6,"ᱢᱮ,",24,2,"▁ᱢᱮ | ,",False
7,ᱢᱩᱜᱷᱚᱞ,24,2,▁ᱢᱩᱜ | ᱷᱚᱞ,False
8,ᱫᱳᱠᱟᱱ,23,2,▁ᱫᱳ | ᱠᱟᱱ,False
9,"ᱞᱮᱠᱟᱛᱮ,",21,2,"▁ᱞᱮᱠᱟᱛᱮ | ,",False



--- Most frequent words already represented by one piece ---


,word,frequency,piece_count,existing_pieces,already_single_piece
0,᱾,1932,1,▁᱾,True
1,ᱨᱮ,986,1,▁ᱨᱮ,True
2,ᱫᱚ,942,1,▁ᱫᱚ,True
3,ᱨᱮᱭᱟᱜ,867,1,▁ᱨᱮᱭᱟᱜ,True
4,ᱟᱨ,827,1,▁ᱟᱨ,True
5,ᱠᱚ,638,1,▁ᱠᱚ,True
6,ᱤᱧ,365,1,▁ᱤᱧ,True
7,ᱞᱟᱹᱜᱤᱫ,357,1,▁ᱞᱟᱹᱜᱤᱫ,True
8,ᱠᱟᱱᱟ,337,1,▁ᱠᱟᱱᱟ,True
9,ᱢᱤᱫᱴᱟᱝ,304,1,▁ᱢᱤᱫᱴᱟᱝ,True



Analysis only: no vocabulary or model changes made.


In [84]:

import unicodedata
from collections import Counter

sp = tok.tgt_spm

# Count whitespace-delimited words across the full training corpus.
word_counts = Counter()
for text in santali_train["Santali"].dropna().astype(str):
    word_counts.update(text.split())

candidate_rows = []

for word, frequency in word_counts.items():
    # Exclude tokens consisting entirely of punctuation or symbols.
    if all(
        unicodedata.category(char)[0] in {"P", "S"}
        for char in word
    ):
        continue

    pieces = sp.encode(word, out_type=str)

    if len(pieces) > 1 and frequency >= 5:
        candidate_rows.append({
            "word": word,
            "frequency": frequency,
            "piece_count": len(pieces),
            "existing_pieces": " | ".join(pieces),
            "potential_piece_reduction": len(pieces) - 1,
        })

candidate_df = pd.DataFrame(candidate_rows)

if not candidate_df.empty:
    candidate_df = candidate_df.sort_values(
        ["frequency", "potential_piece_reduction"],
        ascending=[False, False]
    ).reset_index(drop=True)

print("FULL TRAINING CORPUS ANALYSIS")
print("=" * 60)
print("Unique whitespace-delimited words:", len(word_counts))
print("Words split into multiple pieces and seen at least 5 times:",
      len(candidate_df))

if not candidate_df.empty:
    print("\nTotal occurrences of these candidate words:",
          int(candidate_df["frequency"].sum()))
    print("Potential token reduction if each were a single piece:",
          int((
              candidate_df["frequency"]
              * candidate_df["potential_piece_reduction"]
          ).sum()))

    print("\n--- Top 50 candidates ---")
    display(candidate_df.head(50))
else:
    print("No candidates met the current criteria.")

print("\nAnalysis only: no tokenizer or model changes made.")


FULL TRAINING CORPUS ANALYSIS
Unique whitespace-delimited words: 41757
Words split into multiple pieces and seen at least 5 times: 3274

Total occurrences of these candidate words: 45812
Potential token reduction if each were a single piece: 60193

--- Top 50 candidates ---


,word,frequency,piece_count,existing_pieces,potential_piece_reduction
0,ᱤᱧᱟᱹᱜ,807,2,▁ᱤᱧ | ᱟᱹᱜ,1
1,"ᱨᱮ,",293,2,"▁ᱨᱮ | ,",1
2,"ᱠᱟᱱᱟ,",286,2,"▁ᱠᱟᱱᱟ | ,",1
3,"ᱡᱮ,",256,2,"▁ᱡᱮ | ,",1
4,"ᱞᱮᱠᱟᱛᱮ,",242,2,"▁ᱞᱮᱠᱟᱛᱮ | ,",1
5,ᱤᱧᱤᱧ,232,2,▁ᱤᱧ | ᱤᱧ,1
6,"ᱞᱮᱱᱟ,",226,2,"▁ᱞᱮᱱᱟ | ,",1
7,"ᱞᱮᱫᱟ,",203,2,"▁ᱞᱮᱫᱟ | ,",1
8,ᱩᱱᱫᱚ,181,2,▁ᱩᱱ | ᱫᱚ,1
9,ᱫᱳᱠᱟᱱ,178,2,▁ᱫᱳ | ᱠᱟᱱ,1



Analysis only: no tokenizer or model changes made.


In [85]:

import unicodedata
import pandas as pd

def is_word_only(text):
    """True if the token contains no punctuation or symbol characters."""
    return bool(text) and all(
        unicodedata.category(char)[0] not in {"P", "S"}
        for char in text
    )

# candidate_df was produced from the full training corpus in the last cell.
assert "candidate_df" in globals(), (
    "Run the full-corpus candidate analysis first."
)

clean_candidates = candidate_df[
    candidate_df["word"].map(is_word_only)
].copy()

print("=" * 65)
print("REFINED SANTALI VOCABULARY CANDIDATES")
print("=" * 65)

print("\nCandidates before excluding punctuation-bearing forms:",
      len(candidate_df))
print("Clean word-form candidates:", len(clean_candidates))
print("Excluded punctuation/symbol-bearing forms:",
      len(candidate_df) - len(clean_candidates))

print("\n--- Candidate counts by minimum training frequency ---")

rows = []
for threshold in [5, 10, 20, 50, 100]:
    subset = clean_candidates[
        clean_candidates["frequency"] >= threshold
    ]
    rows.append({
        "min_frequency": threshold,
        "candidate_types": len(subset),
        "total_occurrences": int(subset["frequency"].sum()),
        "estimated_piece_reduction": int(
            (
                subset["frequency"]
                * subset["potential_piece_reduction"]
            ).sum()
        ),
    })

display(pd.DataFrame(rows))

print("\n--- Top 30 clean candidates occurring at least 20 times ---")
display(
    clean_candidates[
        clean_candidates["frequency"] >= 20
    ]
    .sort_values(
        ["frequency", "potential_piece_reduction"],
        ascending=[False, False]
    )
    .head(30)
    .reset_index(drop=True)
)

print("\nAnalysis only: no vocabulary or model changes made.")


REFINED SANTALI VOCABULARY CANDIDATES

Candidates before excluding punctuation-bearing forms: 3274
Clean word-form candidates: 3045
Excluded punctuation/symbol-bearing forms: 229

--- Candidate counts by minimum training frequency ---


,min_frequency,candidate_types,total_occurrences,estimated_piece_reduction
0,5,3045,40747,54551
1,10,1232,29163,38005
2,20,451,18853,23896
3,50,97,8520,9796
4,100,21,3530,4009



--- Top 30 clean candidates occurring at least 20 times ---


,word,frequency,piece_count,existing_pieces,potential_piece_reduction
0,ᱤᱧᱟᱹᱜ,807,2,▁ᱤᱧ | ᱟᱹᱜ,1
1,ᱤᱧᱤᱧ,232,2,▁ᱤᱧ | ᱤᱧ,1
2,ᱩᱱᱫᱚ,181,2,▁ᱩᱱ | ᱫᱚ,1
3,ᱫᱳᱠᱟᱱ,178,2,▁ᱫᱳ | ᱠᱟᱱ,1
4,ᱢᱩᱜᱷᱚᱞ,169,2,▁ᱢᱩᱜ | ᱷᱚᱞ,1
5,ᱱᱩᱱᱟᱹᱜ,155,2,▁ᱱᱩ | ᱱᱟᱹᱜ,1
6,ᱟᱞᱮᱭᱟᱜ,154,2,▁ᱟᱞᱮ | ᱭᱟᱜ,1
7,ᱤᱧᱨᱮᱱ,147,2,▁ᱤᱧ | ᱨᱮᱱ,1
8,ᱟᱹᱲᱤᱥ,138,2,▁ᱟᱹᱲ | ᱤᱥ,1
9,ᱦᱟᱦᱟᱲᱟᱜ,134,3,▁ᱦᱟᱦ | ᱟᱲ | ᱟᱜ,2



Analysis only: no vocabulary or model changes made.



## Section 8.9 — Santali Vocabulary Extension: Findings and Technical Validation

### 1. Objective

The objective of this investigation is to determine whether extending the pretrained IndicTrans2 vocabulary with frequent Santali words or subwords can improve target-side tokenization and, ultimately, English-to-Santali translation quality.

The experiment compares the existing tokenizer and embeddings against a vocabulary-extended configuration while preserving the original vocabulary IDs and pretrained weights. Vocabulary extension is a hypothesis to evaluate, not an assumed improvement.

### 2. Baseline Model and Vocabulary Structure

The current experiment uses `ai4bharat/indictrans2-indic-indic-1B` with the custom `IndicTransTokenizer`.

| Component | Observed configuration |
|---|---:|
| Model configuration vocabulary size | 122,672 |
| Encoder embedding rows | 122,706 |
| Decoder embedding rows | 122,672 |
| Output projection rows | 122,672 |
| Encoder/decoder embedding dimension | 1,024 |
| Source tokenizer vocabulary size | 122,706 |
| Target tokenizer vocabulary size | 122,672 |
| Santali target-side UNK rate in the inspected sample | 0% |

The source and target vocabularies have different sizes. The decoder embedding and output projection are separate tensors, despite the configuration's `tie_word_embeddings=True` setting. They must not be manually tied or resized using assumptions that apply to a standard encoder-decoder model.

The Santali target SentencePiece model contains 128,000 pieces, while the target tokenizer vocabulary mapping contains 122,672 entries. These counts refer to different representations and should not be treated as interchangeable.

### 3. Santali Tokenization Analysis

The target SentencePiece tokenizer was examined using Santali training data.

#### Sample analysis: 2,000 sentences

| Measurement | Observed result |
|---|---:|
| Whitespace-delimited words | 38,974 |
| SentencePiece tokens | 56,205 |
| Average SentencePiece tokens per whitespace word | 1.442 |
| UNK pieces observed | 0 |

The average of 1.442 tokens per whitespace-delimited word indicates that some words are segmented into multiple subword pieces. However, this alone does not demonstrate a vocabulary coverage problem: SentencePiece segmentation is expected, and a word split into multiple valid pieces is not necessarily an unknown word.

#### Full Santali training-corpus candidate analysis

The training corpus contains 41,757 unique whitespace-delimited word forms.

Candidate counts were calculated for words that occur at least a specified number of times and are segmented into more than one SentencePiece piece. A refined analysis excluded forms containing Unicode punctuation or symbol characters.

| Minimum frequency | Candidate forms | Total occurrences | Estimated maximum token reduction |
|---:|---:|---:|---:|
| 5 | 3,045 | 40,747 | 54,551 |
| 10 | 1,232 | 29,163 | 38,005 |
| 20 | 451 | 18,853 | 23,896 |
| 50 | 97 | 8,520 | 9,796 |
| 100 | 21 | 3,530 | 4,009 |

The token-reduction estimates are theoretical upper bounds. They assume each selected multi-piece form becomes one token and do not account for context, punctuation, word-boundary behavior, or the effect of the tokenizer implementation.

A provisional candidate pool of 97 forms at a minimum frequency of 50 is a reasonable starting point for a controlled experiment. The threshold should be fixed before comparing model performance.

### 4. What the Candidate Analysis Establishes

The investigation establishes that:

1. Some frequent Santali word forms are segmented into multiple SentencePiece pieces.
2. A frequency-based candidate list can be extracted from the training corpus.
3. A smaller candidate pool can prioritize frequently occurring forms rather than adding thousands of rare forms.
4. The existing tokenizer produced no UNK pieces in the inspected 2,000-sentence sample, so the primary motivation for extension is reducing fragmentation, not correcting demonstrated UNK coverage failures.
5. Reduced token counts are a potential efficiency benefit, not proof of improved translation quality.

The candidate list must be constructed from training data only. Validation and test data must not be used to select vocabulary entries.




### 7. Required Validation Before Model Modification

The next steps are:

1. Fix or bypass the incompatible vocabulary-inspection method without changing the pretrained tokenizer's original IDs.
2. Test candidate encoding before and after addition using the actual target-side encoding path.
3. Confirm that a candidate receives a unique ID and is emitted as exactly one target token.
4. Verify that ordinary words, special tokens, source-side English encoding, and language-tag handling remain unchanged.
5. Inspect the custom model's embedding-resizing and output-projection behavior before expanding any model tensor.
6. Initialize added vectors using a documented strategy, such as compositional initialization from the original subword embeddings, and compare it with an appropriate alternative.
7. Verify that the extended model can perform a finite forward and backward pass, with valid gradients for the new vectors and LoRA parameters.
8. Train the extended-vocabulary model under a controlled setup and evaluate it on the same held-out test set and with the same decoding and evaluation configuration as the original-vocabulary baseline.

### 8. Experimental Comparison and Success Criteria

The primary comparison is:

- **Baseline:** Original vocabulary + LoRA fine-tuning.
- **Extended-vocabulary experiment:** Extended vocabulary + documented new-vector initialization + LoRA fine-tuning.

The following must be measured separately:

- SacreBLEU and chrF on the held-out test set.
- Validation loss and selected checkpoint epoch.
- Average target-side token count and tokenizer fertility.
- Candidate-token frequency and actual reduction in target token count.
- Training stability, memory usage, and inference cost.

A successful vocabulary extension should preserve valid tokenization and model execution, demonstrate the intended reduction in token fragmentation, and ideally improve held-out translation quality. If tokenization improves but translation metrics do not, that is still a useful negative result and should be reported accurately.

**Current status:** Candidate extraction and initial tokenization analysis are complete. Safe target-side token addition, model embedding expansion, and a controlled extended-vocabulary comparison remain to be validated.


In [86]:

import inspect

print("=" * 65)
print("CUSTOM TOKENIZER IMPLEMENTATION")
print("=" * 65)

for method_name in [
    "_tgt_tokenize",
    "_tokenize",
    "_convert_token_to_id",
    "add_tokens",
    "_add_tokens",
    "get_tgt_vocab",
    "get_vocab",
    "save_vocabulary",
]:
    method = getattr(type(tok), method_name, None)

    print(f"\n{'-' * 65}")
    print(f"METHOD: {method_name}")

    if method is None:
        print("Not defined on this class.")
        continue

    try:
        print(inspect.getsource(method))
    except (OSError, TypeError) as exc:
        print("Source unavailable:", exc)

print("\nInspection only: no tokenizer or model changes made.")


CUSTOM TOKENIZER IMPLEMENTATION

-----------------------------------------------------------------
METHOD: _tgt_tokenize
    def _tgt_tokenize(self, text: str) -> List[str]:
        return self.spm.EncodeAsPieces(text)


-----------------------------------------------------------------
METHOD: _tokenize
    def _tokenize(self, text, **kwargs):
        """
        Converts a string into a sequence of tokens (string), using the tokenizer. Split in words for word-based
        vocabulary or sub-words for sub-word-based vocabularies (BPE/SentencePieces/WordPieces).

        Do NOT take care of added tokens.
        """
        raise NotImplementedError


-----------------------------------------------------------------
METHOD: _convert_token_to_id
    def _convert_token_to_id(self, token: str) -> int:
        return self.encoder.get(token, self.unk_token_id)


-----------------------------------------------------------------
METHOD: add_tokens
    def add_tokens(
        self, new_tokens: 

In [88]:
# Inspect target vocabulary safely without modifying the working tokenizer.

from transformers import AutoTokenizer

test_tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token
)

candidate_word = "ᱤᱧᱟᱹᱜ"

# Read the target vocabulary directly to avoid the get_tgt_vocab() bug.
target_vocab = dict(test_tok.tgt_encoder)

print("Candidate word:", candidate_word)
print("Target encoder size:", len(target_vocab))
print("Source encoder size:", len(test_tok.src_encoder))
print("Tokenizer length:", len(test_tok))
print("SentencePiece pieces:", test_tok._tgt_tokenize(candidate_word))
print("Original target encoder ID:", target_vocab.get(candidate_word))
print("Original tokenizer UNK ID:", test_tok.unk_token_id)

Candidate word: ᱤᱧᱟᱹᱜ
Target encoder size: 122672
Source encoder size: 122706
Tokenizer length: 122706
SentencePiece pieces: ['▁ᱤᱧ', 'ᱟᱹᱜ']
Original target encoder ID: None
Original tokenizer UNK ID: 3


In [89]:
# Test target-side token addition on the isolated tokenizer only.

from transformers import AddedToken

candidate_word = "ᱤᱧᱟᱹᱜ"

original_target_size = len(test_tok.tgt_encoder)
original_tokenizer_length = len(test_tok)

print("Before addition:")
print("Target encoder size:", original_target_size)
print("Tokenizer length:", original_tokenizer_length)
print("Target pieces:", test_tok._tgt_tokenize(candidate_word))

# Add the candidate as a standalone token.
num_added = test_tok.add_tokens(
    [AddedToken(candidate_word, normalized=False, special=False)]
)

print("\nAfter addition:")
print("Number of tokens added:", num_added)
print("Target encoder size:", len(test_tok.tgt_encoder))
print("Tokenizer length:", len(test_tok))

# Compare ordinary target encoding before/after addition.
encoded = test_tok(
    text_target=[candidate_word],
    add_special_tokens=False
)

print("Target token IDs:", encoded["input_ids"])
print("Target token count:", len(encoded["input_ids"][0]))
print("New token ID:", test_tok.convert_tokens_to_ids(candidate_word))
print("Target encoder contains candidate:", candidate_word in test_tok.tgt_encoder)

Before addition:
Target encoder size: 122672
Tokenizer length: 122706
Target pieces: ['▁ᱤᱧ', 'ᱟᱹᱜ']

After addition:
Number of tokens added: 1
Target encoder size: 122672
Tokenizer length: 122707
Target token IDs: [[122706]]
Target token count: 1
New token ID: 122706
Target encoder contains candidate: False


In [90]:
import inspect

print("Model class:", type(model).__name__)

print("\n--- resize_token_embeddings ---")
try:
    print(inspect.getsource(type(model).resize_token_embeddings))
except (OSError, TypeError, AttributeError) as e:
    print("Could not retrieve class implementation:", repr(e))

print("\n--- get_output_embeddings ---")
try:
    print(inspect.getsource(type(model).get_output_embeddings))
except (OSError, TypeError, AttributeError) as e:
    print("Could not retrieve class implementation:", repr(e))

print("\n--- set_output_embeddings ---")
try:
    print(inspect.getsource(type(model).set_output_embeddings))
except (OSError, TypeError, AttributeError) as e:
    print("Could not retrieve class implementation:", repr(e))

Model class: IndicTransForConditionalGeneration

--- resize_token_embeddings ---
    def resize_token_embeddings(
        self,
        new_num_tokens: Optional[int] = None,
        pad_to_multiple_of: Optional[int] = None,
        mean_resizing: bool = True,
    ) -> nn.Embedding:
        """
        Resizes input token embeddings matrix of the model if `new_num_tokens != config.vocab_size`.

        Takes care of tying weights embeddings afterwards if the model class has a `tie_weights()` method.

        Arguments:
            new_num_tokens (`int`, *optional*):
                The new number of tokens in the embedding matrix. Increasing the size will add newly initialized
                vectors at the end. Reducing the size will remove vectors from the end. If not provided or `None`, just
                returns a pointer to the input tokens `torch.nn.Embedding` module of the model without doing anything.
            pad_to_multiple_of (`int`, *optional*):
                If set w

In [91]:
import inspect

cls = type(model)

for method_name in [
    "_resize_token_embeddings",
    "_get_resized_embeddings",
    "tie_weights",
    "_tie_weights",
]:
    print(f"\n{'=' * 20} {method_name} {'=' * 20}")

    method = getattr(cls, method_name, None)

    if method is None:
        print("Not defined directly on this class.")
        continue

    try:
        print(inspect.getsource(method))
    except (OSError, TypeError, AttributeError):
        print("Source unavailable; method may be inherited.")


==================== _resize_token_embeddings ====================
    def _resize_token_embeddings(self, new_num_tokens, pad_to_multiple_of=None, mean_resizing=True):
        old_embeddings = self.get_input_embeddings()
        new_embeddings = self._get_resized_embeddings(
            old_embeddings, new_num_tokens, pad_to_multiple_of, mean_resizing
        )
        if hasattr(old_embeddings, "_hf_hook"):
            hook = old_embeddings._hf_hook
            add_hook_to_module(new_embeddings, hook)
        old_embeddings_requires_grad = old_embeddings.weight.requires_grad
        new_embeddings.requires_grad_(old_embeddings_requires_grad)
        self.set_input_embeddings(new_embeddings)
        is_quantized = hasattr(self, "hf_quantizer") and self.hf_quantizer is not None

        # Update new_num_tokens with the actual size of new_embeddings
        if pad_to_multiple_of is not None:
            if is_deepspeed_zero3_enabled() and not is_quantized:
                import deepspe

In [92]:
import torch

encoder_emb = model.get_input_embeddings()
decoder_emb = model.model.decoder.embed_tokens
output_emb = model.get_output_embeddings()

print("Encoder embedding shape:", tuple(encoder_emb.weight.shape))
print("Decoder embedding shape:", tuple(decoder_emb.weight.shape))
print("Output projection shape:", tuple(output_emb.weight.shape))

print("\nVocabulary configuration:")
print("config.vocab_size:", model.config.vocab_size)
print("config.encoder_vocab_size:", model.config.encoder_vocab_size)
print("config.decoder_vocab_size:", model.config.decoder_vocab_size)
print("config.tie_word_embeddings:", model.config.tie_word_embeddings)
print(
    "config.share_decoder_input_output_embed:",
    model.config.share_decoder_input_output_embed
)

print("\nWeight relationships:")
print(
    "Decoder and output projection share storage:",
    decoder_emb.weight.data_ptr() == output_emb.weight.data_ptr()
)
print(
    "Decoder and output projection values identical:",
    torch.equal(decoder_emb.weight, output_emb.weight)
)

print("\nTokenizer:")
print("Original target vocabulary size:", len(test_tok.tgt_encoder))
print("Candidate added-token ID:", test_tok.convert_tokens_to_ids("ᱤᱧᱟᱹᱜ"))
print("Candidate ID is within decoder embedding range:",
      test_tok.convert_tokens_to_ids("ᱤᱧᱟᱹᱜ") < decoder_emb.num_embeddings)

Encoder embedding shape: (122706, 1024)
Decoder embedding shape: (122672, 1024)
Output projection shape: (122672, 1024)

Vocabulary configuration:
config.vocab_size: 122672
config.encoder_vocab_size: 122706
config.decoder_vocab_size: 122672
config.tie_word_embeddings: True
config.share_decoder_input_output_embed: False

Weight relationships:
Decoder and output projection share storage: False
Decoder and output projection values identical: False

Tokenizer:
Original target vocabulary size: 122672
Candidate added-token ID: 122706
Candidate ID is within decoder embedding range: False


In [93]:
import copy
import torch
import torch.nn as nn

# Snapshot the original target layers for an isolated resizing test.
test_decoder = copy.deepcopy(model.model.decoder.embed_tokens).cpu()
test_lm_head = copy.deepcopy(model.lm_head).cpu()

old_decoder_weights = test_decoder.weight.detach().clone()
old_lm_head_weights = test_lm_head.weight.detach().clone()

old_target_size = test_decoder.num_embeddings
new_target_size = old_target_size + 1

# Expand the decoder embedding and preserve all existing rows.
expanded_decoder = nn.Embedding(
    new_target_size,
    test_decoder.embedding_dim,
    padding_idx=test_decoder.padding_idx,
    dtype=test_decoder.weight.dtype,
    device="cpu",
)
expanded_decoder.weight.data[:old_target_size].copy_(old_decoder_weights)

# Expand the output projection and preserve all existing rows.
expanded_lm_head = nn.Linear(
    test_lm_head.in_features,
    new_target_size,
    bias=test_lm_head.bias is not None,
    dtype=test_lm_head.weight.dtype,
    device="cpu",
)

expanded_lm_head.weight.data[:old_target_size].copy_(old_lm_head_weights)

if test_lm_head.bias is not None:
    expanded_lm_head.bias.data[:old_target_size].copy_(
        test_lm_head.bias.detach().cpu()
    )

print("Original target vocabulary size:", old_target_size)
print("Test target vocabulary size:", new_target_size)
print("Expanded decoder shape:", tuple(expanded_decoder.weight.shape))
print("Expanded output projection shape:", tuple(expanded_lm_head.weight.shape))

print(
    "All original decoder rows preserved:",
    torch.equal(
        expanded_decoder.weight[:old_target_size],
        old_decoder_weights
    )
)
print(
    "All original output-projection rows preserved:",
    torch.equal(
        expanded_lm_head.weight[:old_target_size],
        old_lm_head_weights
    )
)
print(
    "Original encoder remains unchanged:",
    tuple(model.get_input_embeddings().weight.shape) == (122706, 1024)
)

Original target vocabulary size: 122672
Test target vocabulary size: 122673
Expanded decoder shape: (122673, 1024)
Expanded output projection shape: (122673, 1024)
All original decoder rows preserved: True
All original output-projection rows preserved: True
Original encoder remains unchanged: True


In [94]:
import torch

candidate_word = "ᱤᱧᱟᱹᱜ"

# Existing SentencePiece decomposition.
pieces = test_tok._tgt_tokenize(candidate_word)
piece_ids = [test_tok.tgt_encoder[p] for p in pieces if p in test_tok.tgt_encoder]

print("Candidate:", candidate_word)
print("Pieces:", pieces)
print("Piece IDs:", piece_ids)

assert len(piece_ids) == len(pieces), (
    "At least one piece is missing from the target encoder vocabulary."
)
assert all(0 <= idx < old_target_size for idx in piece_ids)

with torch.no_grad():
    # Initialize the added decoder embedding from the existing piece embeddings.
    expanded_decoder.weight[old_target_size].copy_(
        test_decoder.weight[piece_ids].mean(dim=0)
    )

    # Initialize the added output-projection row from the existing piece rows.
    expanded_lm_head.weight[old_target_size].copy_(
        test_lm_head.weight[piece_ids].mean(dim=0)
    )

print("\nInitialization checks:")
print(
    "Decoder new row matches piece-embedding mean:",
    torch.allclose(
        expanded_decoder.weight[old_target_size],
        test_decoder.weight[piece_ids].mean(dim=0)
    )
)
print(
    "Output new row matches piece-row mean:",
    torch.allclose(
        expanded_lm_head.weight[old_target_size],
        test_lm_head.weight[piece_ids].mean(dim=0)
    )
)
print(
    "Existing decoder rows unchanged:",
    torch.equal(expanded_decoder.weight[:old_target_size], old_decoder_weights)
)
print(
    "Existing output rows unchanged:",
    torch.equal(expanded_lm_head.weight[:old_target_size], old_lm_head_weights)
)

Candidate: ᱤᱧᱟᱹᱜ
Pieces: ['▁ᱤᱧ', 'ᱟᱹᱜ']
Piece IDs: [76527, 89023]

Initialization checks:
Decoder new row matches piece-embedding mean: True
Output new row matches piece-row mean: True
Existing decoder rows unchanged: True
Existing output rows unchanged: True


In [95]:
import inspect

print("Model forward signature:")
print(inspect.signature(model.forward))

print("\nDecoder forward signature:")
print(inspect.signature(model.model.decoder.forward))

print("\nModel generation preparation method:")
try:
    print(inspect.signature(model.prepare_inputs_for_generation))
except (AttributeError, TypeError, ValueError) as e:
    print("Unavailable:", repr(e))

Model forward signature:
(input_ids: Optional[torch.LongTensor] = None, attention_mask: Optional[torch.Tensor] = None, decoder_input_ids: Optional[torch.LongTensor] = None, decoder_attention_mask: Optional[torch.LongTensor] = None, head_mask: Optional[torch.Tensor] = None, decoder_head_mask: Optional[torch.Tensor] = None, cross_attn_head_mask: Optional[torch.Tensor] = None, encoder_outputs: Optional[Tuple[Tuple[torch.FloatTensor]]] = None, past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None, inputs_embeds: Optional[torch.FloatTensor] = None, decoder_inputs_embeds: Optional[torch.FloatTensor] = None, labels: Optional[torch.LongTensor] = None, use_cache: Optional[bool] = None, output_attentions: Optional[bool] = None, output_hidden_states: Optional[bool] = None, return_dict: Optional[bool] = None) -> Union[Tuple[torch.Tensor], transformers.modeling_outputs.Seq2SeqLMOutput]

Decoder forward signature:
(input_ids: Optional[torch.Tensor] = None, attention_mask: Optional[torch.

In [97]:
import torch

model.eval()

source_text = "Hello, how are you?"
formatted_source = f"eng_Latn sat_Olck {source_text}"

source_batch = tok(
    formatted_source,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

source_batch = {
    key: value.to(DEVICE)
    for key, value in source_batch.items()
    if isinstance(value, torch.Tensor)
}

# Use the original SentencePiece decomposition for the target.
candidate_word = "ᱤᱧᱟᱹᱜ"
target_pieces = test_tok._tgt_tokenize(candidate_word)
target_ids = [test_tok.tgt_encoder[p] for p in target_pieces]

decoder_input_ids = torch.tensor(
    [[tok.pad_token_id] + target_ids],
    dtype=torch.long,
    device=DEVICE,
)

with torch.inference_mode():
    outputs = model(
        input_ids=source_batch["input_ids"],
        attention_mask=source_batch.get("attention_mask"),
        decoder_input_ids=decoder_input_ids,
        use_cache=False,
        return_dict=True,
    )

print("Formatted source:", formatted_source)
print("Source input shape:", tuple(source_batch["input_ids"].shape))
print("Decoder input shape:", tuple(decoder_input_ids.shape))
print("Logits shape:", tuple(outputs.logits.shape))
print("Configured decoder vocabulary:", model.config.decoder_vocab_size)
print("Actual output vocabulary:", outputs.logits.shape[-1])
print("Logits finite:", bool(torch.isfinite(outputs.logits).all()))

Formatted source: eng_Latn sat_Olck Hello, how are you?
Source input shape: (1, 10)
Decoder input shape: (1, 3)
Logits shape: (1, 3, 122672)
Configured decoder vocabulary: 122672
Actual output vocabulary: 122672
Logits finite: True


In [98]:
import copy
import torch
import torch.nn as nn

candidate_word = "ᱤᱧᱟᱹᱜ"
old_vocab_size = model.model.decoder.embed_tokens.num_embeddings
new_vocab_size = old_vocab_size + 1

# Create an isolated model copy on CPU for structural validation.
test_model = copy.deepcopy(model).cpu()
test_model.eval()

# Preserve original target-layer weights.
old_decoder = test_model.model.decoder.embed_tokens
old_lm_head = test_model.lm_head

old_decoder_weights = old_decoder.weight.detach().clone()
old_lm_weights = old_lm_head.weight.detach().clone()

# Expand decoder embeddings.
new_decoder = nn.Embedding(
    new_vocab_size,
    old_decoder.embedding_dim,
    padding_idx=old_decoder.padding_idx,
    dtype=old_decoder.weight.dtype,
)
new_decoder.weight.data[:old_vocab_size].copy_(old_decoder_weights)
new_decoder.weight.data[old_vocab_size].copy_(
    old_decoder_weights[[76527, 89023]].mean(dim=0)
)

# Expand output projection.
new_lm_head = nn.Linear(
    old_lm_head.in_features,
    new_vocab_size,
    bias=old_lm_head.bias is not None,
    dtype=old_lm_head.weight.dtype,
)
new_lm_head.weight.data[:old_vocab_size].copy_(old_lm_weights)
new_lm_head.weight.data[old_vocab_size].copy_(
    old_lm_weights[[76527, 89023]].mean(dim=0)
)

if old_lm_head.bias is not None:
    new_lm_head.bias.data[:old_vocab_size].copy_(old_lm_head.bias.detach())
    new_lm_head.bias.data[old_vocab_size].zero_()

# Install the expanded layers only in the isolated copy.
test_model.model.decoder.embed_tokens = new_decoder
test_model.set_output_embeddings(new_lm_head)
test_model.config.decoder_vocab_size = new_vocab_size
test_model.config.vocab_size = new_vocab_size
test_model.vocab_size = new_vocab_size

# Check preservation and run a forward pass with the new target token ID.
assert torch.equal(
    test_model.model.decoder.embed_tokens.weight[:old_vocab_size],
    old_decoder_weights,
)
assert torch.equal(
    test_model.lm_head.weight[:old_vocab_size],
    old_lm_weights,
)

source_cpu = {k: v.cpu() for k, v in source_batch.items()}
decoder_ids_new = torch.tensor([[tok.pad_token_id, old_vocab_size, 2]])

with torch.inference_mode():
    extended_outputs = test_model(
        input_ids=source_cpu["input_ids"],
        attention_mask=source_cpu.get("attention_mask"),
        decoder_input_ids=decoder_ids_new,
        use_cache=False,
        return_dict=True,
    )

print("New target token ID:", old_vocab_size)
print("Decoder embedding shape:", tuple(test_model.model.decoder.embed_tokens.weight.shape))
print("Output projection shape:", tuple(test_model.lm_head.weight.shape))
print("Extended logits shape:", tuple(extended_outputs.logits.shape))
print("Original decoder rows preserved:", torch.equal(
    test_model.model.decoder.embed_tokens.weight[:old_vocab_size],
    old_decoder_weights
))
print("Original output rows preserved:", torch.equal(
    test_model.lm_head.weight[:old_vocab_size],
    old_lm_weights
))
print("Logits finite:", bool(torch.isfinite(extended_outputs.logits).all()))
print("Working model unchanged:", model.model.decoder.embed_tokens.num_embeddings == old_vocab_size)

New target token ID: 122672
Decoder embedding shape: (122673, 1024)
Output projection shape: (122673, 1024)
Extended logits shape: (1, 3, 122673)
Original decoder rows preserved: True
Original output rows preserved: True
Logits finite: True
Working model unchanged: True


In [99]:
import inspect

print("Tokenizer class:", type(test_tok).__name__)
print("Tokenizer length:", len(test_tok))
print("Target encoder size:", len(test_tok.tgt_encoder))
print("Added vocabulary entry:", test_tok.get_added_vocab())

print("\nTarget vocabulary save method:")
print(inspect.signature(test_tok.save_vocabulary))

print("\nTokenizer save method:")
print(inspect.signature(test_tok.save_pretrained))

print("\nTarget encoder candidate present:",
      candidate_word in test_tok.tgt_encoder)

print("Candidate token ID:",
      test_tok.convert_tokens_to_ids(candidate_word))

Tokenizer class: IndicTransTokenizer
Tokenizer length: 122707
Target encoder size: 122672
Added vocabulary entry: {'<s>': 0, '<pad>': 1, '</s>': 2, '<unk>': 3, 'ᱤᱧᱟᱹᱜ': 122706}

Target vocabulary save method:
(save_directory: str, filename_prefix: Optional[str] = None) -> Tuple[str, ...]

Tokenizer save method:
(save_directory: Union[str, os.PathLike], legacy_format: Optional[bool] = None, filename_prefix: Optional[str] = None, push_to_hub: bool = False, **kwargs) -> Tuple[str]

Target encoder candidate present: False
Candidate token ID: 122706


In [100]:
from transformers import AutoTokenizer
import inspect

tokenizer_probe = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("Tokenizer class:", type(tokenizer_probe).__name__)

print("\nTarget encoder/decoder methods:")
for method_name in [
    "_tgt_tokenize",
    "_convert_token_to_id",
    "convert_tokens_to_ids",
    "convert_ids_to_tokens",
    "decode",
    "batch_decode",
    "save_vocabulary",
]:
    method = getattr(tokenizer_probe, method_name, None)
    if method is not None:
        try:
            print(f"\n{method_name}{inspect.signature(method)}")
        except (TypeError, ValueError):
            print(f"\n{method_name}: signature unavailable")

print("\nTarget encoder type:", type(tokenizer_probe.tgt_encoder).__name__)
print("Target SentencePiece model type:", type(tokenizer_probe.tgt_spm).__name__)

probe_text = "ᱤᱧᱟᱹᱜ"
probe_pieces = tokenizer_probe._tgt_tokenize(probe_text)
probe_ids = [tokenizer_probe.tgt_encoder.get(p, tokenizer_probe.unk_token_id)
             for p in probe_pieces]

print("\nProbe text:", probe_text)
print("Original target pieces:", probe_pieces)
print("Original target IDs:", probe_ids)
print("Target decode:", tokenizer_probe.batch_decode(
    [probe_ids], skip_special_tokens=True
))

Tokenizer class: IndicTransTokenizer

Target encoder/decoder methods:

_tgt_tokenize(text: str) -> List[str]

_convert_token_to_id(token: str) -> int

convert_tokens_to_ids(tokens: Union[str, List[str]]) -> Union[int, List[int]]

convert_ids_to_tokens(ids: Union[int, List[int]], skip_special_tokens: bool = False) -> Union[str, List[str]]

decode(token_ids: Union[int, List[int], ForwardRef('np.ndarray'), ForwardRef('torch.Tensor'), ForwardRef('tf.Tensor')], skip_special_tokens: bool = False, clean_up_tokenization_spaces: bool = None, **kwargs) -> str

batch_decode(sequences: Union[List[int], List[List[int]], ForwardRef('np.ndarray'), ForwardRef('torch.Tensor'), ForwardRef('tf.Tensor')], skip_special_tokens: bool = False, clean_up_tokenization_spaces: bool = None, **kwargs) -> List[str]

save_vocabulary(save_directory: str, filename_prefix: Optional[str] = None) -> Tuple[str, ...]

Target encoder type: dict
Target SentencePiece model type: SentencePieceProcessor

Probe text: ᱤᱧᱟᱹᱜ
Origin

In [101]:
import inspect

for method_name in [
    "convert_tokens_to_ids",
    "convert_ids_to_tokens",
    "decode",
    "_convert_token_to_id",
    "_convert_id_to_token",
]:
    method = getattr(type(tokenizer_probe), method_name, None)
    if method is not None:
        print(f"\n{'=' * 20} {method_name} {'=' * 20}")
        try:
            print(inspect.getsource(method))
        except (OSError, TypeError):
            print("Source unavailable; method signature:",
                  inspect.signature(method))

print("\nTarget encoder sample:")
print(list(tokenizer_probe.tgt_encoder.items())[:8])

print("\nSpecial token IDs:")
for token_name in ["bos_token_id", "pad_token_id",
                   "eos_token_id", "unk_token_id"]:
    print(token_name, getattr(tokenizer_probe, token_name, None))


==================== convert_tokens_to_ids ====================
    def convert_tokens_to_ids(self, tokens: Union[str, List[str]]) -> Union[int, List[int]]:
        """
        Converts a token string (or a sequence of tokens) in a single integer id (or a sequence of ids), using the
        vocabulary.

        Args:
            tokens (`str` or `List[str]`): One or several token(s) to convert to token id(s).

        Returns:
            `int` or `List[int]`: The token id or list of token ids.
        """
        if tokens is None:
            return None

        if isinstance(tokens, str):
            return self._convert_token_to_id_with_added_voc(tokens)

        ids = []
        for token in tokens:
            ids.append(self._convert_token_to_id_with_added_voc(token))
        return ids


==================== convert_ids_to_tokens ====================
    def convert_ids_to_tokens(
        self, ids: Union[int, List[int]], skip_special_tokens: bool = False
    ) -> Union[str, 

In [102]:
import inspect

for method_name in [
    "_tokenize",
    "_decode",
    "convert_tokens_to_string",
    "_convert_token_to_id_with_added_voc",
    "get_src_vocab",
    "get_tgt_vocab",
]:
    method = getattr(type(tokenizer_probe), method_name, None)

    if method is not None:
        print(f"\n{'=' * 18} {method_name} {'=' * 18}")
        try:
            print(inspect.getsource(method))
        except (OSError, TypeError):
            print("Source unavailable")


================== _tokenize ==================
    def _tokenize(self, text, **kwargs):
        """
        Converts a string into a sequence of tokens (string), using the tokenizer. Split in words for word-based
        vocabulary or sub-words for sub-word-based vocabularies (BPE/SentencePieces/WordPieces).

        Do NOT take care of added tokens.
        """
        raise NotImplementedError


================== _decode ==================
    def _decode(
        self,
        token_ids: Union[int, List[int]],
        skip_special_tokens: bool = False,
        clean_up_tokenization_spaces: bool = None,
        spaces_between_special_tokens: bool = True,
        **kwargs,
    ) -> str:
        self._switch_to_target_mode()
        decoded_token_ids = super()._decode(
            token_ids=token_ids,
            skip_special_tokens=skip_special_tokens,
            clean_up_tokenization_spaces=clean_up_tokenization_spaces,
            spaces_between_special_tokens=spaces_between_spe

In [103]:
import inspect

print("Tokenizer mode and vocabulary attributes:")
for attr in [
    "encoder",
    "decoder",
    "src_encoder",
    "tgt_encoder",
    "src_decoder",
    "tgt_decoder",
    "src_vocab_size",
    "tgt_vocab_size",
    "cur_encoder",
    "cur_decoder",
]:
    print(f"{attr}:",
          type(getattr(tokenizer_probe, attr, None)).__name__,
          "present =", hasattr(tokenizer_probe, attr))

print("\nMode-switching methods:")
for method_name in [
    "_switch_to_input_mode",
    "_switch_to_target_mode",
]:
    method = getattr(type(tokenizer_probe), method_name, None)
    if method is not None:
        try:
            print(f"\n{method_name}:")
            print(inspect.getsource(method))
        except (OSError, TypeError):
            print(method_name, "source unavailable")

print("\nOriginal vocabulary sizes:")
print("Source:", len(tokenizer_probe.src_encoder))
print("Target:", len(tokenizer_probe.tgt_encoder))
print("Decoder:", len(tokenizer_probe.tgt_decoder))

print("\nTokenizer's current mode:")
print("encoder is source encoder:",
      tokenizer_probe.encoder is tokenizer_probe.src_encoder)
print("encoder is target encoder:",
      tokenizer_probe.encoder is tokenizer_probe.tgt_encoder)

Tokenizer mode and vocabulary attributes:
encoder: dict present = True
decoder: dict present = True
src_encoder: dict present = True
tgt_encoder: dict present = True
src_decoder: dict present = True
tgt_decoder: dict present = True
src_vocab_size: int present = True
tgt_vocab_size: int present = True
cur_encoder: NoneType present = False
cur_decoder: NoneType present = False

Mode-switching methods:

_switch_to_input_mode:
    def _switch_to_input_mode(self) -> None:
        self.spm = self.src_spm
        self.padding_side = "left"
        self.encoder = self.src_encoder
        self.decoder = self.src_decoder
        self._tokenize = self._src_tokenize


_switch_to_target_mode:
    def _switch_to_target_mode(self) -> None:
        self.spm = self.tgt_spm
        self.padding_side = "right"
        self.encoder = self.tgt_encoder
        self.decoder = self.tgt_decoder
        self._tokenize = self._tgt_tokenize


Original vocabulary sizes:
Source: 122706
Target: 122672
Decoder: 12267

In [104]:
from transformers import AutoTokenizer

extension_probe = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

candidate_word = "ᱤᱧᱟᱹᱜ"
new_token_id = len(extension_probe.tgt_encoder)

# Save the original state for invariant checks.
source_size_before = len(extension_probe.src_encoder)
target_size_before = len(extension_probe.tgt_encoder)
source_candidate_before = extension_probe.src_encoder.get(candidate_word)

# Extend only the target-side lookup dictionaries.
extension_probe.tgt_encoder[candidate_word] = new_token_id
extension_probe.tgt_decoder[new_token_id] = candidate_word

print("Assigned target ID:", new_token_id)
print("Target vocabulary size:", len(extension_probe.tgt_encoder))
print("Target decoder entry:", extension_probe.tgt_decoder[new_token_id])

print("\nSource vocabulary unchanged:",
      len(extension_probe.src_encoder) == source_size_before)
print("Source candidate mapping unchanged:",
      extension_probe.src_encoder.get(candidate_word) == source_candidate_before)
print("Original target IDs unchanged:",
      all(extension_probe.tgt_encoder.get(token) == token_id
          for token, token_id in tokenizer_probe.tgt_encoder.items()))

# Check the mapping after explicitly switching to target mode.
extension_probe._switch_to_target_mode()
print("\nTarget-mode encoder is target encoder:",
      extension_probe.encoder is extension_probe.tgt_encoder)
print("Target-mode decoder is target decoder:",
      extension_probe.decoder is extension_probe.tgt_decoder)
print("New token maps to:", extension_probe.convert_tokens_to_ids(candidate_word))

# Restore the ordinary input mode.
extension_probe._switch_to_input_mode()

Assigned target ID: 122672
Target vocabulary size: 122673
Target decoder entry: ᱤᱧᱟᱹᱜ

Source vocabulary unchanged: True
Source candidate mapping unchanged: True
Original target IDs unchanged: True

Target-mode encoder is target encoder: True
Target-mode decoder is target decoder: True
New token maps to: 122706


In [105]:
print("Candidate word:", candidate_word)

print("\nTarget encoder ID:",
      extension_probe.tgt_encoder.get(candidate_word))

print("Target decoder token:",
      extension_probe.tgt_decoder.get(new_token_id))

print("Added-token encoder ID:",
      extension_probe._added_tokens_encoder.get(candidate_word))

print("Added-token decoder entry:",
      extension_probe._added_tokens_decoder.get(new_token_id))

print("\nConversion while in target mode:")
extension_probe._switch_to_target_mode()
print("convert_tokens_to_ids:", extension_probe.convert_tokens_to_ids(candidate_word))
print("_convert_token_to_id:", extension_probe._convert_token_to_id(candidate_word))

print("\nConversion after returning to source mode:")
extension_probe._switch_to_input_mode()
print("convert_tokens_to_ids:", extension_probe.convert_tokens_to_ids(candidate_word))

Candidate word: ᱤᱧᱟᱹᱜ

Target encoder ID: 122672
Target decoder token: ᱤᱧᱟᱹᱜ
Added-token encoder ID: 122706
Added-token decoder entry: None

Conversion while in target mode:
convert_tokens_to_ids: 122706
_convert_token_to_id: 122672

Conversion after returning to source mode:
convert_tokens_to_ids: 122706


In [106]:
import inspect

for method_name in [
    "__call__",
    "_call_one",
    "encode",
    "prepare_for_model",
    "_src_tokenize",
    "_tgt_tokenize",
]:
    method = getattr(type(extension_probe), method_name, None)

    if method is not None:
        print(f"\n{'=' * 16} {method_name} {'=' * 16}")
        try:
            print(inspect.getsource(method))
        except (OSError, TypeError):
            print("Source unavailable")


================ __call__ ================
    @add_end_docstrings(ENCODE_KWARGS_DOCSTRING, ENCODE_PLUS_ADDITIONAL_KWARGS_DOCSTRING)
    def __call__(
        self,
        text: Union[TextInput, PreTokenizedInput, List[TextInput], List[PreTokenizedInput]] = None,
        text_pair: Optional[Union[TextInput, PreTokenizedInput, List[TextInput], List[PreTokenizedInput]]] = None,
        text_target: Union[TextInput, PreTokenizedInput, List[TextInput], List[PreTokenizedInput]] = None,
        text_pair_target: Optional[
            Union[TextInput, PreTokenizedInput, List[TextInput], List[PreTokenizedInput]]
        ] = None,
        add_special_tokens: bool = True,
        padding: Union[bool, str, PaddingStrategy] = False,
        truncation: Union[bool, str, TruncationStrategy] = None,
        max_length: Optional[int] = None,
        stride: int = 0,
        is_split_into_words: bool = False,
        pad_to_multiple_of: Optional[int] = None,
        padding_side: Optional[bool] = Non

In [107]:
probe_word = "ᱤᱧᱟᱹᱜ"
probe_sentence = f"{probe_word} ᱥᱟᱱᱛᱟᱲᱤ"

print("Word only:")
print(tokenizer_probe._tgt_tokenize(probe_word))

print("\nWord in a sentence:")
print(tokenizer_probe._tgt_tokenize(probe_sentence))

print("\nCurrent target IDs:")
print(tokenizer_probe(
    text_target=[probe_word, probe_sentence],
    add_special_tokens=False
)["input_ids"])

Word only:
['▁ᱤᱧ', 'ᱟᱹᱜ']

Word in a sentence:
['▁ᱤᱧ', 'ᱟᱹᱜ', '▁ᱥᱟᱱᱛᱟᱲᱤ']

Current target IDs:
[[122706], [122706, 117114]]


In [108]:
from transformers import AutoTokenizer

clean_tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

probe_word = "ᱤᱧᱟᱹᱜ"
probe_sentence = f"{probe_word} ᱥᱟᱱᱛᱟᱲᱤ"

print("Added vocabulary:", clean_tok.get_added_vocab())
print("Target vocabulary size:", len(clean_tok.tgt_encoder))

print("\nWord pieces:", clean_tok._tgt_tokenize(probe_word))
print("Word piece IDs:", [
    clean_tok.tgt_encoder.get(piece, clean_tok.unk_token_id)
    for piece in clean_tok._tgt_tokenize(probe_word)
])

print("\nTarget encoding:")
print(clean_tok(
    text_target=[probe_word, probe_sentence],
    add_special_tokens=False
)["input_ids"])

print("\nSource vocabulary size:", len(clean_tok.src_encoder))
print("Target vocabulary size:", len(clean_tok.tgt_encoder))

Added vocabulary: {'<s>': 0, '<pad>': 1, '</s>': 2, '<unk>': 3, 'ᱤᱧᱟᱹᱜ': 122706}
Target vocabulary size: 122672

Word pieces: ['▁ᱤᱧ', 'ᱟᱹᱜ']
Word piece IDs: [76527, 89023]

Target encoding:
[[122706], [122706, 117114]]

Source vocabulary size: 122706
Target vocabulary size: 122672


In [109]:
print("Tokenizer class:", type(clean_tok).__name__)
print("Tokenizer name/path:", getattr(clean_tok, "name_or_path", None))

print("\nAdded-token mappings:")
print("_added_tokens_encoder:", clean_tok._added_tokens_encoder)
print("_added_tokens_decoder:", clean_tok._added_tokens_decoder)

print("\nSpecial-token map:")
print(clean_tok.special_tokens_map)

print("\nSpecial token IDs:")
print(clean_tok.all_special_ids)

print("\nDoes candidate appear in source vocabulary?",
      probe_word in clean_tok.src_encoder)
print("Does candidate appear in target vocabulary?",
      probe_word in clean_tok.tgt_encoder)

print("\nOriginal target SentencePiece pieces:",
      clean_tok.tgt_spm.EncodeAsPieces(probe_word))
print("Direct target vocabulary IDs:",
      [clean_tok.tgt_encoder.get(piece, clean_tok.unk_token_id)
       for piece in clean_tok.tgt_spm.EncodeAsPieces(probe_word)])

Tokenizer class: IndicTransTokenizer
Tokenizer name/path: ai4bharat/indictrans2-indic-indic-1B

Added-token mappings:
_added_tokens_encoder: {'<s>': 0, '<pad>': 1, '</s>': 2, '<unk>': 3, 'ᱤᱧᱟᱹᱜ': 122706}
_added_tokens_decoder: {0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 122706: AddedToken("ᱤᱧᱟᱹᱜ", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False)}

Special-token map:
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}

Special token IDs:
[0, 2, 3, 1]

Does candidate appear in source vocabulary? False
Does candidate appear in target vocabulary? False

Original targ


# Section 8.10: Findings Summary and Implementation Plan

## 1. Research Objective

Investigate whether extending the target-side vocabulary with frequent, multi-piece Santali word forms and initializing their embeddings using existing subword embeddings can improve English-to-Santali translation compared with the original-vocabulary LoRA baseline.

The experiment will use the same dataset splits and evaluation metrics as the baseline to make the comparison fair.

## 2. Baseline Results

The baseline uses `ai4bharat/indictrans2-indic-indic-1B` with LoRA fine-tuning.

- LoRA rank: 16
- LoRA alpha: 32
- Target modules: `q_proj`, `v_proj`
- Best validation loss: 3.0152 at epoch 7
- Test SacreBLEU: 8.3859
- Test chrF: 41.0353
- Test examples: 994

The baseline adapter, training history, predictions, evaluation metrics, and metadata have been backed up. These artifacts must remain unchanged.

## 3. Tokenization Findings

The tokenizer analysis showed:

- The sampled Santali corpus averaged approximately 1.442 SentencePiece tokens per whitespace-separated word.
- No target SentencePiece UNKs were observed in the analyzed sample.
- Some frequent Santali word forms are split into multiple SentencePiece pieces.
- The earlier frequency-threshold analysis identified 97 candidate word types with frequency >= 50, representing 8,520 occurrences.

The estimated reduction in SentencePiece pieces is an upper-bound estimate. Actual tokenization changes and translation quality must be measured after implementation.

## 4. Model and Tokenizer Architecture

The inspected vocabulary and embedding dimensions are:

| Component | Vocabulary size |
|---|---:|
| Source tokenizer | 122706 |
| Target tokenizer | 122672 |
| Encoder embeddings | 122706 × 1024 |
| Decoder embeddings | 122672 × 1024 |
| Output projection (`lm_head`) | 122672 × 1024 |

The model uses separate source and target SentencePiece vocabularies. The decoder embedding and output projection are distinct, so both must be extended consistently.

## 5. Critical Finding: Generic Token Addition Is Unsafe

Using the tokenizer's generic `add_tokens()` method increased the shared tokenizer length but did not expand the target decoder vocabulary.

Consequently, target encoding could produce a token ID greater than or equal to the decoder's vocabulary size. Such an ID is invalid for the existing decoder.

Therefore:

- Do not use generic `add_tokens()` as the target-vocabulary extension method.
- Do not blindly use `resize_token_embeddings()` for this custom architecture.
- Do not modify or overwrite the original baseline model or artifacts.
- Maintain an explicit mapping between each new Santali word and its assigned target token ID.

## 6. Next Implementation Plan

### Step 1: Finalize Candidate Words

Use the existing cleaned candidate analysis to select frequent word forms that are split into multiple target SentencePiece pieces. Begin with a frequency threshold of 50 and verify that candidates are not duplicates or existing single-token vocabulary entries.

### Step 2: Implement Target-Side Tokenization

Build a target-side tokenizer adapter that recognizes selected word forms and maps them to explicit new IDs. Ordinary target text must continue to use the original target SentencePiece vocabulary. Source-side English tokenization must remain unchanged.

### Step 3: Expand Decoder Parameters

Create a separate model instance for the extended-vocabulary experiment. Expand the decoder input embedding and `lm_head` independently, preserving all original rows.

### Step 4: Initialize New Embeddings

Initialize each new token's decoder embedding and output-projection row using the corresponding original SentencePiece pieces. Record the initialization strategy so it can be reproduced and compared in future experiments.

### Step 5: Validate the Extended Model

Before training, verify:

- New token IDs are within the expanded decoder vocabulary.
- Original token IDs and embedding rows remain unchanged.
- Encoding and decoding behave correctly.
- The expanded model produces finite loss.
- Backpropagation works and the intended new parameters receive gradients.

### Step 6: Train the Extended-Vocabulary Model

Fine-tune the extended model using LoRA with the baseline's training setup wherever applicable. Keep the baseline and extended experiment outputs separate.

Save the vocabulary mapping, initialization metadata, training history, best adapter/checkpoint, and evaluation predictions.

### Step 7: Evaluate and Compare

Evaluate on the same validation and test splits using:

- SacreBLEU
- chrF
- Exact match
- Output/reference length ratio
- Tokenization efficiency and candidate-word coverage

Compare the extended-vocabulary model against the original-vocabulary baseline. Do not assume that fewer tokens necessarily lead to better translation quality.

## 7. Immediate Next Action

Begin with candidate finalization, then implement and test the target-side tokenizer adapter. Proceed to model expansion only after the token-ID mapping is validated. Start training only after the expanded model passes the structural and gradient checks.


## Section 9 — Vocabulary Extension + LoRA Fine-Tuning

### 9.1 Finalize the Candidate Vocabulary

Select frequent Santali word forms from the training corpus that are currently split into multiple SentencePiece pieces. Use a fixed minimum frequency of 50, keep the candidate selection deterministic, and do not modify the original tokenizer or model.

In [110]:
# Section 9.1: Finalize the vocabulary-extension candidates

import pandas as pd

# Reuse the candidates computed in Section 8.9.
assert "clean_candidates" in globals(), (
    "Run the existing Section 8.9 candidate-cleaning cell first."
)

assert not clean_candidates.empty, "The candidate table is empty."

MIN_FREQUENCY = 50

# Select frequent, clean word forms that need multiple pieces.
extension_candidates = (
    clean_candidates.loc[
        (clean_candidates["frequency"] >= MIN_FREQUENCY)
        & (clean_candidates["piece_count"] > 1)
    ]
    .drop_duplicates(subset=["word"])
    .sort_values(
        ["frequency", "potential_piece_reduction", "word"],
        ascending=[False, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Keep only the columns needed for the next implementation steps.
extension_candidates = extension_candidates[
    [
        "word",
        "frequency",
        "piece_count",
        "existing_pieces",
        "potential_piece_reduction",
    ]
].copy()

# Basic consistency checks.
assert extension_candidates["word"].is_unique
assert (extension_candidates["frequency"] >= MIN_FREQUENCY).all()
assert (extension_candidates["piece_count"] > 1).all()
assert extension_candidates["word"].notna().all()

print("Candidate vocabulary finalized")
print("=" * 55)
print("Minimum training frequency:", MIN_FREQUENCY)
print("Number of candidate word forms:", len(extension_candidates))
print(
    "Total training occurrences:",
    int(extension_candidates["frequency"].sum()),
)
print(
    "Estimated maximum piece reduction:",
    int(
        (
            extension_candidates["frequency"]
            * extension_candidates["potential_piece_reduction"]
        ).sum()
    ),
)

display(extension_candidates.head(20))

print("\nOriginal tokenizer unchanged:", len(tok) == 122706)
print("Original target vocabulary size:", len(tok.tgt_encoder))

Candidate vocabulary finalized
Minimum training frequency: 50
Number of candidate word forms: 97
Total training occurrences: 8520
Estimated maximum piece reduction: 9796


,word,frequency,piece_count,existing_pieces,potential_piece_reduction
0,ᱤᱧᱟᱹᱜ,807,2,▁ᱤᱧ | ᱟᱹᱜ,1
1,ᱤᱧᱤᱧ,232,2,▁ᱤᱧ | ᱤᱧ,1
2,ᱩᱱᱫᱚ,181,2,▁ᱩᱱ | ᱫᱚ,1
3,ᱫᱳᱠᱟᱱ,178,2,▁ᱫᱳ | ᱠᱟᱱ,1
4,ᱢᱩᱜᱷᱚᱞ,169,2,▁ᱢᱩᱜ | ᱷᱚᱞ,1
5,ᱱᱩᱱᱟᱹᱜ,155,2,▁ᱱᱩ | ᱱᱟᱹᱜ,1
6,ᱟᱞᱮᱭᱟᱜ,154,2,▁ᱟᱞᱮ | ᱭᱟᱜ,1
7,ᱤᱧᱨᱮᱱ,147,2,▁ᱤᱧ | ᱨᱮᱱ,1
8,ᱟᱹᱲᱤᱥ,138,2,▁ᱟᱹᱲ | ᱤᱥ,1
9,ᱦᱟᱦᱟᱲᱟᱜ,134,3,▁ᱦᱟᱦ | ᱟᱲ | ᱟᱜ,2



Original tokenizer unchanged: True
Original target vocabulary size: 122672


In [111]:
# Section 9.2: Create an explicit target-side vocabulary mapping
# This cell does NOT modify the tokenizer or model.

BASE_TARGET_VOCAB_SIZE = len(tok.tgt_encoder)

# Keep the original candidate order from Section 9.1.
candidate_words = extension_candidates["word"].tolist()

# Verify candidate words do not already exist as single target-vocabulary entries.
existing_target_tokens = set(tok.tgt_encoder.keys())

already_single_token = [
    word for word in candidate_words
    if word in existing_target_tokens
]

if already_single_token:
    print(
        f"Found {len(already_single_token)} candidates already present "
        "as single target-vocabulary entries."
    )
    display(pd.DataFrame({"word": already_single_token[:20]}))
else:
    print("No candidates already exist as single target-vocabulary entries.")

# Assign new IDs after the original target vocabulary.
candidate_to_id = {
    word: BASE_TARGET_VOCAB_SIZE + i
    for i, word in enumerate(candidate_words)
}

id_to_candidate = {
    token_id: word
    for word, token_id in candidate_to_id.items()
}

# Validate mapping invariants.
assert len(candidate_to_id) == len(candidate_words)
assert len(id_to_candidate) == len(candidate_words)
assert len(set(candidate_to_id.values())) == len(candidate_to_id)
assert min(candidate_to_id.values()) == BASE_TARGET_VOCAB_SIZE
assert max(candidate_to_id.values()) == (
    BASE_TARGET_VOCAB_SIZE + len(candidate_words) - 1
)

print("\nMapping created successfully.")
print("Original target vocabulary size:", BASE_TARGET_VOCAB_SIZE)
print("Number of new candidate IDs:", len(candidate_to_id))
print("Extended target vocabulary size:", BASE_TARGET_VOCAB_SIZE + len(candidate_to_id))
print("\nFirst 10 mappings:")

display(
    pd.DataFrame(
        [
            {"word": word, "new_token_id": token_id}
            for word, token_id in list(candidate_to_id.items())[:10]
        ]
    )
)

# Confirm that the original tokenizer is still unchanged.
assert len(tok.tgt_encoder) == BASE_TARGET_VOCAB_SIZE
print("\nOriginal target tokenizer remains unchanged.")

Found 2 candidates already present as single target-vocabulary entries.


,word
0,ᱨᱮᱧ
1,ᱨᱤᱧ



Mapping created successfully.
Original target vocabulary size: 122672
Number of new candidate IDs: 97
Extended target vocabulary size: 122769

First 10 mappings:


,word,new_token_id
0,ᱤᱧᱟᱹᱜ,122672
1,ᱤᱧᱤᱧ,122673
2,ᱩᱱᱫᱚ,122674
3,ᱫᱳᱠᱟᱱ,122675
4,ᱢᱩᱜᱷᱚᱞ,122676
5,ᱱᱩᱱᱟᱹᱜ,122677
6,ᱟᱞᱮᱭᱟᱜ,122678
7,ᱤᱧᱨᱮᱱ,122679
8,ᱟᱹᱲᱤᱥ,122680
9,ᱦᱟᱦᱟᱲᱟᱜ,122681



Original target tokenizer remains unchanged.


In [112]:
# Section 9.3: Exclude candidates already present in the target vocabulary

existing_target_tokens = set(tok.tgt_encoder.keys())

already_existing_words = [
    word for word in candidate_words
    if word in existing_target_tokens
]

new_candidate_words = [
    word for word in candidate_words
    if word not in existing_target_tokens
]

# Assign consecutive IDs only to genuinely new target tokens.
candidate_to_id = {
    word: BASE_TARGET_VOCAB_SIZE + i
    for i, word in enumerate(new_candidate_words)
}

id_to_candidate = {
    token_id: word
    for word, token_id in candidate_to_id.items()
}

# Validation
assert len(already_existing_words) == 2
assert len(new_candidate_words) == 95
assert len(candidate_to_id) == len(new_candidate_words)
assert len(set(candidate_to_id.values())) == len(candidate_to_id)
assert all(word not in existing_target_tokens for word in candidate_to_id)

assert len(tok.tgt_encoder) == BASE_TARGET_VOCAB_SIZE

print("Already-existing candidates:", already_existing_words)
print("Original target vocabulary size:", BASE_TARGET_VOCAB_SIZE)
print("Genuine new tokens:", len(new_candidate_words))
print("Corrected extended vocabulary size:",
      BASE_TARGET_VOCAB_SIZE + len(new_candidate_words))
print("First 10 corrected mappings:")

display(pd.DataFrame([
    {"word": word, "new_token_id": token_id}
    for word, token_id in list(candidate_to_id.items())[:10]
]))

print("\nOriginal tokenizer remains unchanged.")

Already-existing candidates: ['ᱨᱮᱧ', 'ᱨᱤᱧ']
Original target vocabulary size: 122672
Genuine new tokens: 95
Corrected extended vocabulary size: 122767
First 10 corrected mappings:


,word,new_token_id
0,ᱤᱧᱟᱹᱜ,122672
1,ᱤᱧᱤᱧ,122673
2,ᱩᱱᱫᱚ,122674
3,ᱫᱳᱠᱟᱱ,122675
4,ᱢᱩᱜᱷᱚᱞ,122676
5,ᱱᱩᱱᱟᱹᱜ,122677
6,ᱟᱞᱮᱭᱟᱜ,122678
7,ᱤᱧᱨᱮᱱ,122679
8,ᱟᱹᱲᱤᱥ,122680
9,ᱦᱟᱦᱟᱲᱟᱜ,122681



Original tokenizer remains unchanged.


In [113]:
# Section 9.4: Define a non-mutating target-tokenizer adapter
# This cell defines functions only; it does not modify tok or model.

import re

# Longer words first in case candidate forms overlap.
_extension_words_sorted = sorted(
    new_candidate_words,
    key=lambda word: (-len(word), word)
)

# Match candidates at whitespace-delimited boundaries.
# Punctuation remains attached to the surrounding text and is handled
# by the original tokenizer.
_extension_pattern = re.compile(
    r"(?<!\S)("
    + "|".join(re.escape(word) for word in _extension_words_sorted)
    + r")(?!\S)"
) if _extension_words_sorted else None


def tokenize_target_with_extension(text):
    """
    Return target token IDs using:
      - new IDs for recognized candidate words;
      - original target-tokenizer IDs for all other text.

    This is a diagnostic adapter, not yet the final model-training adapter.
    """
    if not isinstance(text, str):
        raise TypeError("Target text must be a string.")

    if not text:
        return []

    token_ids = []
    cursor = 0

    for match in _extension_pattern.finditer(text):
        # Encode the preceding segment with the original target tokenizer.
        prefix = text[cursor:match.start()]
        if prefix:
            encoded_prefix = tok(
                text_target=[prefix],
                truncation=False,
            )["input_ids"][0]
            token_ids.extend(encoded_prefix)

        # Replace the matched candidate with its new ID.
        token_ids.append(candidate_to_id[match.group(1)])
        cursor = match.end()

    # Encode the remaining segment.
    suffix = text[cursor:]
    if suffix:
        encoded_suffix = tok(
            text_target=[suffix],
            truncation=False,
        )["input_ids"][0]
        token_ids.extend(encoded_suffix)

    return token_ids


print("Adapter function defined.")
print("Number of extension candidates:", len(new_candidate_words))
print("Original target vocabulary size:", len(tok.tgt_encoder))
print("Original tokenizer remains unmodified.")

Adapter function defined.
Number of extension candidates: 95
Original target vocabulary size: 122672
Original tokenizer remains unmodified.


In [114]:
# Section 9.5: Validate the target-tokenizer adapter

# Test one frequent candidate in different contexts.
test_word = new_candidate_words[0]

test_cases = {
    "isolated": test_word,
    "with_leading_space": " " + test_word,
    "inside_sentence": "ABC " + test_word + " XYZ",
    "with_punctuation": test_word + ",",
    "repeated": test_word + " " + test_word,
}

for case_name, text in test_cases.items():
    original_ids = tok(
        text_target=[text],
        truncation=False,
    )["input_ids"][0]

    adapted_ids = tokenize_target_with_extension(text)

    print(f"\n--- {case_name} ---")
    print("Text:", repr(text))
    print("Original IDs:", original_ids)
    print("Adapted IDs:", adapted_ids)
    print(
        "New candidate ID count:",
        sum(token_id in id_to_candidate for token_id in adapted_ids)
    )

print("\nSpecial-token IDs:")
print("BOS:", tok.tgt_bos_token_id)
print("PAD:", tok.tgt_pad_token_id)
print("EOS:", tok.tgt_eos_token_id)
print("UNK:", tok.tgt_unk_token_id)

print("\nOriginal target vocabulary size:", len(tok.tgt_encoder))
print("Adapter test completed; no tokenizer/model mutation performed.")


--- isolated ---
Text: 'ᱤᱧᱟᱹᱜ'
Original IDs: [76527, 89023, 2]
Adapted IDs: [122672]
New candidate ID count: 1

--- with_leading_space ---
Text: ' ᱤᱧᱟᱹᱜ'
Original IDs: [76527, 89023, 2]
Adapted IDs: [2, 122672]
New candidate ID count: 1

--- inside_sentence ---
Text: 'ABC ᱤᱧᱟᱹᱜ XYZ'
Original IDs: [1613, 27984, 76527, 89023, 3081, 9079, 16348, 2]
Adapted IDs: [1613, 27984, 2, 122672, 3081, 9079, 16348, 2]
New candidate ID count: 1

--- with_punctuation ---
Text: 'ᱤᱧᱟᱹᱜ,'
Original IDs: [76527, 89023, 94120, 2]
Adapted IDs: [76527, 89023, 94120, 2]
New candidate ID count: 0

--- repeated ---
Text: 'ᱤᱧᱟᱹᱜ ᱤᱧᱟᱹᱜ'
Original IDs: [76527, 89023, 76527, 89023, 2]
Adapted IDs: [122672, 2, 122672]
New candidate ID count: 2

Special-token IDs:


AttributeError: 'IndicTransTokenizer' object has no attribute 'tgt_bos_token_id'

In [115]:
# Section 9.6: Inspect target SentencePiece behavior before fixing the adapter

print("Target SentencePiece object:", type(tok.tgt_spm))
print("Has EncodeAsIds:", hasattr(tok.tgt_spm, "EncodeAsIds"))
print("Has encode:", hasattr(tok.tgt_spm, "encode"))
print("Has DecodeIds:", hasattr(tok.tgt_spm, "DecodeIds"))
print("Has decode:", hasattr(tok.tgt_spm, "decode"))

test_texts = [
    "Hello",
    " Hello",
    "ABC XYZ",
    "ABC " + new_candidate_words[0] + " XYZ",
]

for text in test_texts:
    original_ids = tok(
        text_target=[text],
        truncation=False,
    )["input_ids"][0]

    print("\nText:", repr(text))
    print("Original tokenizer IDs:", original_ids)

    if hasattr(tok.tgt_spm, "EncodeAsIds"):
        print("Direct SentencePiece IDs:", tok.tgt_spm.EncodeAsIds(text))
    elif hasattr(tok.tgt_spm, "encode"):
        print("Direct SentencePiece IDs:", tok.tgt_spm.encode(text, out_type=int))

print("\nKnown special-token IDs: BOS=0, PAD=1, EOS=2, UNK=3")
print("Original target vocabulary size:", len(tok.tgt_encoder))

Target SentencePiece object: <class 'sentencepiece.SentencePieceProcessor'>
Has EncodeAsIds: True
Has encode: True
Has DecodeIds: True
Has decode: True

Text: 'Hello'
Original tokenizer IDs: [36485, 64643, 2]
Direct SentencePiece IDs: [3635, 52321]

Text: ' Hello'
Original tokenizer IDs: [36485, 64643, 2]
Direct SentencePiece IDs: [3635, 52321]

Text: 'ABC XYZ'
Original tokenizer IDs: [1613, 27984, 3081, 9079, 16348, 2]
Direct SentencePiece IDs: [385, 34325, 8650, 121839, 121856]

Text: 'ABC ᱤᱧᱟᱹᱜ XYZ'
Original tokenizer IDs: [1613, 27984, 76527, 89023, 3081, 9079, 16348, 2]
Direct SentencePiece IDs: [385, 34325, 12124, 1450, 8650, 121839, 121856]

Known special-token IDs: BOS=0, PAD=1, EOS=2, UNK=3
Original target vocabulary size: 122672


In [116]:
# Section 9.7: Inspect SentencePiece and target encoder ID alignment
# Read-only diagnostic; does not modify the tokenizer or model.

test_text = "ABC XYZ"
sp_ids = tok.tgt_spm.EncodeAsIds(test_text)

print("Test text:", repr(test_text))
print("Direct SentencePiece IDs:", sp_ids)

print("\nSentencePiece pieces and IDs:")
for sp_id in sp_ids:
    piece = tok.tgt_spm.IdToPiece(sp_id)
    encoder_id = tok.tgt_encoder.get(piece)

    print({
        "sentencepiece_id": sp_id,
        "piece": piece,
        "target_encoder_id": encoder_id,
        "ids_match": encoder_id == sp_id if encoder_id is not None else False,
    })

# Check representative entries from the SentencePiece vocabulary.
print("\nRepresentative vocabulary alignment checks:")
for sp_id in [0, 1, 2, 3, 100, 1000, 10000]:
    if sp_id < tok.tgt_spm.GetPieceSize():
        piece = tok.tgt_spm.IdToPiece(sp_id)
        encoder_id = tok.tgt_encoder.get(piece)
        print(
            f"SP ID={sp_id}, piece={piece!r}, "
            f"target encoder ID={encoder_id}"
        )

print("\nSentencePiece vocabulary size:", tok.tgt_spm.GetPieceSize())
print("Target encoder vocabulary size:", len(tok.tgt_encoder))

Test text: 'ABC XYZ'
Direct SentencePiece IDs: [385, 34325, 8650, 121839, 121856]

SentencePiece pieces and IDs:
{'sentencepiece_id': 385, 'piece': '▁A', 'target_encoder_id': 1613, 'ids_match': False}
{'sentencepiece_id': 34325, 'piece': 'BC', 'target_encoder_id': 27984, 'ids_match': False}
{'sentencepiece_id': 8650, 'piece': '▁X', 'target_encoder_id': 3081, 'ids_match': False}
{'sentencepiece_id': 121839, 'piece': 'Y', 'target_encoder_id': 9079, 'ids_match': False}
{'sentencepiece_id': 121856, 'piece': 'Z', 'target_encoder_id': 16348, 'ids_match': False}

Representative vocabulary alignment checks:
SP ID=0, piece='<unk>', target encoder ID=3
SP ID=1, piece='<s>', target encoder ID=0
SP ID=2, piece='</s>', target encoder ID=2
SP ID=3, piece='▁<dnt>', target encoder ID=None
SP ID=100, piece='▁ن', target encoder ID=11974
SP ID=1000, piece='▁ba', target encoder ID=57532
SP ID=10000, piece='▁stat', target encoder ID=59623

SentencePiece vocabulary size: 128000
Target encoder vocabulary siz

In [117]:
# Section 9.8: Verify candidate-piece conversion to model target IDs
# Read-only diagnostic; does not modify tokenizer or model.

def inspect_candidate_pieces(word):
    sp_ids = tok.tgt_spm.EncodeAsIds(word)
    rows = []

    for sp_id in sp_ids:
        piece = tok.tgt_spm.IdToPiece(sp_id)
        model_id = tok.tgt_encoder.get(piece)

        rows.append({
            "word": word,
            "sentencepiece_id": sp_id,
            "piece": piece,
            "model_target_id": model_id,
            "valid_model_id": (
                model_id is not None
                and 0 <= model_id < len(tok.tgt_encoder)
            ),
        })

    return rows


# Inspect several frequent candidates.
sample_words = new_candidate_words[:5]
piece_rows = []

for word in sample_words:
    piece_rows.extend(inspect_candidate_pieces(word))

piece_check_df = pd.DataFrame(piece_rows)
display(piece_check_df)

print("\nSummary")
print("Pieces inspected:", len(piece_check_df))
print("Pieces with valid model IDs:", int(piece_check_df["valid_model_id"].sum()))
print("Pieces missing from target encoder:", int((~piece_check_df["valid_model_id"]).sum()))

# Check the complete candidate list.
all_candidate_piece_rows = []

for word in new_candidate_words:
    all_candidate_piece_rows.extend(inspect_candidate_pieces(word))

all_candidate_pieces_df = pd.DataFrame(all_candidate_piece_rows)

print("\nAcross all 95 candidates:")
print("Total SentencePiece pieces:", len(all_candidate_pieces_df))
print(
    "Pieces missing from target encoder:",
    int((~all_candidate_pieces_df["valid_model_id"]).sum())
)

,word,sentencepiece_id,piece,model_target_id,valid_model_id
0,ᱤᱧᱟᱹᱜ,12124,▁ᱤᱧ,76527,True
1,ᱤᱧᱟᱹᱜ,1450,ᱟᱹᱜ,89023,True
2,ᱤᱧᱤᱧ,12124,▁ᱤᱧ,76527,True
3,ᱤᱧᱤᱧ,5670,ᱤᱧ,95824,True
4,ᱩᱱᱫᱚ,3994,▁ᱩᱱ,86053,True
5,ᱩᱱᱫᱚ,2585,ᱫᱚ,80590,True
6,ᱫᱳᱠᱟᱱ,46582,▁ᱫᱳ,101262,True
7,ᱫᱳᱠᱟᱱ,6628,ᱠᱟᱱ,88576,True
8,ᱢᱩᱜᱷᱚᱞ,48709,▁ᱢᱩᱜ,92396,True
9,ᱢᱩᱜᱷᱚᱞ,80973,ᱷᱚᱞ,92222,True



Summary
Pieces inspected: 10
Pieces with valid model IDs: 10
Pieces missing from target encoder: 0

Across all 95 candidates:
Total SentencePiece pieces: 207
Pieces missing from target encoder: 0


In [118]:
# Section 9.9: Corrected target-tokenizer adapter prototype
# Does not modify the original tokenizer or model.

import re

# Candidate words sorted longest-first.
_extension_words_sorted = sorted(
    new_candidate_words,
    key=lambda word: (-len(word), word)
)

# Match candidate words at whitespace boundaries or before punctuation.
_extension_pattern = re.compile(
    r"(?<!\S)("
    + "|".join(re.escape(word) for word in _extension_words_sorted)
    + r")(?!\S|[^\w\s])"
) if _extension_words_sorted else None


def encode_original_target_segment(segment):
    """Encode text without adding EOS; convert SP pieces to model target IDs."""
    if not segment:
        return []

    sp_ids = tok.tgt_spm.EncodeAsIds(segment)
    model_ids = []

    for sp_id in sp_ids:
        piece = tok.tgt_spm.IdToPiece(sp_id)
        model_id = tok.tgt_encoder.get(piece)

        if model_id is None:
            raise ValueError(
                f"SentencePiece token {piece!r} (SP ID {sp_id}) "
                "is missing from the target encoder."
            )

        model_ids.append(model_id)

    return model_ids


def tokenize_target_with_extension_v2(text, add_eos=True):
    """
    Encode target text with new IDs for candidate words.
    Adds EOS only once at the end when requested.
    """
    if not isinstance(text, str):
        raise TypeError("Target text must be a string.")

    token_ids = []
    cursor = 0

    for match in _extension_pattern.finditer(text):
        prefix = text[cursor:match.start()]
        token_ids.extend(encode_original_target_segment(prefix))

        token_ids.append(candidate_to_id[match.group(1)])
        cursor = match.end()

    token_ids.extend(encode_original_target_segment(text[cursor:]))

    if add_eos:
        token_ids.append(2)  # EOS

    return token_ids


print("Corrected adapter prototype defined.")
print("Candidate words:", len(new_candidate_words))
print("Original target vocabulary size:", len(tok.tgt_encoder))
print("Extended target vocabulary size:",
      len(tok.tgt_encoder) + len(candidate_to_id))
print("Original tokenizer and model remain unchanged.")

Corrected adapter prototype defined.
Candidate words: 95
Original target vocabulary size: 122672
Extended target vocabulary size: 122767
Original tokenizer and model remain unchanged.


In [119]:
# Section 9.10: Validate corrected adapter v2

test_word = new_candidate_words[0]

test_cases = {
    "isolated": test_word,
    "leading_space": " " + test_word,
    "inside_sentence": "ABC " + test_word + " XYZ",
    "with_comma": test_word + ",",
    "repeated": test_word + " " + test_word,
}

for case_name, text in test_cases.items():
    adapted_ids = tokenize_target_with_extension_v2(text, add_eos=True)

    new_id_positions = [
        i for i, token_id in enumerate(adapted_ids)
        if token_id in id_to_candidate
    ]

    eos_positions = [
        i for i, token_id in enumerate(adapted_ids)
        if token_id == 2
    ]

    print(f"\n--- {case_name} ---")
    print("Text:", repr(text))
    print("Adapted IDs:", adapted_ids)
    print("Recognized candidates:", len(new_id_positions))
    print("EOS positions:", eos_positions)
    print(
        "EOS appears only at the end:",
        not eos_positions or eos_positions == [len(adapted_ids) - 1]
    )

print("\nOriginal tokenizer vocabulary size:", len(tok.tgt_encoder))
print("Extended vocabulary size:", len(tok.tgt_encoder) + len(candidate_to_id))


--- isolated ---
Text: 'ᱤᱧᱟᱹᱜ'
Adapted IDs: [122672, 2]
Recognized candidates: 1
EOS positions: [1]
EOS appears only at the end: True

--- leading_space ---
Text: ' ᱤᱧᱟᱹᱜ'
Adapted IDs: [122672, 2]
Recognized candidates: 1
EOS positions: [1]
EOS appears only at the end: True

--- inside_sentence ---
Text: 'ABC ᱤᱧᱟᱹᱜ XYZ'
Adapted IDs: [1613, 27984, 122672, 3081, 9079, 16348, 2]
Recognized candidates: 1
EOS positions: [6]
EOS appears only at the end: True

--- with_comma ---
Text: 'ᱤᱧᱟᱹᱜ,'
Adapted IDs: [76527, 89023, 94120, 2]
Recognized candidates: 0
EOS positions: [3]
EOS appears only at the end: True

--- repeated ---
Text: 'ᱤᱧᱟᱹᱜ ᱤᱧᱟᱹᱜ'
Adapted IDs: [122672, 122672, 2]
Recognized candidates: 2
EOS positions: [2]
EOS appears only at the end: True

Original tokenizer vocabulary size: 122672
Extended vocabulary size: 122767


In [120]:
# Section 9.11: Update candidate matching to allow punctuation

_extension_pattern = re.compile(
    r"(?<!\S)("
    + "|".join(re.escape(word) for word in _extension_words_sorted)
    + r")(?!\S)"
)

print("Updated candidate matching pattern.")
print("The adapter function will use the updated pattern.")

Updated candidate matching pattern.
The adapter function will use the updated pattern.


In [121]:
# Verify punctuation and whitespace behavior after updating the pattern

test_cases = [
    new_candidate_words[0],
    new_candidate_words[0] + ",",
    new_candidate_words[0] + ".",
    new_candidate_words[0] + "!",
    "ABC " + new_candidate_words[0] + ", XYZ",
    new_candidate_words[0] + " " + new_candidate_words[0],
]

for text in test_cases:
    ids = tokenize_target_with_extension_v2(text, add_eos=True)
    recognized = sum(token_id in id_to_candidate for token_id in ids)

    print(
        repr(text),
        "| recognized candidates:", recognized,
        "| EOS count:", ids.count(2),
        "| EOS at end:", bool(ids) and ids[-1] == 2,
    )

'ᱤᱧᱟᱹᱜ' | recognized candidates: 1 | EOS count: 1 | EOS at end: True
'ᱤᱧᱟᱹᱜ,' | recognized candidates: 0 | EOS count: 1 | EOS at end: True
'ᱤᱧᱟᱹᱜ.' | recognized candidates: 0 | EOS count: 1 | EOS at end: True
'ᱤᱧᱟᱹᱜ!' | recognized candidates: 0 | EOS count: 1 | EOS at end: True
'ABC ᱤᱧᱟᱹᱜ, XYZ' | recognized candidates: 0 | EOS count: 1 | EOS at end: True
'ᱤᱧᱟᱹᱜ ᱤᱧᱟᱹᱜ' | recognized candidates: 2 | EOS count: 1 | EOS at end: True


In [122]:
# Section 9.12: Correct punctuation-aware candidate matching

# Match a candidate if it starts at the beginning of the text or after
# whitespace, and ends before whitespace, punctuation, or end-of-text.
_extension_pattern = re.compile(
    r"(?<!\S)("
    + "|".join(re.escape(word) for word in _extension_words_sorted)
    + r")(?=$|[\s\.,!?;:()\[\]{}\"'।॥])"
)

print("Updated punctuation-aware matching pattern.")

Updated punctuation-aware matching pattern.


In [123]:
# Verify punctuation-aware matching

test_cases = [
    new_candidate_words[0],
    new_candidate_words[0] + ",",
    new_candidate_words[0] + ".",
    new_candidate_words[0] + "!",
    "ABC " + new_candidate_words[0] + ", XYZ",
    new_candidate_words[0] + " " + new_candidate_words[0],
    "X" + new_candidate_words[0] + "Y",  # Must not match inside a larger word
]

for text in test_cases:
    ids = tokenize_target_with_extension_v2(text, add_eos=True)
    recognized = sum(token_id in id_to_candidate for token_id in ids)

    print(
        repr(text),
        "| recognized candidates:", recognized,
        "| EOS count:", ids.count(2),
        "| EOS at end:", bool(ids) and ids[-1] == 2,
    )

'ᱤᱧᱟᱹᱜ' | recognized candidates: 1 | EOS count: 1 | EOS at end: True
'ᱤᱧᱟᱹᱜ,' | recognized candidates: 1 | EOS count: 1 | EOS at end: True
'ᱤᱧᱟᱹᱜ.' | recognized candidates: 1 | EOS count: 1 | EOS at end: True
'ᱤᱧᱟᱹᱜ!' | recognized candidates: 1 | EOS count: 1 | EOS at end: True
'ABC ᱤᱧᱟᱹᱜ, XYZ' | recognized candidates: 1 | EOS count: 1 | EOS at end: True
'ᱤᱧᱟᱹᱜ ᱤᱧᱟᱹᱜ' | recognized candidates: 2 | EOS count: 1 | EOS at end: True
'XᱤᱧᱟᱹᱜY' | recognized candidates: 0 | EOS count: 1 | EOS at end: True


In [124]:
# Section 9.13: Inspect model embedding layers before expansion
# Read-only inspection; do not resize or modify the model yet.

print("Model class:", type(model).__name__)
print("Configured vocabulary size:", model.config.vocab_size)
print("Configured encoder vocabulary size:", model.config.encoder_vocab_size)
print("Configured decoder vocabulary size:", model.config.decoder_vocab_size)

print("\nEncoder embedding:")
print("Shape:", tuple(model.model.encoder.embed_tokens.weight.shape))

print("\nDecoder embedding:")
print("Shape:", tuple(model.model.decoder.embed_tokens.weight.shape))

print("\nOutput projection (lm_head):")
print("Shape:", tuple(model.lm_head.weight.shape))

print("\nEmbedding/output weight sharing:")
print(
    "Same parameter object:",
    model.model.decoder.embed_tokens.weight is model.lm_head.weight
)
print(
    "Same underlying storage:",
    model.model.decoder.embed_tokens.weight.data_ptr()
    == model.lm_head.weight.data_ptr()
)

print("\nPlanned vocabulary extension:")
print("New tokens:", len(candidate_to_id))
print("Original target vocabulary:", BASE_TARGET_VOCAB_SIZE)
print("Expected extended target vocabulary:",
      BASE_TARGET_VOCAB_SIZE + len(candidate_to_id))

Model class: IndicTransForConditionalGeneration
Configured vocabulary size: 122672
Configured encoder vocabulary size: 122706
Configured decoder vocabulary size: 122672

Encoder embedding:
Shape: (122706, 1024)

Decoder embedding:
Shape: (122672, 1024)

Output projection (lm_head):
Shape: (122672, 1024)

Embedding/output weight sharing:
Same parameter object: False
Same underlying storage: False

Planned vocabulary extension:
New tokens: 95
Original target vocabulary: 122672
Expected extended target vocabulary: 122767


In [125]:
# Section 9.14: Create an isolated model for vocabulary extension

import copy
import torch

# Keep an independent copy of the current model.
# This may temporarily require additional RAM/VRAM.
extended_model = copy.deepcopy(model)

# Record original weights for later integrity checks.
original_decoder_weights = (
    extended_model.model.decoder.embed_tokens.weight.detach().cpu().clone()
)
original_lm_head_weights = (
    extended_model.lm_head.weight.detach().cpu().clone()
)

# Verify the copy is independent of the original model.
assert (
    extended_model.model.decoder.embed_tokens.weight.data_ptr()
    != model.model.decoder.embed_tokens.weight.data_ptr()
)
assert (
    extended_model.lm_head.weight.data_ptr()
    != model.lm_head.weight.data_ptr()
)

# Verify copied weights initially match exactly.
assert torch.equal(
    original_decoder_weights,
    model.model.decoder.embed_tokens.weight.detach().cpu()
)
assert torch.equal(
    original_lm_head_weights,
    model.lm_head.weight.detach().cpu()
)

print("Isolated experimental model created.")
print("Decoder embedding snapshot:", tuple(original_decoder_weights.shape))
print("LM head snapshot:", tuple(original_lm_head_weights.shape))
print("Experimental model weights are independent:", True)
print("Original model remains unchanged:", True)

Isolated experimental model created.
Decoder embedding snapshot: (122672, 1024)
LM head snapshot: (122672, 1024)
Experimental model weights are independent: True
Original model remains unchanged: True


In [126]:
# Section 9.15: Inspect model type and active LoRA adapters

print("Original model type:", type(model).__name__)
print("Experimental model type:", type(extended_model).__name__)

# Check whether either model has PEFT/LoRA configuration.
for name, current_model in [
    ("Original model", model),
    ("Experimental model", extended_model),
]:
    print(f"\n{name}:")
    print("Has peft_config:", hasattr(current_model, "peft_config"))

    if hasattr(current_model, "peft_config"):
        print("Adapter names:", list(current_model.peft_config.keys()))

    lora_modules = [
        module_name
        for module_name, _ in current_model.named_modules()
        if "lora_" in module_name.lower()
    ]

    print("Number of LoRA-related modules:", len(lora_modules))
    print("Sample LoRA modules:", lora_modules[:5])

Original model type: IndicTransForConditionalGeneration
Experimental model type: IndicTransForConditionalGeneration

Original model:
Has peft_config: False
Number of LoRA-related modules: 0
Sample LoRA modules: []

Experimental model:
Has peft_config: False
Number of LoRA-related modules: 0
Sample LoRA modules: []


In [127]:
# Section 9.16: Inspect decoder embedding and LM head before expansion

import torch

decoder_embedding = extended_model.model.decoder.embed_tokens
lm_head = extended_model.lm_head

print("Decoder embedding:")
print("  Shape:", tuple(decoder_embedding.weight.shape))
print("  Device:", decoder_embedding.weight.device)
print("  Dtype:", decoder_embedding.weight.dtype)
print("  Padding index:", decoder_embedding.padding_idx)

print("\nLM head:")
print("  Shape:", tuple(lm_head.weight.shape))
print("  Device:", lm_head.weight.device)
print("  Dtype:", lm_head.weight.dtype)
print("  Has bias:", lm_head.bias is not None)

print("\nVocabulary configuration:")
print("  Current config.vocab_size:", extended_model.config.vocab_size)
print("  Current decoder_vocab_size:", extended_model.config.decoder_vocab_size)
print("  Original target vocabulary:", BASE_TARGET_VOCAB_SIZE)
print("  New candidate rows:", len(new_candidate_words))
print("  Planned extended vocabulary:", BASE_TARGET_VOCAB_SIZE + len(new_candidate_words))

assert decoder_embedding.num_embeddings == BASE_TARGET_VOCAB_SIZE
assert lm_head.out_features == BASE_TARGET_VOCAB_SIZE
assert decoder_embedding.weight.device == lm_head.weight.device
assert decoder_embedding.weight.dtype == lm_head.weight.dtype

print("\nStructural checks passed.")

Decoder embedding:
  Shape: (122672, 1024)
  Device: cuda:0
  Dtype: torch.float16
  Padding index: 1

LM head:
  Shape: (122672, 1024)
  Device: cuda:0
  Dtype: torch.float16
  Has bias: False

Vocabulary configuration:
  Current config.vocab_size: 122672
  Current decoder_vocab_size: 122672
  Original target vocabulary: 122672
  New candidate rows: 95
  Planned extended vocabulary: 122767

Structural checks passed.


In [128]:
# Section 9.17: Expand decoder embeddings and LM head

import torch
import torch.nn as nn

old_vocab_size = BASE_TARGET_VOCAB_SIZE
new_vocab_size = old_vocab_size + len(new_candidate_words)

old_embedding = extended_model.model.decoder.embed_tokens
old_lm_head = extended_model.lm_head

device = old_embedding.weight.device
dtype = old_embedding.weight.dtype
hidden_size = old_embedding.embedding_dim

# Create expanded layers on the same device and with the same dtype.
new_embedding = nn.Embedding(
    num_embeddings=new_vocab_size,
    embedding_dim=hidden_size,
    padding_idx=old_embedding.padding_idx,
    device=device,
    dtype=dtype,
)

new_lm_head = nn.Linear(
    in_features=hidden_size,
    out_features=new_vocab_size,
    bias=False,
    device=device,
    dtype=dtype,
)

# Preserve every original row exactly.
with torch.no_grad():
    new_embedding.weight[:old_vocab_size].copy_(old_embedding.weight)
    new_lm_head.weight[:old_vocab_size].copy_(old_lm_head.weight)

# Replace the model layers.
extended_model.model.decoder.embed_tokens = new_embedding
extended_model.lm_head = new_lm_head

# Update vocabulary configuration.
extended_model.config.vocab_size = new_vocab_size
extended_model.config.decoder_vocab_size = new_vocab_size

print("Expanded decoder embedding:", tuple(new_embedding.weight.shape))
print("Expanded LM head:", tuple(new_lm_head.weight.shape))
print("Device:", new_embedding.weight.device)
print("Dtype:", new_embedding.weight.dtype)
print("Updated config.vocab_size:", extended_model.config.vocab_size)
print("Updated config.decoder_vocab_size:", extended_model.config.decoder_vocab_size)

# Verify original rows are unchanged.
assert torch.equal(
    new_embedding.weight[:old_vocab_size],
    original_decoder_weights.to(device=device, dtype=dtype),
)
assert torch.equal(
    new_lm_head.weight[:old_vocab_size],
    original_lm_head_weights.to(device=device, dtype=dtype),
)

assert new_embedding.num_embeddings == new_vocab_size
assert new_lm_head.out_features == new_vocab_size

print("\nExpansion successful.")
print("Original embedding and LM-head rows preserved exactly.")
print("New rows are allocated and ready for initialization.")

Expanded decoder embedding: (122767, 1024)
Expanded LM head: (122767, 1024)
Device: cuda:0
Dtype: torch.float16
Updated config.vocab_size: 122767
Updated config.decoder_vocab_size: 122767

Expansion successful.
Original embedding and LM-head rows preserved exactly.
New rows are allocated and ready for initialization.


In [129]:
# Section 9.18: Initialize new token rows from constituent subword embeddings

import torch

initialized_candidates = 0
initialization_errors = []

with torch.no_grad():
    for word in new_candidate_words:
        new_id = candidate_to_id[word]

        # Encode the candidate into its original SentencePiece pieces.
        sp_ids = tok.tgt_spm.EncodeAsIds(word)

        # Convert SentencePiece IDs to model target-vocabulary IDs.
        piece_model_ids = []
        for sp_id in sp_ids:
            piece = tok.tgt_spm.IdToPiece(sp_id)
            model_id = tok.tgt_encoder.get(piece)

            if model_id is None or not (0 <= model_id < BASE_TARGET_VOCAB_SIZE):
                initialization_errors.append(
                    (word, piece, model_id)
                )
                piece_model_ids = []
                break

            piece_model_ids.append(model_id)

        if not piece_model_ids:
            continue

        # The candidate must actually be composed of multiple original pieces.
        if len(piece_model_ids) < 2:
            initialization_errors.append(
                (word, "Expected multiple pieces", piece_model_ids)
            )
            continue

        # Initialize each new row as the mean of its constituent rows.
        index_tensor = torch.tensor(
            piece_model_ids,
            device=extended_model.model.decoder.embed_tokens.weight.device,
            dtype=torch.long,
        )

        extended_model.model.decoder.embed_tokens.weight[new_id].copy_(
            extended_model.model.decoder.embed_tokens.weight[index_tensor]
            .mean(dim=0)
        )

        extended_model.lm_head.weight[new_id].copy_(
            extended_model.lm_head.weight[index_tensor].mean(dim=0)
        )

        initialized_candidates += 1

print("Candidates requested:", len(new_candidate_words))
print("Candidates initialized:", initialized_candidates)
print("Initialization errors:", len(initialization_errors))
print("Expanded decoder embedding:", tuple(
    extended_model.model.decoder.embed_tokens.weight.shape
))
print("Expanded LM head:", tuple(extended_model.lm_head.weight.shape))

if initialization_errors:
    print("\nFirst initialization errors:")
    for error in initialization_errors[:10]:
        print(error)

assert initialized_candidates == len(new_candidate_words), (
    "Not all candidates were initialized. Inspect initialization_errors."
)
assert not initialization_errors

print("\nAll new rows initialized successfully.")

Candidates requested: 95
Candidates initialized: 95
Initialization errors: 0
Expanded decoder embedding: (122767, 1024)
Expanded LM head: (122767, 1024)

All new rows initialized successfully.


In [130]:
# Section 9.19: Validate original and newly initialized vocabulary rows

import torch

decoder_weights = extended_model.model.decoder.embed_tokens.weight
lm_head_weights = extended_model.lm_head.weight

old_vocab_size = BASE_TARGET_VOCAB_SIZE
new_vocab_size = old_vocab_size + len(new_candidate_words)

# Check original rows remain unchanged.
decoder_original_unchanged = torch.equal(
    decoder_weights[:old_vocab_size],
    original_decoder_weights.to(
        device=decoder_weights.device,
        dtype=decoder_weights.dtype,
    ),
)

lm_head_original_unchanged = torch.equal(
    lm_head_weights[:old_vocab_size],
    original_lm_head_weights.to(
        device=lm_head_weights.device,
        dtype=lm_head_weights.dtype,
    ),
)

# Check all newly initialized values are finite.
new_decoder_rows = decoder_weights[old_vocab_size:new_vocab_size]
new_lm_head_rows = lm_head_weights[old_vocab_size:new_vocab_size]

decoder_finite = torch.isfinite(new_decoder_rows).all().item()
lm_head_finite = torch.isfinite(new_lm_head_rows).all().item()

print("Original decoder rows unchanged:", decoder_original_unchanged)
print("Original LM-head rows unchanged:", lm_head_original_unchanged)
print("New decoder rows finite:", decoder_finite)
print("New LM-head rows finite:", lm_head_finite)
print("New decoder rows shape:", tuple(new_decoder_rows.shape))
print("New LM-head rows shape:", tuple(new_lm_head_rows.shape))
print(
    "Decoder embedding and LM head remain separate:",
    decoder_weights.data_ptr() != lm_head_weights.data_ptr(),
)

assert decoder_original_unchanged
assert lm_head_original_unchanged
assert decoder_finite
assert lm_head_finite
assert decoder_weights.data_ptr() != lm_head_weights.data_ptr()

print("\nAll vocabulary initialization checks passed.")

Original decoder rows unchanged: True
Original LM-head rows unchanged: True
New decoder rows finite: True
New LM-head rows finite: True
New decoder rows shape: (95, 1024)
New LM-head rows shape: (95, 1024)
Decoder embedding and LM head remain separate: True

All vocabulary initialization checks passed.


# **section 10:adding lora to vocabulary extended model** 

In [132]:
# Section 10.1: Locate baseline adapter files

import os

baseline_dir = "/kaggle/working/santali_baseline_lora"

for root, dirs, files in os.walk(baseline_dir):
    level = root.replace(baseline_dir, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for filename in sorted(files):
        print(f"{indent}  {filename}")

santali_baseline_lora/
  baseline_evaluation_metadata.json
  baseline_test_metrics.json
  baseline_test_predictions.csv
  training_history.json
  best_adapter/
    README.md
    adapter_config.json
    adapter_model.safetensors


In [133]:
# Section 10.1: Load the exact baseline LoRA configuration

import os
import json

adapter_dir = "/kaggle/working/santali_baseline_lora/best_adapter"
config_path = os.path.join(adapter_dir, "adapter_config.json")

with open(config_path, "r", encoding="utf-8") as f:
    baseline_lora_config = json.load(f)

keys_to_check = [
    "peft_type",
    "task_type",
    "r",
    "lora_alpha",
    "target_modules",
    "lora_dropout",
    "bias",
    "modules_to_save",
    "fan_in_fan_out",
    "init_lora_weights",
]

print("Baseline adapter configuration loaded successfully.\n")

for key in keys_to_check:
    print(f"{key}: {baseline_lora_config.get(key, '<not specified>')}")

Baseline adapter configuration loaded successfully.

peft_type: LORA
task_type: SEQ_2_SEQ_LM
r: 16
lora_alpha: 32
target_modules: ['v_proj', 'q_proj']
lora_dropout: 0.05
bias: none
modules_to_save: None
fan_in_fan_out: False
init_lora_weights: True


In [134]:
# Section 10.2: Prepare the exact baseline LoRA configuration

from peft import LoraConfig, TaskType

assert baseline_lora_config["r"] == 16
assert baseline_lora_config["lora_alpha"] == 32
assert set(baseline_lora_config["target_modules"]) == {"q_proj", "v_proj"}
assert baseline_lora_config["lora_dropout"] == 0.05

extended_lora_config = LoraConfig(
    r=baseline_lora_config["r"],
    lora_alpha=baseline_lora_config["lora_alpha"],
    target_modules=baseline_lora_config["target_modules"],
    lora_dropout=baseline_lora_config["lora_dropout"],
    bias=baseline_lora_config["bias"],
    task_type=TaskType.SEQ_2_SEQ_LM,
    fan_in_fan_out=baseline_lora_config["fan_in_fan_out"],
    init_lora_weights=baseline_lora_config["init_lora_weights"],
)

print("Extended-experiment LoRA configuration prepared.")
print("Rank:", extended_lora_config.r)
print("Alpha:", extended_lora_config.lora_alpha)
print("Target modules:", extended_lora_config.target_modules)
print("Dropout:", extended_lora_config.lora_dropout)
print("Bias:", extended_lora_config.bias)
print("Task type:", extended_lora_config.task_type)
print("Initialization:", extended_lora_config.init_lora_weights)

Extended-experiment LoRA configuration prepared.
Rank: 16
Alpha: 32
Target modules: {'v_proj', 'q_proj'}
Dropout: 0.05
Bias: none
Task type: TaskType.SEQ_2_SEQ_LM
Initialization: True


In [135]:
# Section 10.3: Inspect parameters before attaching LoRA

decoder_embedding = extended_model.model.decoder.embed_tokens
lm_head = extended_model.lm_head

print("Decoder embedding requires_grad:", decoder_embedding.weight.requires_grad)
print("LM head requires_grad:", lm_head.weight.requires_grad)

print("Decoder embedding shape:", tuple(decoder_embedding.weight.shape))
print("LM head shape:", tuple(lm_head.weight.shape))

print("New vocabulary row range:")
print("  Start ID:", BASE_TARGET_VOCAB_SIZE)
print("  End ID (exclusive):", BASE_TARGET_VOCAB_SIZE + len(new_candidate_words))

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert decoder_embedding.weight.shape[0] == 122767
assert lm_head.weight.shape[0] == 122767
assert len(new_candidate_words) == 95

print("\nPre-LoRA inspection passed.")

Decoder embedding requires_grad: True
LM head requires_grad: True
Decoder embedding shape: (122767, 1024)
LM head shape: (122767, 1024)
New vocabulary row range:
  Start ID: 122672
  End ID (exclusive): 122767

PyTorch version: 2.11.0+cu128
CUDA available: True

Pre-LoRA inspection passed.


In [136]:
# Section 10.4: Attach baseline-matched LoRA adapters

from peft import get_peft_model

# Ensure LoRA has not already been attached.
assert not hasattr(extended_model, "peft_config"), (
    "LoRA appears to be attached already. Do not attach it twice."
)

extended_model = get_peft_model(
    extended_model,
    extended_lora_config,
)

print("LoRA adapters attached.")
print("\nTrainable parameter summary:")
extended_model.print_trainable_parameters()

print("\nAdapter configuration:")
print(extended_model.peft_config["default"])

# Confirm expected attention projections were adapted.
lora_module_names = [
    name for name, _ in extended_model.named_modules()
    if "lora_A" in name or "lora_B" in name
]

print("\nNumber of LoRA parameter modules:", len(lora_module_names))
print("Sample LoRA modules:")
for name in lora_module_names[:10]:
    print(" ", name)

assert len(lora_module_names) > 0
print("\nLoRA attachment check passed.")

LoRA adapters attached.

Trainable parameter summary:
trainable params: 3,538,944 || all params: 1,211,830,272 || trainable%: 0.2920

Adapter configuration:
LoraConfig(task_type=<TaskType.SEQ_2_SEQ_LM: 'SEQ_2_SEQ_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path='ai4bharat/indictrans2-indic-indic-1B', revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_conf

In [137]:
# Section 10.5: Inspect embedding and LM-head trainability after LoRA

decoder_embedding = (
    extended_model.base_model.model.model.decoder.embed_tokens
)
lm_head = extended_model.base_model.model.lm_head

print("Decoder embedding shape:", tuple(decoder_embedding.weight.shape))
print("LM head shape:", tuple(lm_head.weight.shape))

print("\nDecoder embedding requires_grad:",
      decoder_embedding.weight.requires_grad)
print("LM head requires_grad:",
      lm_head.weight.requires_grad)

print("\nTrainable parameter names containing embedding/head:")
for name, parameter in extended_model.named_parameters():
    if any(term in name.lower() for term in [
        "embed_tokens",
        "lm_head",
    ]):
        if parameter.requires_grad:
            print(name, tuple(parameter.shape))

print("\nNew vocabulary IDs:")
print("Start:", BASE_TARGET_VOCAB_SIZE)
print("End (exclusive):", BASE_TARGET_VOCAB_SIZE + len(new_candidate_words))

print("\nLoRA parameter count:")
lora_trainable_count = sum(
    parameter.numel()
    for name, parameter in extended_model.named_parameters()
    if parameter.requires_grad and "lora_" in name.lower()
)
print(lora_trainable_count)

assert lora_trainable_count == 3_538_944

print("\nInspection complete.")

Decoder embedding shape: (122767, 1024)
LM head shape: (122767, 1024)

Decoder embedding requires_grad: False
LM head requires_grad: False

Trainable parameter names containing embedding/head:

New vocabulary IDs:
Start: 122672
End (exclusive): 122767

LoRA parameter count:
3538944

Inspection complete.


In [138]:
# Section 10.6: Verify expanded layer access after LoRA attachment

peft_base = extended_model.base_model.model

decoder_embedding = peft_base.model.decoder.embed_tokens
lm_head = peft_base.lm_head

print("Decoder embedding module:", type(decoder_embedding).__name__)
print("Decoder embedding shape:", tuple(decoder_embedding.weight.shape))
print("Decoder embedding frozen:", not decoder_embedding.weight.requires_grad)

print("\nLM head module:", type(lm_head).__name__)
print("LM head shape:", tuple(lm_head.weight.shape))
print("LM head frozen:", not lm_head.weight.requires_grad)

print("\nOriginal vocabulary size:", BASE_TARGET_VOCAB_SIZE)
print("New vocabulary rows:", len(new_candidate_words))
print(
    "Extended vocabulary size:",
    BASE_TARGET_VOCAB_SIZE + len(new_candidate_words),
)

assert decoder_embedding.weight.shape[0] == 122767
assert lm_head.weight.shape[0] == 122767
assert not decoder_embedding.weight.requires_grad
assert not lm_head.weight.requires_grad

print("\nLayer-access checks passed.")

Decoder embedding module: Embedding
Decoder embedding shape: (122767, 1024)
Decoder embedding frozen: True

LM head module: Linear
LM head shape: (122767, 1024)
LM head frozen: True

Original vocabulary size: 122672
New vocabulary rows: 95
Extended vocabulary size: 122767

Layer-access checks passed.


In [139]:
# Section 10.7: Inspect IndicTrans forward implementation

import inspect

base_model = extended_model.base_model.model

print("Model class:", type(base_model).__name__)

print("\nModel forward signature:")
print(inspect.signature(base_model.forward))

print("\nForward method source (first 100 lines):")
forward_source = inspect.getsource(type(base_model).forward)
forward_lines = forward_source.splitlines()

for line_number, line in enumerate(forward_lines[:100], start=1):
    print(f"{line_number:03d}: {line}")

print("\nDecoder class:", type(base_model.model.decoder).__name__)
print("Decoder forward signature:")
print(inspect.signature(base_model.model.decoder.forward))

Model class: IndicTransForConditionalGeneration

Model forward signature:
(input_ids: Optional[torch.LongTensor] = None, attention_mask: Optional[torch.Tensor] = None, decoder_input_ids: Optional[torch.LongTensor] = None, decoder_attention_mask: Optional[torch.LongTensor] = None, head_mask: Optional[torch.Tensor] = None, decoder_head_mask: Optional[torch.Tensor] = None, cross_attn_head_mask: Optional[torch.Tensor] = None, encoder_outputs: Optional[Tuple[Tuple[torch.FloatTensor]]] = None, past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None, inputs_embeds: Optional[torch.FloatTensor] = None, decoder_inputs_embeds: Optional[torch.FloatTensor] = None, labels: Optional[torch.LongTensor] = None, use_cache: Optional[bool] = None, output_attentions: Optional[bool] = None, output_hidden_states: Optional[bool] = None, return_dict: Optional[bool] = None) -> Union[Tuple[torch.Tensor], transformers.modeling_outputs.Seq2SeqLMOutput]

Forward method source (first 100 lines):
001:     de

In [140]:
# Section 10.8: Inspect embedding accessors and decoder embedding usage

import inspect

base_model = extended_model.base_model.model

print("Input embedding accessor:")
print(inspect.signature(base_model.get_input_embeddings))
print("Current input embedding:", type(base_model.get_input_embeddings()).__name__)

print("\nOutput embedding accessor:")
print(inspect.signature(base_model.get_output_embeddings))
print("Current output embedding:", type(base_model.get_output_embeddings()).__name__)

print("\nDecoder embedding attribute:")
print("Has embed_tokens:", hasattr(base_model.model.decoder, "embed_tokens"))
print("Embedding type:", type(base_model.model.decoder.embed_tokens).__name__)

print("\nDecoder forward source (first 100 lines):")
decoder_source = inspect.getsource(type(base_model.model.decoder).forward)

for line_number, line in enumerate(decoder_source.splitlines()[:100], start=1):
    print(f"{line_number:03d}: {line}")

print("\nModel embedding accessor implementations:")
print("get_input_embeddings:")
print(inspect.getsource(type(base_model).get_input_embeddings))
print("\nget_output_embeddings:")
print(inspect.getsource(type(base_model).get_output_embeddings))

Input embedding accessor:
()
Current input embedding: Embedding

Output embedding accessor:
()
Current output embedding: Linear

Decoder embedding attribute:
Has embed_tokens: True
Embedding type: Embedding

Decoder forward source (first 100 lines):
001:     def forward(
002:         self,
003:         input_ids: Optional[torch.Tensor] = None,
004:         attention_mask: Optional[torch.Tensor] = None,
005:         encoder_hidden_states: Optional[torch.Tensor] = None,
006:         encoder_attention_mask: Optional[torch.Tensor] = None,
007:         head_mask: Optional[torch.Tensor] = None,
008:         cross_attn_head_mask: Optional[torch.Tensor] = None,
009:         past_key_values: Optional[List[torch.FloatTensor]] = None,
010:         inputs_embeds: Optional[torch.Tensor] = None,
011:         use_cache: Optional[bool] = None,
012:         output_attentions: Optional[bool] = None,
013:         output_hidden_states: Optional[bool] = None,
014:         return_dict: Optional[bool] = None

In [141]:
# Section 10.9: Verify decoder embedding lookup and vocabulary configuration

import inspect

base_model = extended_model.base_model.model
decoder = base_model.model.decoder

decoder_source = inspect.getsource(type(decoder).forward).splitlines()

print("Decoder embedding lookup lines:")
for i, line in enumerate(decoder_source, start=1):
    if "embed_tokens" in line or "inputs_embeds" in line:
        start = max(0, i - 4)
        end = min(len(decoder_source), i + 3)

        print(f"\n--- Lines {start + 1} to {end} ---")
        for j in range(start, end):
            print(f"{j + 1:03d}: {decoder_source[j]}")

print("\nVocabulary configuration:")
for key in [
    "vocab_size",
    "encoder_vocab_size",
    "decoder_vocab_size",
    "pad_token_id",
    "decoder_start_token_id",
    "eos_token_id",
    "tie_word_embeddings",
    "share_decoder_input_output_embed",
]:
    print(f"{key}: {getattr(base_model.config, key, None)}")

print("\nActual layer dimensions:")
print("Encoder embedding:", tuple(base_model.model.encoder.embed_tokens.weight.shape))
print("Decoder embedding:", tuple(decoder.embed_tokens.weight.shape))
print("LM head:", tuple(base_model.lm_head.weight.shape))

print("\nCandidate ID range:")
print("Original target vocabulary:", BASE_TARGET_VOCAB_SIZE)
print("Number of added candidates:", len(candidate_to_id))
print("Minimum new ID:", min(candidate_to_id.values()))
print("Maximum new ID:", max(candidate_to_id.values()))

Decoder embedding lookup lines:

--- Lines 7 to 13 ---
007:         head_mask: Optional[torch.Tensor] = None,
008:         cross_attn_head_mask: Optional[torch.Tensor] = None,
009:         past_key_values: Optional[List[torch.FloatTensor]] = None,
010:         inputs_embeds: Optional[torch.Tensor] = None,
011:         use_cache: Optional[bool] = None,
012:         output_attentions: Optional[bool] = None,
013:         output_hidden_states: Optional[bool] = None,

--- Lines 64 to 70 ---
064: 
065:                 If `past_key_values` are used, the user can optionally input only the last `decoder_input_ids` (those
066:                 that don't have their past key value states given to this model) of shape `(batch_size, 1)` instead of
067:                 all `decoder_input_ids` of shape `(batch_size, sequence_length)`. inputs_embeds (`torch.FloatTensor` of
068:                 shape `(batch_size, sequence_length, hidden_size)`, *optional*): Optionally, instead of passing
069:          

In [142]:
# Section 10.10: Snapshot extended rows before creating trainable row modules

import torch

base_model = extended_model.base_model.model

decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

old_vocab_size = BASE_TARGET_VOCAB_SIZE
new_vocab_size = len(candidate_to_id)

assert decoder_embedding.weight.shape[0] == old_vocab_size + new_vocab_size
assert lm_head.weight.shape[0] == old_vocab_size + new_vocab_size

# Snapshot original rows and initialized extension rows.
# These are verification copies, not trainable parameters.
decoder_old_rows_before = (
    decoder_embedding.weight[:old_vocab_size].detach().cpu().clone()
)
lm_head_old_rows_before = (
    lm_head.weight[:old_vocab_size].detach().cpu().clone()
)

decoder_new_rows_before = (
    decoder_embedding.weight[old_vocab_size:].detach().cpu().clone()
)
lm_head_new_rows_before = (
    lm_head.weight[old_vocab_size:].detach().cpu().clone()
)

print("Decoder embedding shape:", tuple(decoder_embedding.weight.shape))
print("LM head shape:", tuple(lm_head.weight.shape))

print("\nOriginal vocabulary rows:", old_vocab_size)
print("New vocabulary rows:", new_vocab_size)

print("\nDecoder old rows require_grad:", decoder_embedding.weight.requires_grad)
print("LM head old rows require_grad:", lm_head.weight.requires_grad)

print("\nDecoder new rows finite:",
      bool(torch.isfinite(decoder_new_rows_before).all()))
print("LM head new rows finite:",
      bool(torch.isfinite(lm_head_new_rows_before).all()))

print("\nDecoder new-row standard deviation:",
      decoder_new_rows_before.float().std().item())
print("LM-head new-row standard deviation:",
      lm_head_new_rows_before.float().std().item())

print("\nSnapshots saved for later verification.")

Decoder embedding shape: (122767, 1024)
LM head shape: (122767, 1024)

Original vocabulary rows: 122672
New vocabulary rows: 95

Decoder old rows require_grad: False
LM head old rows require_grad: False

Decoder new rows finite: True
LM head new rows finite: True

Decoder new-row standard deviation: 0.0387381874024868
LM-head new-row standard deviation: 0.033125247806310654

Snapshots saved for later verification.


In [143]:
# Section 10.11: Define modules for frozen original rows and trainable extension rows

import torch
import torch.nn as nn
import torch.nn.functional as F


class FrozenBaseExtendedEmbedding(nn.Module):
    """
    Original vocabulary rows stay frozen.
    Only the appended vocabulary rows are trainable.
    """

    def __init__(self, original_weight, new_weight, padding_idx=None):
        super().__init__()

        self.original_vocab_size = original_weight.shape[0]
        self.num_new_tokens = new_weight.shape[0]
        self.embedding_dim = original_weight.shape[1]
        self.padding_idx = padding_idx

        # Frozen original vocabulary rows
        self.original_weight = nn.Parameter(
            original_weight.detach().clone(),
            requires_grad=False
        )

        # Trainable extension rows
        self.new_weight = nn.Parameter(
            new_weight.detach().clone(),
            requires_grad=True
        )

    @property
    def weight(self):
        """Combined weight view for inspection and compatibility."""
        return torch.cat(
            [self.original_weight, self.new_weight],
            dim=0
        )

    @property
    def num_embeddings(self):
        return self.original_vocab_size + self.num_new_tokens

    def forward(self, input_ids):
        combined_weight = torch.cat(
            [self.original_weight, self.new_weight],
            dim=0
        )

        return F.embedding(
            input_ids,
            combined_weight,
            padding_idx=self.padding_idx
        )


class FrozenBaseExtendedLMHead(nn.Module):
    """
    Original output-projection rows stay frozen.
    Only the appended vocabulary rows are trainable.
    """

    def __init__(self, original_weight, new_weight, bias=None):
        super().__init__()

        self.original_vocab_size = original_weight.shape[0]
        self.num_new_tokens = new_weight.shape[0]
        self.hidden_size = original_weight.shape[1]

        self.original_weight = nn.Parameter(
            original_weight.detach().clone(),
            requires_grad=False
        )

        self.new_weight = nn.Parameter(
            new_weight.detach().clone(),
            requires_grad=True
        )

        if bias is None:
            self.register_parameter("bias", None)
        else:
            self.bias = nn.Parameter(
                bias.detach().clone(),
                requires_grad=False
            )

    @property
    def weight(self):
        return torch.cat(
            [self.original_weight, self.new_weight],
            dim=0
        )

    @property
    def out_features(self):
        return self.original_vocab_size + self.num_new_tokens

    @property
    def in_features(self):
        return self.hidden_size

    def forward(self, hidden_states):
        combined_weight = torch.cat(
            [self.original_weight, self.new_weight],
            dim=0
        )

        return F.linear(
            hidden_states,
            combined_weight,
            self.bias
        )


print("Module classes defined successfully.")

print("\nEmbedding module supports:")
print("- Frozen original rows")
print("- Trainable new rows")
print("- Padding index:", getattr(
    extended_model.base_model.model.model.decoder.embed_tokens,
    "padding_idx",
    None
))

print("\nLM-head module supports:")
print("- Frozen original output rows")
print("- Trainable new output rows")
print("- Bias:", extended_model.base_model.model.lm_head.bias is not None)

print("\nNo model layers have been replaced yet.")

Module classes defined successfully.

Embedding module supports:
- Frozen original rows
- Trainable new rows
- Padding index: 1

LM-head module supports:
- Frozen original output rows
- Trainable new output rows
- Bias: False

No model layers have been replaced yet.


### Section 10.11 — Define Modules for Frozen and Trainable Vocabulary Rows

To ensure that vocabulary extension does not modify the original pretrained vocabulary, we define custom modules that separate the original rows from the newly added rows.

- **Frozen original rows:** The original decoder embedding and LM-head weights remain unchanged.
- **Trainable extension rows:** Only the 95 newly added vocabulary rows are trainable in each module.
- **Decoder embedding:** Maps the extended target token IDs to their embedding vectors.
- **LM head:** Produces output logits for both the original vocabulary and the 95 new tokens.

The modules concatenate the frozen original weights and trainable extension weights during the forward pass. This allows gradients to update the new rows without updating the original rows, including through AdamW weight decay.

At this stage, only the module classes are defined. The existing model has not yet been modified.


In [144]:
# Section 10.12: Replace decoder embedding and LM head with split-weight modules

base_model = extended_model.base_model.model
decoder = base_model.model.decoder

# Keep references to the existing expanded layers
old_decoder_embedding = decoder.embed_tokens
old_lm_head = base_model.lm_head

# Split the initialized weights into original and newly added rows
decoder_original_weight = old_decoder_embedding.weight[:BASE_TARGET_VOCAB_SIZE].detach()
decoder_new_weight = old_decoder_embedding.weight[BASE_TARGET_VOCAB_SIZE:].detach()

lm_original_weight = old_lm_head.weight[:BASE_TARGET_VOCAB_SIZE].detach()
lm_new_weight = old_lm_head.weight[BASE_TARGET_VOCAB_SIZE:].detach()

# Build replacement modules on the same device and with the same dtype
new_decoder_embedding = FrozenBaseExtendedEmbedding(
    original_weight=decoder_original_weight,
    new_weight=decoder_new_weight,
    padding_idx=old_decoder_embedding.padding_idx,
).to(
    device=old_decoder_embedding.weight.device,
    dtype=old_decoder_embedding.weight.dtype,
)

new_lm_head = FrozenBaseExtendedLMHead(
    original_weight=lm_original_weight,
    new_weight=lm_new_weight,
    bias=old_lm_head.bias,
).to(
    device=old_lm_head.weight.device,
    dtype=old_lm_head.weight.dtype,
)

# Replace only the decoder embedding and output projection
decoder.embed_tokens = new_decoder_embedding
base_model.lm_head = new_lm_head

# Verify module shapes and trainability
print("Decoder embedding replaced:", type(decoder.embed_tokens).__name__)
print("LM head replaced:", type(base_model.lm_head).__name__)

print("\nDecoder embedding:")
print("Original rows:", decoder.embed_tokens.original_weight.shape[0])
print("New rows:", decoder.embed_tokens.new_weight.shape[0])
print("Original rows trainable:", decoder.embed_tokens.original_weight.requires_grad)
print("New rows trainable:", decoder.embed_tokens.new_weight.requires_grad)

print("\nLM head:")
print("Original rows:", base_model.lm_head.original_weight.shape[0])
print("New rows:", base_model.lm_head.new_weight.shape[0])
print("Original rows trainable:", base_model.lm_head.original_weight.requires_grad)
print("New rows trainable:", base_model.lm_head.new_weight.requires_grad)

print("\nOutput vocabulary size:", base_model.lm_head.out_features)
print("Configured decoder vocabulary:", base_model.config.decoder_vocab_size)

Decoder embedding replaced: FrozenBaseExtendedEmbedding
LM head replaced: FrozenBaseExtendedLMHead

Decoder embedding:
Original rows: 122672
New rows: 95
Original rows trainable: False
New rows trainable: True

LM head:
Original rows: 122672
New rows: 95
Original rows trainable: False
New rows trainable: True

Output vocabulary size: 122767
Configured decoder vocabulary: 122767


In [145]:
# Section 10.13: Verify original weights and LoRA integrity

base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

# Compare frozen original rows with snapshots from Section 10.10
decoder_old_unchanged = torch.equal(
    decoder_embedding.original_weight.detach().cpu(),
    decoder_old_rows_before,
)

lm_old_unchanged = torch.equal(
    lm_head.original_weight.detach().cpu(),
    lm_head_old_rows_before,
)

# Compare extension initialization with its snapshots
decoder_new_unchanged = torch.equal(
    decoder_embedding.new_weight.detach().cpu(),
    decoder_new_rows_before,
)

lm_new_unchanged = torch.equal(
    lm_head.new_weight.detach().cpu(),
    lm_head_new_rows_before,
)

# Check LoRA modules still exist
lora_parameters = [
    (name, param)
    for name, param in extended_model.named_parameters()
    if "lora_" in name
]

lora_trainable_count = sum(
    param.numel() for _, param in lora_parameters if param.requires_grad
)

print("Original decoder rows unchanged:", decoder_old_unchanged)
print("Original LM-head rows unchanged:", lm_old_unchanged)
print("Initialized decoder extension unchanged:", decoder_new_unchanged)
print("Initialized LM-head extension unchanged:", lm_new_unchanged)

print("\nLoRA parameter tensors found:", len(lora_parameters))
print("Trainable LoRA parameters:", lora_trainable_count)

print("\nTrainable extension parameters:")
print("Decoder new rows:", decoder_embedding.new_weight.requires_grad)
print("LM-head new rows:", lm_head.new_weight.requires_grad)

assert decoder_old_unchanged
assert lm_old_unchanged
assert decoder_new_unchanged
assert lm_new_unchanged
assert lora_trainable_count > 0

print("\nPASS: Original and initialized rows are preserved; LoRA parameters remain present.")

Original decoder rows unchanged: True
Original LM-head rows unchanged: True
Initialized decoder extension unchanged: True
Initialized LM-head extension unchanged: True

LoRA parameter tensors found: 216
Trainable LoRA parameters: 3538944

Trainable extension parameters:
Decoder new rows: True
LM-head new rows: True

PASS: Original and initialized rows are preserved; LoRA parameters remain present.


In [146]:
# Section 10.14: Test forward pass and gradient flow

import torch

base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

extended_model.train()
extended_model.zero_grad(set_to_none=True)

device = next(extended_model.parameters()).device

# Create a small synthetic source batch using valid original source IDs.
encoder_vocab_size = base_model.config.encoder_vocab_size
pad_id = base_model.config.pad_token_id
eos_id = base_model.config.eos_token_id

source_ids = torch.tensor(
    [
        [5, 10, 20, eos_id],
        [7, 12, 25, eos_id],
    ],
    dtype=torch.long,
    device=device,
)

source_attention_mask = torch.ones_like(source_ids)

# Include one new target ID in the labels.
first_new_id = BASE_TARGET_VOCAB_SIZE

labels = torch.tensor(
    [
        [first_new_id, 10, eos_id],
        [20, first_new_id + 1, eos_id],
    ],
    dtype=torch.long,
    device=device,
)

# Run forward pass and loss computation.
outputs = extended_model(
    input_ids=source_ids,
    attention_mask=source_attention_mask,
    labels=labels,
    use_cache=False,
    return_dict=True,
)

print("Logits shape:", tuple(outputs.logits.shape))
print("Loss:", outputs.loss.item())
print("Loss finite:", bool(torch.isfinite(outputs.loss)))
print("Logits finite:", bool(torch.isfinite(outputs.logits).all()))

assert outputs.logits.shape[-1] == BASE_TARGET_VOCAB_SIZE + len(candidate_to_id)
assert torch.isfinite(outputs.loss)
assert torch.isfinite(outputs.logits).all()

# Backpropagate without updating any weights.
outputs.loss.backward()

decoder_grad = decoder_embedding.new_weight.grad
lm_head_grad = lm_head.new_weight.grad

lora_grads = [
    param.grad
    for name, param in extended_model.named_parameters()
    if "lora_" in name and param.requires_grad and param.grad is not None
]

print("\nGradient checks:")
print(
    "Decoder extension gradient exists:",
    decoder_grad is not None,
)
print(
    "LM-head extension gradient exists:",
    lm_head_grad is not None,
)
print("LoRA tensors with gradients:", len(lora_grads))

if decoder_grad is not None:
    print(
        "Decoder extension gradient finite:",
        bool(torch.isfinite(decoder_grad).all()),
    )
    print("Decoder extension gradient norm:", decoder_grad.float().norm().item())

if lm_head_grad is not None:
    print(
        "LM-head extension gradient finite:",
        bool(torch.isfinite(lm_head_grad).all()),
    )
    print("LM-head extension gradient norm:", lm_head_grad.float().norm().item())

if lora_grads:
    print(
        "All LoRA gradients finite:",
        all(bool(torch.isfinite(grad).all()) for grad in lora_grads),
    )

print("\nNo optimizer step was performed.")

Logits shape: (2, 3, 122767)
Loss: 9.484375
Loss finite: True
Logits finite: True

Gradient checks:
Decoder extension gradient exists: True
LM-head extension gradient exists: True
LoRA tensors with gradients: 216
Decoder extension gradient finite: True
Decoder extension gradient norm: 8.985474586486816
LM-head extension gradient finite: True
LM-head extension gradient norm: 7.6920084953308105
All LoRA gradients finite: True

No optimizer step was performed.


In [147]:
# Section 10.15: Audit trainable parameters and frozen vocabulary weights

base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

# Original vocabulary weights must remain frozen.
assert decoder_embedding.original_weight.requires_grad is False
assert lm_head.original_weight.requires_grad is False

assert decoder_embedding.original_weight.grad is None
assert lm_head.original_weight.grad is None

# Count the intended trainable parameter groups.
lora_params = [
    param
    for name, param in extended_model.named_parameters()
    if "lora_" in name and param.requires_grad
]

extension_params = [
    decoder_embedding.new_weight,
    lm_head.new_weight,
]

other_trainable_params = [
    (name, param)
    for name, param in extended_model.named_parameters()
    if param.requires_grad
    and "lora_" not in name
    and param is not decoder_embedding.new_weight
    and param is not lm_head.new_weight
]

lora_count = sum(p.numel() for p in lora_params)
extension_count = sum(p.numel() for p in extension_params)
other_count = sum(p.numel() for _, p in other_trainable_params)

print("Trainable LoRA parameters:", lora_count)
print("Trainable decoder extension parameters:", decoder_embedding.new_weight.numel())
print("Trainable LM-head extension parameters:", lm_head.new_weight.numel())
print("Total trainable extension parameters:", extension_count)
print("Unexpected trainable parameters:", other_count)

print("\nFrozen original decoder rows have no gradients:",
      decoder_embedding.original_weight.grad is None)
print("Frozen original LM-head rows have no gradients:",
      lm_head.original_weight.grad is None)

print("\nUnexpected parameter names:")
for name, _ in other_trainable_params[:20]:
    print(name)

assert lora_count == 3_538_944
assert extension_count == 2 * len(candidate_to_id) * decoder_embedding.embedding_dim
assert other_count == 0

print("\nPASS: Only LoRA parameters and the 190 new vocabulary rows are trainable.")

Trainable LoRA parameters: 3538944
Trainable decoder extension parameters: 97280
Trainable LM-head extension parameters: 97280
Total trainable extension parameters: 194560
Unexpected trainable parameters: 0

Frozen original decoder rows have no gradients: True
Frozen original LM-head rows have no gradients: True

Unexpected parameter names:

PASS: Only LoRA parameters and the 190 new vocabulary rows are trainable.


In [148]:
# Section 10.16: Clear test gradients and configure optimizer

import torch

# Clear all gradients left by the synthetic forward/backward test.
extended_model.zero_grad(set_to_none=True)

# Explicitly include only LoRA parameters and the new vocabulary rows.
base_model = extended_model.base_model.model

decoder_new_weight = base_model.model.decoder.embed_tokens.new_weight
lm_head_new_weight = base_model.lm_head.new_weight

lora_params = [
    param
    for name, param in extended_model.named_parameters()
    if "lora_" in name and param.requires_grad
]

extension_params = [
    decoder_new_weight,
    lm_head_new_weight,
]

trainable_params = lora_params + extension_params

# Verify no duplicate parameter objects.
parameter_ids = [id(param) for param in trainable_params]
assert len(parameter_ids) == len(set(parameter_ids))

# Use the baseline optimizer hyperparameters.
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-4,
    weight_decay=0.01,
)

# Verify the optimizer contains exactly the intended parameters.
optimizer_param_ids = {
    id(param)
    for group in optimizer.param_groups
    for param in group["params"]
}

assert optimizer_param_ids == set(parameter_ids)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])
print("Optimizer parameter tensors:", len(parameter_ids))
print("Total optimizer parameters:", sum(p.numel() for p in trainable_params))

print("\nAll gradients cleared:",
      all(p.grad is None for p in extended_model.parameters()))

print("Frozen original decoder rows have no gradients:",
      decoder_new_weight.grad is None
      and base_model.model.decoder.embed_tokens.original_weight.grad is None)

print("Frozen original LM-head rows have no gradients:",
      lm_head_new_weight.grad is None
      and base_model.lm_head.original_weight.grad is None)

print("\nOptimizer configured. No training step has been performed.")

Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 0.01
Optimizer parameter tensors: 218
Total optimizer parameters: 3733504

All gradients cleared: True
Frozen original decoder rows have no gradients: True
Frozen original LM-head rows have no gradients: True

Optimizer configured. No training step has been performed.


In [149]:
# Section 10.17: Audit extended target tokenization on real examples

# Inspect existing dataset variables without modifying them.
print("Available dataset variables:")
for name in ["train_df", "val_df", "test_df",
             "train_data", "val_data", "train_dataset", "val_dataset"]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: type={type(obj).__name__}, length={len(obj) if hasattr(obj, '__len__') else 'N/A'}")

print("\nTokenizer function available:",
      "tokenize_target_with_extension_v2" in globals())
print("Candidate mapping available:",
      "candidate_to_id" in globals())

assert "tokenize_target_with_extension_v2" in globals()
assert "candidate_to_id" in globals()

# Identify likely DataFrame split variables without guessing their column names.
for name in ["train_df", "val_df"]:
    if name in globals():
        obj = globals()[name]
        if hasattr(obj, "columns"):
            print(f"\n{name} columns:", list(obj.columns))
            print(obj.head(2).to_dict(orient="records"))

Available dataset variables:
train_df: type=DataFrame, length=17883
val_df: type=DataFrame, length=993
test_df: type=DataFrame, length=994
train_dataset: type=TranslationDataset, length=17883
val_dataset: type=TranslationDataset, length=993

Tokenizer function available: True
Candidate mapping available: True

train_df columns: ['English', 'Santali']
[{'English': "Instacart's Shopper app looks like a green Instacart bag on a white background.", 'Santali': 'ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾'}, {'English': "Later he became founder and leader of the Kerala People's Party, a political party in the Indian state of Kerala.", 'Santali': 'ᱛᱟᱭᱚᱢ ᱛᱮ ᱩᱱᱤ ᱵᱷᱟᱨᱚᱛ ᱨᱮᱭᱟᱜ ᱠᱮᱨᱟᱞᱟ ᱯᱚᱱᱚᱛ ᱨᱮᱭᱟᱜ ᱢᱤᱫ ᱨᱟᱡᱽ ᱟᱹᱨᱤ ᱜᱟᱫᱮᱞ ᱠᱮᱨᱟᱞᱟ ᱯᱤᱯᱚᱞᱥ ᱯᱟᱨᱴᱤ ᱨᱮᱱ ᱛᱷᱟᱯᱱᱟᱣᱤᱡ ᱟᱨ ᱱᱮᱛᱟᱭ ᱦᱩᱭ ᱞᱮᱱᱟ ᱾'}]

val_df columns: ['English', 'Santali']
[{'English': "According to Minhaj, Iltutmish re-asserted Delhi's control over Badaun, Awadh, Banaras, and Siwalik in a series of campaigns.", 'Santali': 'ᱢᱤᱱ

In [151]:
# Section 10.18: Validate extended target tokenization on real Santali data

EXTENDED_VOCAB_SIZE = BASE_TARGET_VOCAB_SIZE + len(candidate_to_id)

eos_id = extended_model.config.eos_token_id
assert eos_id is not None, "EOS ID is missing from model configuration."

def audit_target_examples(df, split_name, n=100):
    total_tokens = 0
    new_token_occurrences = 0
    examples_with_new_tokens = 0
    eos_errors = 0
    invalid_id_examples = 0

    for text in df["Santali"].head(n).fillna("").astype(str):
        ids = tokenize_target_with_extension_v2(text, add_eos=True)

        if not ids or ids[-1] != eos_id or ids.count(eos_id) != 1:
            eos_errors += 1

        if any(i < 0 or i >= EXTENDED_VOCAB_SIZE for i in ids):
            invalid_id_examples += 1

        new_count = sum(i >= BASE_TARGET_VOCAB_SIZE for i in ids)
        new_token_occurrences += new_count
        total_tokens += len(ids)

        if new_count:
            examples_with_new_tokens += 1

    print(f"\n{split_name}: checked {min(n, len(df))} examples")
    print("Total target tokens:", total_tokens)
    print("New vocabulary token occurrences:", new_token_occurrences)
    print("Examples containing new tokens:", examples_with_new_tokens)
    print("EOS errors:", eos_errors)
    print("Invalid-ID examples:", invalid_id_examples)

    assert eos_errors == 0, f"{split_name}: EOS validation failed"
    assert invalid_id_examples == 0, f"{split_name}: invalid token IDs found"

audit_target_examples(train_df, "Training")
audit_target_examples(val_df, "Validation")

print("\nPASS: Sampled targets have valid IDs and one terminal EOS.")

ValueError: SentencePiece token '▁ᱯᱚᱧᱪᱟᱭᱚᱛ' (SP ID 28551) is missing from the target encoder.

In [152]:
# Section 10.18a: Diagnose the missing SentencePiece mapping

piece = "▁ᱯᱚᱧᱪᱟᱭᱚᱛ"
sp_id = tok.tgt_spm.PieceToId(piece)

print("Piece:", repr(piece))
print("SentencePiece ID:", sp_id)
print("SentencePiece reports unknown:",
      tok.tgt_spm.IsUnknown(sp_id))

print("\nExact target-encoder lookup:")
print(tok.tgt_encoder.get(piece, "NOT FOUND"))

print("\nRelated target-encoder entries:")
matches = [
    (key, value)
    for key, value in tok.tgt_encoder.items()
    if "ᱯᱚᱧᱪᱟᱭᱚᱛ" in key
]

for item in matches[:20]:
    print(item)

print("\nTokenizer vocabulary sizes:")
print("Target encoder:", len(tok.tgt_encoder))
print("SentencePiece vocabulary:", tok.tgt_spm.GetPieceSize())
print("Base target vocabulary size:", BASE_TARGET_VOCAB_SIZE)

Piece: '▁ᱯᱚᱧᱪᱟᱭᱚᱛ'
SentencePiece ID: 28551
SentencePiece reports unknown: False

Exact target-encoder lookup:
NOT FOUND

Related target-encoder entries:

Tokenizer vocabulary sizes:
Target encoder: 122672
SentencePiece vocabulary: 128000
Base target vocabulary size: 122672


In [153]:
# Section 10.18b: Measure SentencePiece-to-model vocabulary coverage

sp_size = tok.tgt_spm.GetPieceSize()
mapped = 0
missing = []
unknown_pieces = 0

for sp_id in range(sp_size):
    piece = tok.tgt_spm.IdToPiece(sp_id)

    if tok.tgt_spm.IsUnknown(sp_id):
        unknown_pieces += 1

    if piece in tok.tgt_encoder:
        mapped += 1
    elif len(missing) < 15:
        missing.append((sp_id, piece))

print("SentencePiece vocabulary size:", sp_size)
print("Target model vocabulary size:", len(tok.tgt_encoder))
print("SentencePiece pieces found in target encoder:", mapped)
print("Unmapped SentencePiece pieces:", sp_size - mapped)
print("Unknown SentencePiece IDs:", unknown_pieces)

print("\nFirst unmapped pieces:")
for sp_id, piece in missing:
    print(sp_id, repr(piece))

print("\nTarget encoder ID range:",
      min(tok.tgt_encoder.values()), max(tok.tgt_encoder.values()))

SentencePiece vocabulary size: 128000
Target model vocabulary size: 122672
SentencePiece pieces found in target encoder: 120178
Unmapped SentencePiece pieces: 7822
Unknown SentencePiece IDs: 1

First unmapped pieces:
3 '▁<dnt>'
4 '▁</dnt>'
870 '▁᱑'
1098 '▁᱒'
1330 '▁ꯑꯃꯅꯤ꯫'
1367 '▁꯱'
1696 '▁᱒᱐'
1776 '▁꯲'
1936 '꯰꯰'
2010 '▁᱑᱙'
2211 '▁ᱥᱤᱧᱚᱛ'
2249 '꯭ꯔ'
2534 '▁अहइँ'
2554 '▁बिचणो'
2555 '▁हत्त्यारा'

Target encoder ID range: 0 122671


In [154]:
# Section 10.18c: Inspect the original target-tokenizer mapping logic

import inspect

print("Tokenizer class:", type(tok))
print("\nTokenizer methods related to target encoding:")

for name in dir(tok):
    if any(term in name.lower() for term in ["tokenize", "encode", "target", "tgt"]):
        attr = getattr(tok, name, None)
        if callable(attr):
            try:
                print(f"\n--- {name} ---")
                print(inspect.getsource(attr)[:5000])
            except (OSError, TypeError):
                print(f"Source unavailable for {name}")

print("\n--- Existing encode_original_target_segment ---")
print(inspect.getsource(encode_original_target_segment))

print("\n--- Existing tokenize_target_with_extension_v2 ---")
print(inspect.getsource(tokenize_target_with_extension_v2))

Tokenizer class: <class 'transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.tokenization_indictrans.IndicTransTokenizer'>

Tokenizer methods related to target encoding:

--- _batch_encode_plus ---
    def _batch_encode_plus(
        self,
        batch_text_or_text_pairs: Union[
            List[TextInput],
            List[TextInputPair],
            List[PreTokenizedInput],
            List[PreTokenizedInputPair],
            List[EncodedInput],
            List[EncodedInputPair],
        ],
        add_special_tokens: bool = True,
        padding_strategy: PaddingStrategy = PaddingStrategy.DO_NOT_PAD,
        truncation_strategy: TruncationStrategy = TruncationStrategy.DO_NOT_TRUNCATE,
        max_length: Optional[int] = None,
        stride: int = 0,
        is_split_into_words: bool = False,
        pad_to_multiple_of: Optional[int] = None,
        padding_side: Optional[bool] = None,
        return_tensors: Optional[Union[str, Tenso

In [155]:
# Section 10.18d: Compare original tokenizer mapping with custom mapping

text = train_df["Santali"].iloc[0]

# Original target-tokenizer behavior
with tok.as_target_tokenizer():
    original_ids = tok.encode(
        text,
        add_special_tokens=True,
        truncation=False
    )

# Explicit SentencePiece -> target encoder mapping
pieces = tok.tgt_spm.EncodeAsPieces(text)
converted_ids = tok.convert_tokens_to_ids(pieces)

print("Sample text:", text[:150])
print("SentencePiece pieces:", len(pieces))
print("Original tokenizer IDs:", len(original_ids))
print("Direct converted IDs:", len(converted_ids))

print("\nFirst 20 pieces and converted IDs:")
for piece, token_id in list(zip(pieces, converted_ids))[:20]:
    print(repr(piece), "->", token_id)

print("\nOriginal tokenizer first 20 IDs:", original_ids[:20])
print("Target encoder unknown-token ID:",
      tok.tgt_encoder.get("<unk>", "not present"))
print("Tokenizer UNK token:", tok.unk_token)
print("Tokenizer UNK ID:", tok.unk_token_id)
print("EOS ID from model config:", extended_model.config.eos_token_id)

Sample text: ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾
SentencePiece pieces: 27
Original tokenizer IDs: 28
Direct converted IDs: 27

First 20 pieces and converted IDs:
'▁ᱤᱱ' -> 95065
'ᱥ' -> 72888
'ᱴᱟ' -> 90964
'ᱠᱟᱨ' -> 95272
'ᱴ' -> 78349
'▁ᱼ' -> 97260
'ᱨᱮᱭᱟᱜ' -> 102616
'▁ᱥᱚᱯ' -> 103875
'ᱟᱨ' -> 78171
'▁ᱮᱯ' -> 98743
'▁ᱫᱚ' -> 48404
'▁ᱯᱳ' -> 103603
'ᱸᱰ' -> 104440
'ᱨᱮ' -> 66869
'▁ᱦ' -> 90036
'ᱟᱹᱲ' -> 87006
'ᱭᱟᱹᱲ' -> 111553
'▁ᱤᱱ' -> 95065
'ᱥ' -> 72888
'ᱴᱟ' -> 90964

Original tokenizer first 20 IDs: [95037, 72860, 90936, 95244, 78321, 97232, 102588, 103847, 78143, 98715, 48377, 103575, 104412, 66841, 90008, 86978, 111525, 95037, 72860, 90936]
Target encoder unknown-token ID: 3
Tokenizer UNK token: <unk>
Tokenizer UNK ID: 3
EOS ID from model config: 2


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [156]:
# Section 10.18e: Inspect target vocabulary ID conversion

text = train_df["Santali"].iloc[0]

with tok.as_target_tokenizer():
    pieces = tok.tokenize(text)
    converted_ids = tok.convert_tokens_to_ids(pieces)
    original_ids = tok.encode(
        text,
        add_special_tokens=True,
        truncation=False
    )

print("First 15 target pieces and their IDs:")
for piece, token_id in list(zip(pieces, converted_ids))[:15]:
    print(repr(piece), "->", token_id)

print("\nOriginal tokenizer IDs:", original_ids[:15])
print("Target encoder size:", len(tok.tgt_encoder))
print("Target decoder size:", len(tok.tgt_decoder))
print("Target encoder sample:")
print(list(tok.tgt_encoder.items())[:10])

print("\nTarget encoder mapping for first piece:")
first_piece = pieces[0]
print("Piece:", repr(first_piece))
print("Encoder ID:", tok.tgt_encoder.get(first_piece))
print("Converted ID:", converted_ids[0])
print("Original first ID:", original_ids[0])

First 15 target pieces and their IDs:
'▁ᱤᱱ' -> 95037
'ᱥ' -> 72860
'ᱴᱟ' -> 90936
'ᱠᱟᱨ' -> 95244
'ᱴ' -> 78321
'▁ᱼ' -> 97232
'ᱨᱮᱭᱟᱜ' -> 102588
'▁ᱥᱚᱯ' -> 103847
'ᱟᱨ' -> 78143
'▁ᱮᱯ' -> 98715
'▁ᱫᱚ' -> 48377
'▁ᱯᱳ' -> 103575
'ᱸᱰ' -> 104412
'ᱨᱮ' -> 66841
'▁ᱦ' -> 90008

Original tokenizer IDs: [95037, 72860, 90936, 95244, 78321, 97232, 102588, 103847, 78143, 98715, 48377, 103575, 104412, 66841, 90008]
Target encoder size: 122672
Target decoder size: 122672
Target encoder sample:
[('<s>', 0), ('<pad>', 1), ('</s>', 2), ('<unk>', 3), ('▁.', 4), ('▁,', 5), ('▁।', 6), ('▁"', 7), ('▁है', 8), ('▁के', 9)]

Target encoder mapping for first piece:
Piece: '▁ᱤᱱ'
Encoder ID: 95037
Converted ID: 95037
Original first ID: 95037


In [157]:
# Section 10.18f: Correct original target-segment encoding

def encode_original_target_segment(segment):
    if not segment:
        return []

    # Use the tokenizer's actual target-vocabulary mapping.
    with tok.as_target_tokenizer():
        pieces = tok.tokenize(segment)
        token_ids = tok.convert_tokens_to_ids(pieces)

    return token_ids


def tokenize_target_with_extension_v2(text, add_eos=True):
    token_ids = []
    cursor = 0

    for match in _extension_pattern.finditer(text):
        # Encode the text preceding the extension candidate.
        prefix = text[cursor:match.start()]
        token_ids.extend(encode_original_target_segment(prefix))

        # Insert the dedicated ID for the extended vocabulary word.
        candidate = match.group(1)
        token_ids.append(candidate_to_id[candidate])

        cursor = match.end()

    # Encode the remaining text.
    token_ids.extend(encode_original_target_segment(text[cursor:]))

    # Add EOS exactly once, using the model's configured EOS ID.
    eos_id = extended_model.config.eos_token_id
    if add_eos and (not token_ids or token_ids[-1] != eos_id):
        token_ids.append(eos_id)

    return token_ids


print("Corrected target encoding function defined.")
print("EOS ID:", extended_model.config.eos_token_id)
print("Extended candidates:", len(candidate_to_id))

Corrected target encoding function defined.
EOS ID: 2
Extended candidates: 95


In [158]:
# Section 10.18g: Validate corrected target encoding

sample_df = train_df.head(100)
eos_id = extended_model.config.eos_token_id
extended_vocab_size = extended_model.config.decoder_vocab_size

errors = []
examples_with_extensions = 0
total_extension_ids = 0
total_tokens = 0

for idx, text in enumerate(sample_df["Santali"]):
    ids = tokenize_target_with_extension_v2(text, add_eos=True)

    if not ids:
        errors.append(f"Row {idx}: empty token sequence")
        continue

    if ids[-1] != eos_id:
        errors.append(f"Row {idx}: EOS missing at end")

    if any(token_id < 0 or token_id >= extended_vocab_size for token_id in ids):
        errors.append(f"Row {idx}: token ID outside extended vocabulary")

    inserted = sum(token_id in id_to_candidate for token_id in ids)
    if inserted:
        examples_with_extensions += 1
        total_extension_ids += inserted

    total_tokens += len(ids)

print("Examples checked:", len(sample_df))
print("Errors:", len(errors))
print("Examples containing extension IDs:", examples_with_extensions)
print("Total inserted extension IDs:", total_extension_ids)
print("Total target tokens:", total_tokens)
print("Extended vocabulary size:", extended_vocab_size)
print("First errors:", errors[:10])

assert not errors, "Validation failed; do not start training yet."
print("\nPASS: Target encoding validation succeeded.")

Examples checked: 100
Errors: 0
Examples containing extension IDs: 22
Total inserted extension IDs: 27
Total target tokens: 2861
Extended vocabulary size: 122767
First errors: []

PASS: Target encoding validation succeeded.


In [159]:
# Section 10.18h: Check how the training dataset tokenizes targets

import inspect

print(inspect.getsource(TranslationDataset.__getitem__))

    def __getitem__(self, idx):
        return self.examples[idx]



In [160]:
# Section 10.18i: Inspect dataset construction

print(inspect.getsource(TranslationDataset.__init__))

print("\nFirst stored example:")
print(train_dataset.examples[0])

    def __init__(self, examples):
        self.examples = examples


First stored example:
{'source': "eng_Latn sat_Olck Instacart 's Shopper app looks like a green Instacart bag on a white background .", 'target': 'ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱼᱨᱮᱭᱟᱜ ᱥᱚᱯᱟᱨ ᱮᱯ ᱫᱚ ᱯᱳᱸᱰᱨᱮ ᱦᱟᱹᱲᱭᱟᱹᱲ ᱤᱱᱥᱴᱟᱠᱟᱨᱴ ᱡᱷᱚᱞᱟ ᱞᱮᱠᱟ ᱧᱮᱞᱚᱜᱼᱟ ᱾'}


In [161]:
# Section 10.18j: Locate the data collator and target-tokenization logic

import inspect

for name, obj in globals().copy().items():
    if callable(obj) and (
        "collat" in name.lower()
        or "tokeniz" in name.lower()
        or "train" in name.lower()
    ):
        try:
            source = inspect.getsource(obj)
            if any(term in source for term in [
                "examples",
                '["target"]',
                "tokenizer(",
                "text_target",
                "labels",
            ]):
                print(f"\n--- {name} ---")
                print(source[:5000])
        except (TypeError, OSError):
            pass


--- collate_translation_batch ---
def collate_translation_batch(examples):
    sources = [example["source"] for example in examples]
    targets = [example["target"] for example in examples]

    source_batch = tok(
        sources,
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt",
    )

    target_batch = tok(
        text_target=targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt",
    )

    labels = target_batch["input_ids"].clone()
    labels[target_batch["attention_mask"] == 0] = -100

    return {
        "input_ids": source_batch["input_ids"],
        "attention_mask": source_batch["attention_mask"],
        "labels": labels,
    }


--- tokenize_target_with_extension ---
def tokenize_target_with_extension(text):
    """
    Return target token IDs using:
      - new IDs for recognized candidate words;
      - original target-tokenizer IDs for a

In [162]:
# Section 10.18k: Collator using extended target vocabulary

import torch

def collate_translation_batch(examples):
    sources = [example["source"] for example in examples]
    targets = [example["target"] for example in examples]

    # Source encoding remains unchanged.
    source_batch = tok(
        sources,
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt",
    )

    # Target labels use the extended vocabulary.
    target_sequences = [
        tokenize_target_with_extension_v2(text, add_eos=True)[:MAX_TARGET_LENGTH]
        for text in targets
    ]

    # Ensure EOS remains at the end after truncation.
    eos_id = extended_model.config.eos_token_id
    target_sequences = [
        (ids if ids and ids[-1] == eos_id
         else ids[:-1] + [eos_id] if len(ids) >= MAX_TARGET_LENGTH
         else ids + [eos_id])
        for ids in target_sequences
    ]

    max_len = max(len(ids) for ids in target_sequences)
    labels = torch.full(
        (len(target_sequences), max_len),
        fill_value=-100,
        dtype=torch.long,
    )

    for i, ids in enumerate(target_sequences):
        labels[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)

    # Padding labels remain -100, so they are ignored by the loss.
    return {
        "input_ids": source_batch["input_ids"],
        "attention_mask": source_batch["attention_mask"],
        "labels": labels,
    }

print("Extended-vocabulary collator defined.")

Extended-vocabulary collator defined.


In [163]:
# Section 10.18l: Final collator check

batch = collate_translation_batch(train_dataset.examples[:4])

labels = batch["labels"]
valid_labels = labels[labels != -100]
extension_ids = set(candidate_to_id.values())

print("Source shape:", tuple(batch["input_ids"].shape))
print("Attention mask shape:", tuple(batch["attention_mask"].shape))
print("Target labels shape:", tuple(labels.shape))
print("Valid target tokens:", valid_labels.numel())
print("Extension IDs in batch:", sum(
    int(token_id) in extension_ids
    for token_id in valid_labels.tolist()
))
print("Padding labels (-100):", int((labels == -100).sum()))
print("EOS at end of each target:")
for row in labels:
    row_valid = row[row != -100]
    print(bool(len(row_valid) and row_valid[-1].item() == extended_model.config.eos_token_id))

print("All valid IDs in range:", bool(
    ((valid_labels >= 0) &
     (valid_labels < extended_model.config.decoder_vocab_size)).all()
))

Source shape: (4, 58)
Attention mask shape: (4, 58)
Target labels shape: (4, 52)
Valid target tokens: 133
Extension IDs in batch: 1
Padding labels (-100): 75
EOS at end of each target:
True
True
True
True
All valid IDs in range: True


In [164]:
# Section 10.19: Inspect existing training configuration

print("Optimizer:", type(optimizer).__name__)
print("Optimizer parameter tensors:",
      sum(len(group["params"]) for group in optimizer.param_groups))
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])

print("\nTraining constants:")
for name in [
    "MAX_SOURCE_LENGTH",
    "MAX_TARGET_LENGTH",
    "BATCH_SIZE",
    "GRAD_ACCUMULATION_STEPS",
    "NUM_EPOCHS",
    "LEARNING_RATE",
    "WEIGHT_DECAY",
    "WARMUP_RATIO",
    "PATIENCE",
    "MIN_DELTA",
]:
    if name in globals():
        print(f"{name} = {globals()[name]}")
    else:
        print(f"{name} = not defined")

Optimizer: AdamW
Optimizer parameter tensors: 218
Learning rate: 0.0001
Weight decay: 0.01

Training constants:
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUMULATION_STEPS = not defined
NUM_EPOCHS = 10
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
PATIENCE = not defined
MIN_DELTA = not defined


In [165]:
# Section 10.20: Prepare training configuration and data loaders

from torch.utils.data import DataLoader

GRAD_ACCUMULATION_STEPS = 4
PATIENCE = 2
MIN_DELTA = 0.001

NUM_EPOCHS = 10
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_translation_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_translation_batch,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Gradient accumulation:", GRAD_ACCUMULATION_STEPS)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUMULATION_STEPS)
print("Epoch limit:", NUM_EPOCHS)
print("Early-stopping patience:", PATIENCE)
print("Minimum validation-loss improvement:", MIN_DELTA)
print("Optimizer reused:", optimizer is not None)
print("Model device:", next(extended_model.parameters()).device)

Training batches: 4471
Validation batches: 249
Gradient accumulation: 4
Effective batch size: 16
Epoch limit: 10
Early-stopping patience: 2
Minimum validation-loss improvement: 0.001
Optimizer reused: True
Model device: cuda:0


In [166]:
# Section 10.21: Prepare extended-model checkpoint locations

import os
import json
import torch

EXTENDED_OUTPUT_DIR = "/kaggle/working/santali_vocab_extended_lora"
BEST_CHECKPOINT_DIR = os.path.join(EXTENDED_OUTPUT_DIR, "best_checkpoint")
FINAL_CHECKPOINT_DIR = os.path.join(EXTENDED_OUTPUT_DIR, "final_checkpoint")

os.makedirs(BEST_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_CHECKPOINT_DIR, exist_ok=True)

# Initialize training state
best_val_loss = float("inf")
epochs_without_improvement = 0
training_history = []

# Save experiment configuration for reproducibility
experiment_config = {
    "base_model": "ai4bharat/indictrans2-indic-indic-1B",
    "train_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "extended_tokens": len(candidate_to_id),
    "extended_vocab_size": extended_model.config.decoder_vocab_size,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRAD_ACCUMULATION_STEPS,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "early_stopping_patience": PATIENCE,
    "min_delta": MIN_DELTA,
}

with open(
    os.path.join(EXTENDED_OUTPUT_DIR, "experiment_config.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(experiment_config, f, indent=2)

print("Output directory:", EXTENDED_OUTPUT_DIR)
print("Best checkpoint directory:", BEST_CHECKPOINT_DIR)
print("Final checkpoint directory:", FINAL_CHECKPOINT_DIR)
print("Extended tokens:", len(candidate_to_id))
print("Configuration saved.")

Output directory: /kaggle/working/santali_vocab_extended_lora
Best checkpoint directory: /kaggle/working/santali_vocab_extended_lora/best_checkpoint
Final checkpoint directory: /kaggle/working/santali_vocab_extended_lora/final_checkpoint
Extended tokens: 95
Configuration saved.


## 10.18 Target Tokenization Debugging and Resolution

Before starting training, we identified and resolved an important issue in the custom vocabulary-extension pipeline.

### Issue 1: Incorrect SentencePiece-to-model ID mapping

Our initial implementation used `tok.tgt_spm.EncodeAsPieces()` and manually mapped the resulting pieces through `tok.tgt_encoder`. The direct conversion did not consistently match the IDs generated by the original target tokenizer.

For example, the same Santali text produced different token IDs when converted directly versus when processed through the tokenizer's target mode. Some SentencePiece pieces were also missing from the target encoder mapping.

**Root cause:** SentencePiece tokenization and the model's target-vocabulary ID conversion must follow the tokenizer's actual target-mode behavior.

**Resolution:** We corrected `encode_original_target_segment()` to use `tok.tokenize()` and `tok.convert_tokens_to_ids()` inside `tok.as_target_tokenizer()`. This preserves the original tokenizer's mapping for existing vocabulary items while allowing the extension function to insert dedicated IDs for selected new vocabulary items.

### Issue 2: The training collator ignored the extended vocabulary

The original `collate_translation_batch()` used:

`tok(text_target=targets, ...)`

Consequently, target labels were generated using the original tokenizer, and the newly added vocabulary IDs were not included in the training labels.

**Resolution:** We updated the collator to call `tokenize_target_with_extension_v2()` for each target sentence. The updated collator pads labels with `-100`, which excludes padding positions from the loss calculation, and preserves the EOS token when truncating target sequences.

### Validation results

The corrected target encoder was tested on 100 training examples:

- Examples checked: 100
- Encoding errors: 0
- Examples containing extension IDs: 22
- Total extension-token insertions: 27
- Extended vocabulary size: 122,767
- EOS and token-ID range checks: passed

The updated collator was also tested on a batch of four examples:

- Source tensor shape: `(4, 58)`
- Target label shape: `(4, 52)`
- Extension IDs present in the batch: 1
- Padding labels correctly masked with `-100`
- EOS present at the end of all four target sequences
- All valid target IDs within the extended vocabulary

### Conclusion

The target-tokenization and batch-collation issues have been resolved, and the initial validation checks pass. The training pipeline now supports both the original target vocabulary and the selected extended vocabulary.

The next step is to train the extended-vocabulary model using LoRA together with the trainable new decoder-embedding and LM-head rows. The baseline checkpoint remains separate and unchanged.

## Section 11: Main Training — Vocabulary Extension + LoRA

### Objective

Train the extended IndicTrans2 model for English-to-Santali translation using LoRA and trainable embeddings and LM-head rows for 95 newly selected vocabulary items.

### Training configuration

- Training examples: 17,883
- Validation examples: 993
- Maximum epochs: 10
- Micro-batch size: 4
- Gradient accumulation steps: 4
- Effective batch size: 16
- Learning rate: \(10^{-4}\)
- Weight decay: 0.01
- Warmup ratio: 0.03
- Early-stopping patience: 2 epochs
- Minimum validation-loss improvement: 0.001

### Progress monitoring

During training, display the current epoch, completed batches, percentage complete, running training loss, learning rate, elapsed time, estimated remaining time, and GPU memory usage.

### Checkpoint and recovery strategy

At the end of every completed epoch, save a resumable checkpoint containing the model's trainable parameters, new vocabulary rows, optimizer and scheduler states, epoch history, best validation loss, early-stopping state, and next epoch number.

If the runtime fails, restore the latest completed-epoch checkpoint and continue from the next epoch. Preserve the best-validation checkpoint separately from the latest checkpoint.

The original baseline adapter and evaluation artifacts must remain unchanged.

In [167]:

import os
import json
import time
import math
import random
import shutil
import numpy as np
import torch
from tqdm.auto import tqdm
from torch.optim.lr_scheduler import LambdaLR

# Persistent working-directory locations
EXTENDED_OUTPUT_DIR = "/kaggle/working/santali_vocab_extended_lora"
LATEST_CHECKPOINT_DIR = os.path.join(EXTENDED_OUTPUT_DIR, "latest_checkpoint")
BEST_CHECKPOINT_DIR = os.path.join(EXTENDED_OUTPUT_DIR, "best_checkpoint")
FINAL_CHECKPOINT_DIR = os.path.join(EXTENDED_OUTPUT_DIR, "final_checkpoint")
HISTORY_PATH = os.path.join(EXTENDED_OUTPUT_DIR, "training_history.json")

os.makedirs(EXTENDED_OUTPUT_DIR, exist_ok=True)

# Training settings
GRAD_ACCUMULATION_STEPS = 4
PATIENCE = 2
MIN_DELTA = 0.001
NUM_EPOCHS = 10
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

steps_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUMULATION_STEPS
)
total_training_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

def lr_lambda(current_step):
    if warmup_steps > 0 and current_step < warmup_steps:
        return float(current_step + 1) / float(warmup_steps)
    remaining = max(1, total_training_steps - warmup_steps)
    return max(
        0.0,
        float(total_training_steps - current_step) / remaining
    )

# Reuse the optimizer configured in Section 10.
scheduler = LambdaLR(optimizer, lr_lambda)

# Training state
start_epoch = 1
global_step = 0
best_val_loss = float("inf")
epochs_without_improvement = 0
training_history = []

# Keep the baseline files separate.
assert os.path.isdir("/kaggle/working/santali_baseline_lora"), (
    "Baseline directory was not found; verify its location before training."
)

print("Section 11.1 initialized.")
print("Training batches per epoch:", len(train_loader))
print("Optimizer steps per epoch:", steps_per_epoch)
print("Total planned optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Latest checkpoint:", LATEST_CHECKPOINT_DIR)
print("Best checkpoint:", BEST_CHECKPOINT_DIR)
print("Training history:", HISTORY_PATH)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB allocated /",
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB total"
)


Section 11.1 initialized.
Training batches per epoch: 4471
Optimizer steps per epoch: 1118
Total planned optimizer steps: 11180
Warmup steps: 335
Latest checkpoint: /kaggle/working/santali_vocab_extended_lora/latest_checkpoint
Best checkpoint: /kaggle/working/santali_vocab_extended_lora/best_checkpoint
Training history: /kaggle/working/santali_vocab_extended_lora/training_history.json
GPU: Tesla T4
GPU memory: 8.26 GiB allocated / 14.56 GiB total


In [180]:

import os
import json
import math
import time
import random
import shutil
import torch
import numpy as np

from tqdm.auto import tqdm
from torch.nn.utils import clip_grad_norm_
from torch.optim.lr_scheduler import LambdaLR
from peft import set_peft_model_state_dict

# ============================================================
# 1. Configuration and safety checks
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_EPOCHS = 10
PATIENCE = 2
MIN_DELTA = 0.001
GRAD_ACCUMULATION_STEPS = 4
MAX_GRAD_NORM = 1.0

os.makedirs(EXTENDED_OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LATEST_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_CHECKPOINT_DIR, exist_ok=True)

base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

assert decoder_embedding.new_weight.dtype == torch.float32
assert lm_head.new_weight.dtype == torch.float32
assert torch.isfinite(decoder_embedding.new_weight).all().item()
assert torch.isfinite(lm_head.new_weight).all().item()

# This run deliberately starts fresh. Do not load a checkpoint
# from the previous, non-scaled training loop.
extended_model.zero_grad(set_to_none=True)

# Rebuild optimizer using the current trainable Parameter objects.
lora_params = [
    p for name, p in extended_model.named_parameters()
    if "lora_" in name and p.requires_grad
]
extension_params = [
    decoder_embedding.new_weight,
    lm_head.new_weight,
]
trainable_params = lora_params + extension_params

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Fresh scheduler for a fresh run.
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUMULATION_STEPS)
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return max(current_step, 1) / max(warmup_steps, 1)
    progress = (current_step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(0.0, 1.0 - progress)

scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

# Fresh run state.
start_epoch = 1
global_step = 0
best_val_loss = float("inf")
epochs_without_improvement = 0
training_history = []

print("Starting a fresh, corrected training run.")
print("Device:", device)
print("Epochs:", NUM_EPOCHS)
print("Training batches per epoch:", len(train_loader))
print("Optimizer steps per epoch:", steps_per_epoch)
print("Planned optimizer steps:", total_steps)
print("Warmup steps:", warmup_steps)
print("GradScaler enabled:", scaler.is_enabled())
print("LoRA tensors:", len(lora_params))
print("Trainable extension parameters:", sum(p.numel() for p in extension_params))
print("Latest checkpoint:", LATEST_CHECKPOINT_DIR)
print("Best checkpoint:", BEST_CHECKPOINT_DIR)

# ============================================================
# 2. Checkpoint saving
# ============================================================

def save_checkpoint(path, epoch, best_loss, patience_count, history):
    """Save the adapter, custom vocabulary rows, and training state."""
    tmp_path = path + "_tmp"
    if os.path.exists(tmp_path):
        shutil.rmtree(tmp_path)
    os.makedirs(tmp_path, exist_ok=True)

    adapter_path = os.path.join(tmp_path, "adapter")
    extended_model.save_pretrained(adapter_path)

    state = {
        "completed_epoch": epoch,
        "global_step": global_step,
        "best_val_loss": best_loss,
        "epochs_without_improvement": patience_count,
        "training_history": history,
        "decoder_new_weight": decoder_embedding.new_weight.detach().cpu().clone(),
        "lm_head_new_weight": lm_head.new_weight.detach().cpu().clone(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        ),
        "num_epochs": NUM_EPOCHS,
        "patience": PATIENCE,
        "min_delta": MIN_DELTA,
    }
    torch.save(state, os.path.join(tmp_path, "training_state.pt"))

    if os.path.exists(path):
        shutil.rmtree(path)
    os.replace(tmp_path, path)

# ============================================================
# 3. Validation
# ============================================================

@torch.no_grad()
def evaluate_validation():
    extended_model.eval()
    total_loss = 0.0
    total_batches = 0

    for batch in tqdm(val_loader, desc="VALIDATION", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            outputs = extended_model(
                **batch,
                use_cache=False,
                return_dict=True,
            )
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite validation loss detected.")

        total_loss += loss.detach().float().item()
        total_batches += 1

    extended_model.train()
    return total_loss / max(total_batches, 1)

# ============================================================
# 4. Main training loop
# ============================================================

run_start = time.time()
stop_early = False

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    extended_model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    processed_batches = 0
    epoch_start = time.time()

    progress = tqdm(
        enumerate(train_loader, start=1),
        total=len(train_loader),
        desc=f"Epoch {epoch}/{NUM_EPOCHS} [TRAIN]",
    )

    for batch_idx, batch in progress:
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            outputs = extended_model(
                **batch,
                use_cache=False,
                return_dict=True,
            )
            loss = outputs.loss

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}, batch {batch_idx}. "
                "Stop and diagnose; do not continue with corrupted gradients."
            )

        # Correct accumulation scaling, including the final partial group.
        group_start = ((batch_idx - 1) // GRAD_ACCUMULATION_STEPS) * GRAD_ACCUMULATION_STEPS
        group_size = min(
            GRAD_ACCUMULATION_STEPS,
            len(train_loader) - group_start,
        )
        scaled_loss = loss / group_size
        scaler.scale(scaled_loss).backward()

        running_loss += loss.detach().float().item()
        processed_batches += 1

        is_group_end = (
            batch_idx % GRAD_ACCUMULATION_STEPS == 0
            or batch_idx == len(train_loader)
        )

        if is_group_end:
            scaler.unscale_(optimizer)

            grad_norm = clip_grad_norm_(
                trainable_params,
                max_norm=MAX_GRAD_NORM,
                error_if_nonfinite=True,
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            scheduler.step()
            global_step += 1

        avg_loss = running_loss / processed_batches
        percent = 100.0 * batch_idx / len(train_loader)
        elapsed = (time.time() - epoch_start) / 60
        eta = elapsed / max(percent / 100.0, 1e-8) - elapsed

        gpu_mem = (
            torch.cuda.memory_allocated() / (1024 ** 3)
            if torch.cuda.is_available() else 0.0
        )

        progress.set_postfix({
            "loss": f"{loss.detach().float().item():.4f}",
            "avg": f"{avg_loss:.4f}",
            "done": f"{percent:.1f}%",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
            "elapsed": f"{elapsed:.1f}m",
            "ETA": f"{max(eta, 0):.1f}m",
            "GPU": f"{gpu_mem:.2f}G",
        })

        del outputs, loss, scaled_loss

    # Validation after each completed epoch.
    train_loss = running_loss / max(processed_batches, 1)
    val_loss = evaluate_validation()
    epoch_minutes = (time.time() - epoch_start) / 60

    improved = val_loss < best_val_loss - MIN_DELTA

    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val_loss,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "global_step": global_step,
        "epoch_minutes": epoch_minutes,
        "improved": improved,
    }
    training_history.append(epoch_record)

    # Save the latest completed epoch first.
    save_checkpoint(
        LATEST_CHECKPOINT_DIR,
        epoch,
        best_val_loss,
        epochs_without_improvement,
        training_history,
    )

    # Save best checkpoint when validation improves sufficiently.
    if improved:
        save_checkpoint(
            BEST_CHECKPOINT_DIR,
            epoch,
            best_val_loss,
            epochs_without_improvement,
            training_history,
        )

    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(training_history, f, indent=2)

    print(
        f"\nEpoch {epoch}/{NUM_EPOCHS} complete | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"best_val_loss={best_val_loss:.4f} | "
        f"epoch_time={epoch_minutes:.1f} min"
    )
    print(
        f"Completed: {100.0 * epoch / NUM_EPOCHS:.1f}% of planned epochs | "
        f"patience={epochs_without_improvement}/{PATIENCE}"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        stop_early = True

    if stop_early:
        break

# Save the final completed state.
if training_history:
    final_epoch = training_history[-1]["epoch"]
    save_checkpoint(
        FINAL_CHECKPOINT_DIR,
        final_epoch,
        best_val_loss,
        epochs_without_improvement,
        training_history,
    )

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(training_history, f, indent=2)

print("\nTraining finished.")
print("Epochs completed:", len(training_history))
print("Best validation loss:", best_val_loss)
print("Total elapsed minutes:", round((time.time() - run_start) / 60, 2))
print("Latest checkpoint:", LATEST_CHECKPOINT_DIR)
print("Best checkpoint:", BEST_CHECKPOINT_DIR)
print("Final checkpoint:", FINAL_CHECKPOINT_DIR)
print("History:", HISTORY_PATH)


Starting a fresh, corrected training run.
Device: cuda
Epochs: 10
Training batches per epoch: 4471
Optimizer steps per epoch: 1118
Planned optimizer steps: 11180
Warmup steps: 335
GradScaler enabled: True
LoRA tensors: 216
Trainable extension parameters: 194560
Latest checkpoint: /kaggle/working/santali_vocab_extended_lora/latest_checkpoint
Best checkpoint: /kaggle/working/santali_vocab_extended_lora/best_checkpoint


Epoch 1/10 [TRAIN]:   0%|          | 0/4471 [00:00<?, ?it/s]

VALIDATION:   0%|          | 0/249 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(



Epoch 1/10 complete | train_loss=2.9727 | val_loss=2.6463 | best_val_loss=2.6463 | epoch_time=14.7 min
Completed: 10.0% of planned epochs | patience=0/2


...

Epoch 3/10 [TRAIN]:   0%|          | 0/4471 [00:00<?, ?it/s]

VALIDATION:   0%|          | 0/249 [00:00<?, ?it/s]


Epoch 3/10 complete | train_loss=2.7769 | val_loss=2.5692 | best_val_loss=2.5692 | epoch_time=14.8 min
Completed: 30.0% of planned epochs | patience=0/2


Epoch 4/10 [TRAIN]:   0%|          | 0/4471 [00:00<?, ?it/s]

RuntimeError: The total norm of order 2.0 for gradients from `parameters` is non-finite, so it cannot be clipped. To disable this error and scale the gradients by the non-finite norm anyway, set `error_if_nonfinite=False`

# **Resuming  training from 3rd epoch**

In [181]:
import os
import torch

print("=== Checkpoint inspection ===")

for label, path in [
    ("Latest", LATEST_CHECKPOINT_DIR),
    ("Best", BEST_CHECKPOINT_DIR),
    ("Final", FINAL_CHECKPOINT_DIR),
]:
    print(f"\n{label}: {path}")
    if os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            for name in files:
                print(" ", os.path.relpath(os.path.join(root, name), path))
    else:
        print("  Directory does not exist")

print("\n=== Current model parameter health ===")

bad_params = []
for name, param in extended_model.named_parameters():
    if not torch.isfinite(param).all().item():
        bad_params.append(name)

print("Non-finite parameters:", len(bad_params))
for name in bad_params[:20]:
    print(" -", name)

print("\n=== Optimizer state health ===")

bad_states = 0
for state in optimizer.state.values():
    for value in state.values():
        if torch.is_tensor(value) and value.is_floating_point():
            if not torch.isfinite(value).all().item():
                bad_states += 1

print("Non-finite optimizer-state tensors:", bad_states)
print("Global step:", global_step)
print("Completed history epochs:", [
    item["epoch"] for item in training_history
])
print("Best validation loss:", best_val_loss)

=== Checkpoint inspection ===

Latest: /kaggle/working/santali_vocab_extended_lora/latest_checkpoint
  training_state.pt
  adapter/adapter_config.json
  adapter/adapter_model.safetensors
  adapter/README.md

Best: /kaggle/working/santali_vocab_extended_lora/best_checkpoint
  training_state.pt
  adapter/adapter_config.json
  adapter/adapter_model.safetensors
  adapter/README.md

Final: /kaggle/working/santali_vocab_extended_lora/final_checkpoint

=== Current model parameter health ===
Non-finite parameters: 0

=== Optimizer state health ===
Non-finite optimizer-state tensors: 0
Global step: 4000
Completed history epochs: [1, 2, 3]
Best validation loss: 2.5691809874461837


In [182]:
import os
import torch

checkpoint_state_path = os.path.join(
    LATEST_CHECKPOINT_DIR,
    "training_state.pt"
)

# Load only the checkpoint state for inspection.
try:
    checkpoint_state = torch.load(
        checkpoint_state_path,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    checkpoint_state = torch.load(
        checkpoint_state_path,
        map_location="cpu",
    )

print("=== Saved checkpoint state ===")
for key in [
    "completed_epoch",
    "global_step",
    "best_val_loss",
    "epochs_without_improvement",
    "num_epochs",
    "patience",
    "min_delta",
]:
    print(f"{key}: {checkpoint_state.get(key, 'MISSING')}")

print("\n=== Saved custom vocabulary rows ===")
for key in ["decoder_new_weight", "lm_head_new_weight"]:
    value = checkpoint_state.get(key)
    if value is None:
        print(f"{key}: MISSING")
    else:
        print(
            f"{key}: shape={tuple(value.shape)}, "
            f"dtype={value.dtype}, "
            f"finite={torch.isfinite(value).all().item()}"
        )

print("\n=== Saved training states ===")
for key in [
    "optimizer_state_dict",
    "scheduler_state_dict",
    "scaler_state_dict",
    "training_history",
    "torch_rng_state",
]:
    print(f"{key}: {'present' if key in checkpoint_state else 'MISSING'}")

print("\nSaved history epochs:", [
    row.get("epoch")
    for row in checkpoint_state.get("training_history", [])
])

=== Saved checkpoint state ===
completed_epoch: 3
global_step: 3354
best_val_loss: 2.5691809874461837
epochs_without_improvement: 0
num_epochs: 10
patience: 2
min_delta: 0.001

=== Saved custom vocabulary rows ===
decoder_new_weight: shape=(95, 1024), dtype=torch.float32, finite=True
lm_head_new_weight: shape=(95, 1024), dtype=torch.float32, finite=True

=== Saved training states ===
optimizer_state_dict: present
scheduler_state_dict: present
scaler_state_dict: present
training_history: present
torch_rng_state: present

Saved history epochs: [1, 2, 3]


In [184]:
import os
import torch
import random
import numpy as np
from peft import set_peft_model_state_dict

# ------------------------------------------------------------
# 1. Load the latest completed-epoch checkpoint
# ------------------------------------------------------------
checkpoint_state_path = os.path.join(
    LATEST_CHECKPOINT_DIR, "training_state.pt"
)
adapter_path = os.path.join(
    LATEST_CHECKPOINT_DIR, "adapter"
)

state = torch.load(
    checkpoint_state_path,
    map_location="cpu",
    weights_only=False,
)

assert state["completed_epoch"] == 3
assert torch.isfinite(state["decoder_new_weight"]).all()
assert torch.isfinite(state["lm_head_new_weight"]).all()

# ------------------------------------------------------------
# 2. Restore the custom FP32 vocabulary rows
# ------------------------------------------------------------
base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

with torch.no_grad():
    decoder_embedding.new_weight.copy_(
        state["decoder_new_weight"].to(
            device=decoder_embedding.new_weight.device,
            dtype=torch.float32,
        )
    )
    lm_head.new_weight.copy_(
        state["lm_head_new_weight"].to(
            device=lm_head.new_weight.device,
            dtype=torch.float32,
        )
    )

# ------------------------------------------------------------
# 3. Restore LoRA adapter weights
# ------------------------------------------------------------
adapter_state_path = os.path.join(
    adapter_path, "adapter_model.safetensors"
)

from safetensors.torch import load_file

adapter_state_dict = load_file(
    adapter_state_path, device="cpu"
)

result = set_peft_model_state_dict(
    extended_model,
    adapter_state_dict,
)

print("Adapter state loaded.")
print("Adapter load result:", result)

# ------------------------------------------------------------
# 4. Restore optimizer, scheduler, and scaler states
# ------------------------------------------------------------
optimizer.load_state_dict(state["optimizer_state_dict"])
scheduler.load_state_dict(state["scheduler_state_dict"])
scaler.load_state_dict(state["scaler_state_dict"])

# ------------------------------------------------------------
# 5. Restore progress and early-stopping history
# ------------------------------------------------------------
start_epoch = state["completed_epoch"] + 1
global_step = state["global_step"]
best_val_loss = state["best_val_loss"]
epochs_without_improvement = state["epochs_without_improvement"]
training_history = state["training_history"]

random.setstate(state["python_rng_state"])
np.random.set_state(state["numpy_rng_state"])
torch.set_rng_state(state["torch_rng_state"])

if torch.cuda.is_available() and state.get("cuda_rng_state_all") is not None:
    torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])

extended_model.zero_grad(set_to_none=True)
extended_model.train()

# ------------------------------------------------------------
# 6. Verify restoration
# ------------------------------------------------------------
print("\n=== Restored state ===")
print("Resume epoch:", start_epoch)
print("Global step:", global_step)
print("Best validation loss:", best_val_loss)
print("Patience counter:", epochs_without_improvement)
print("History epochs:", [x["epoch"] for x in training_history])

print(
    "Decoder new rows finite:",
    torch.isfinite(decoder_embedding.new_weight).all().item(),
)
print(
    "LM-head new rows finite:",
    torch.isfinite(lm_head.new_weight).all().item(),
)
print(
    "All model parameters finite:",
    all(
        torch.isfinite(p).all().item()
        for p in extended_model.parameters()
    ),
)
print("Restoration complete. Training has NOT restarted.")

Adapter state loaded.
Adapter load result: _IncompatibleKeys(missing_keys=['base_model.model.model.encoder.layers.0.self_attn.k_proj.weight', 'base_model.model.model.encoder.layers.0.self_attn.k_proj.bias', 'base_model.model.model.encoder.layers.0.self_attn.v_proj.base_layer.weight', 'base_model.model.model.encoder.layers.0.self_attn.v_proj.base_layer.bias', 'base_model.model.model.encoder.layers.0.self_attn.q_proj.base_layer.weight', 'base_model.model.model.encoder.layers.0.self_attn.q_proj.base_layer.bias', 'base_model.model.model.encoder.layers.0.self_attn.out_proj.weight', 'base_model.model.model.encoder.layers.0.self_attn.out_proj.bias', 'base_model.model.model.encoder.layers.0.self_attn_layer_norm.weight', 'base_model.model.model.encoder.layers.0.self_attn_layer_norm.bias', 'base_model.model.model.encoder.layers.0.fc1.weight', 'base_model.model.model.encoder.layers.0.fc1.bias', 'base_model.model.model.encoder.layers.0.fc2.weight', 'base_model.model.model.encoder.layers.0.fc2.bias

In [185]:
from safetensors.torch import load_file
import torch

adapter_state_path = os.path.join(
    LATEST_CHECKPOINT_DIR,
    "adapter",
    "adapter_model.safetensors",
)

saved_adapter = load_file(adapter_state_path, device="cpu")

current_adapter_params = {
    name: param.detach().cpu()
    for name, param in extended_model.named_parameters()
    if "lora_" in name
}

print("Saved adapter tensors:", len(saved_adapter))
print("Current LoRA parameter tensors:", len(current_adapter_params))

# Compare the saved adapter tensors with the current model.
matched = 0
mismatched = []

for name, saved_tensor in saved_adapter.items():
    candidates = [
        (current_name, current_tensor)
        for current_name, current_tensor in current_adapter_params.items()
        if current_name.endswith(name)
        or name.endswith(current_name)
    ]

    if not candidates:
        continue

    current_name, current_tensor = candidates[0]

    if saved_tensor.shape == current_tensor.shape:
        matched += 1
        if not torch.allclose(
            saved_tensor.float(),
            current_tensor.float(),
            rtol=1e-5,
            atol=1e-6,
        ):
            mismatched.append(name)

print("Matched adapter tensors:", matched)
print("Matched tensors with different values:", len(mismatched))
print("Examples of differences:", mismatched[:5])

Saved adapter tensors: 219
Current LoRA parameter tensors: 216
Matched adapter tensors: 0
Matched tensors with different values: 0
Examples of differences: []


In [186]:
from safetensors.torch import load_file

adapter_state_path = os.path.join(
    LATEST_CHECKPOINT_DIR,
    "adapter",
    "adapter_model.safetensors",
)

saved_adapter = load_file(adapter_state_path, device="cpu")

current_lora = {
    name: param
    for name, param in extended_model.named_parameters()
    if "lora_" in name
}

print("=== First 12 saved adapter keys ===")
for name in list(saved_adapter.keys())[:12]:
    print(name, tuple(saved_adapter[name].shape))

print("\n=== First 12 current LoRA parameter names ===")
for name, param in list(current_lora.items())[:12]:
    print(name, tuple(param.shape))

print("\n=== Saved keys not directly found in current names ===")
current_names = list(current_lora.keys())

for name in list(saved_adapter.keys())[:10]:
    print(
        name,
        "->",
        [
            current_name
            for current_name in current_names
            if name.split(".")[-2:] == current_name.split(".")[-2:]
        ][:3],
    )

=== First 12 saved adapter keys ===
base_model.model.lm_head.new_weight (95, 1024)
base_model.model.model.decoder.layers.0.encoder_attn.q_proj.lora_A.weight (16, 1024)
base_model.model.model.decoder.layers.0.encoder_attn.q_proj.lora_B.weight (1024, 16)
base_model.model.model.decoder.layers.0.encoder_attn.v_proj.lora_A.weight (16, 1024)
base_model.model.model.decoder.layers.0.encoder_attn.v_proj.lora_B.weight (1024, 16)
base_model.model.model.decoder.layers.0.self_attn.q_proj.lora_A.weight (16, 1024)
base_model.model.model.decoder.layers.0.self_attn.q_proj.lora_B.weight (1024, 16)
base_model.model.model.decoder.layers.0.self_attn.v_proj.lora_A.weight (16, 1024)
base_model.model.model.decoder.layers.0.self_attn.v_proj.lora_B.weight (1024, 16)
base_model.model.model.decoder.layers.1.encoder_attn.q_proj.lora_A.weight (16, 1024)
base_model.model.model.decoder.layers.1.encoder_attn.q_proj.lora_B.weight (1024, 16)
base_model.model.model.decoder.layers.1.encoder_attn.v_proj.lora_A.weight (16, 

In [187]:
from safetensors.torch import load_file
import torch
import os

adapter_state_path = os.path.join(
    LATEST_CHECKPOINT_DIR,
    "adapter",
    "adapter_model.safetensors",
)

saved_adapter = load_file(adapter_state_path, device="cpu")

current_lora = {
    name.replace(".lora_A.default.weight", ".lora_A.weight")
        .replace(".lora_B.default.weight", ".lora_B.weight"): param.detach().cpu()
    for name, param in extended_model.named_parameters()
    if "lora_" in name
}

matched = 0
mismatched = []
unmatched = []

for name, saved_tensor in saved_adapter.items():
    if ".lora_" not in name:
        continue

    current_tensor = current_lora.get(name)

    if current_tensor is None:
        unmatched.append(name)
        continue

    if saved_tensor.shape != current_tensor.shape:
        mismatched.append((name, "shape mismatch"))
        continue

    matched += 1

    if not torch.allclose(
        saved_tensor.float(),
        current_tensor.float(),
        rtol=1e-5,
        atol=1e-6,
    ):
        mismatched.append(name)

print("Saved LoRA tensors:", sum(".lora_" in k for k in saved_adapter))
print("Matched LoRA tensors:", matched)
print("Unmatched LoRA tensors:", len(unmatched))
print("Matched tensors with different values:", len(mismatched))
print("Unmatched examples:", unmatched[:5])
print("Different-value examples:", mismatched[:5])

Saved LoRA tensors: 216
Matched LoRA tensors: 216
Unmatched LoRA tensors: 0
Matched tensors with different values: 0
Unmatched examples: []
Different-value examples: []


In [188]:
import os
import torch

checkpoint_path = os.path.join(
    LATEST_CHECKPOINT_DIR,
    "training_state.pt",
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

print("=== Checkpoint state ===")
print("Completed epoch:", checkpoint["completed_epoch"])
print("Global step:", checkpoint["global_step"])
print("Best validation loss:", checkpoint["best_val_loss"])
print("Epochs without improvement:", checkpoint["epochs_without_improvement"])
print("History epochs:", [
    item.get("epoch")
    for item in checkpoint["training_history"]
])

print("\n=== In-memory training state ===")
print("Resume epoch:", start_epoch if "start_epoch" in globals() else "not set")
print("Optimizer:", type(optimizer).__name__ if "optimizer" in globals() else "not set")
print("Scheduler:", type(scheduler).__name__ if "scheduler" in globals() else "not set")
print("GradScaler:", type(scaler).__name__ if "scaler" in globals() else "not set")

print("\n=== Saved optimizer/scheduler/scaler states ===")
for key in ["optimizer_state_dict", "scheduler_state_dict", "scaler_state_dict"]:
    print(f"{key}:", key in checkpoint, 
          "non-empty:", bool(checkpoint.get(key)) if key in checkpoint else False)

=== Checkpoint state ===
Completed epoch: 3
Global step: 3354
Best validation loss: 2.5691809874461837
Epochs without improvement: 0
History epochs: [1, 2, 3]

=== In-memory training state ===
Resume epoch: 4
Optimizer: AdamW
Scheduler: LambdaLR
GradScaler: GradScaler

=== Saved optimizer/scheduler/scaler states ===
optimizer_state_dict: True non-empty: True
scheduler_state_dict: True non-empty: True
scaler_state_dict: True non-empty: True


In [189]:
import ast

# Inspect the notebook's current training-loop source, if it was saved
# in a known variable. Otherwise, display the relevant runtime settings.
print("=== Training configuration ===")
for name in [
    "NUM_EPOCHS",
    "GRAD_ACCUMULATION_STEPS",
    "PATIENCE",
    "MIN_DELTA",
    "EXTENDED_OUTPUT_DIR",
    "BEST_CHECKPOINT_DIR",
    "LATEST_CHECKPOINT_DIR",
    "FINAL_CHECKPOINT_DIR",
]:
    print(f"{name} =", globals().get(name, "<not defined>"))

print("\n=== Training-related functions ===")
for name, obj in list(globals().items()):
    if callable(obj) and any(
        word in name.lower()
        for word in ["train", "validate", "checkpoint", "evaluate"]
    ):
        print(name, getattr(obj, "__name__", type(obj).__name__))

=== Training configuration ===
NUM_EPOCHS = 10
GRAD_ACCUMULATION_STEPS = 4
PATIENCE = 2
MIN_DELTA = 0.001
EXTENDED_OUTPUT_DIR = /kaggle/working/santali_vocab_extended_lora
BEST_CHECKPOINT_DIR = /kaggle/working/santali_vocab_extended_lora/best_checkpoint
LATEST_CHECKPOINT_DIR = /kaggle/working/santali_vocab_extended_lora/latest_checkpoint
FINAL_CHECKPOINT_DIR = /kaggle/working/santali_vocab_extended_lora/final_checkpoint

=== Training-related functions ===
save_training_checkpoint save_training_checkpoint
load_training_checkpoint load_training_checkpoint
save_checkpoint save_checkpoint
evaluate_validation evaluate_validation


In [190]:
import inspect

for name in [
    "evaluate_validation",
    "save_training_checkpoint",
    "load_training_checkpoint",
    "save_checkpoint",
]:
    func = globals().get(name)

    if callable(func):
        print(f"\n{'=' * 25} {name} {'=' * 25}")
        print(inspect.signature(func))
        print(inspect.getsource(func))


========================= evaluate_validation =========================
()
@torch.no_grad()
def evaluate_validation():
    extended_model.eval()
    total_loss = 0.0
    total_batches = 0

    for batch in tqdm(val_loader, desc="VALIDATION", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            outputs = extended_model(
                **batch,
                use_cache=False,
                return_dict=True,
            )
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite validation loss detected.")

        total_loss += loss.detach().float().item()
        total_batches += 1

    extended_model.train()
    return total_loss / max(total_batches, 1)


========================= save_training_checkpoint =========================
(path, next_epo

In [191]:
import os
import torch

checkpoint_path = os.path.join(
    LATEST_CHECKPOINT_DIR,
    "training_state.pt",
)

state = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

required_keys = [
    "completed_epoch",
    "global_step",
    "best_val_loss",
    "epochs_without_improvement",
    "training_history",
    "decoder_new_weight",
    "lm_head_new_weight",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "scaler_state_dict",
    "python_rng_state",
    "numpy_rng_state",
    "torch_rng_state",
    "cuda_rng_state_all",
]

missing = [key for key in required_keys if key not in state]

print("Missing checkpoint keys:", missing)
print("Completed epoch:", state.get("completed_epoch"))
print("Next epoch:", state.get("completed_epoch", 0) + 1)
print("Global step:", state.get("global_step"))
print("Optimizer state entries:", len(state["optimizer_state_dict"]["state"]))
print("Optimizer parameter groups:", len(state["optimizer_state_dict"]["param_groups"]))
print("Scheduler last epoch:", state["scheduler_state_dict"].get("last_epoch"))
print("GradScaler state:", state["scaler_state_dict"])
print("Decoder extension shape:", tuple(state["decoder_new_weight"].shape))
print("LM-head extension shape:", tuple(state["lm_head_new_weight"].shape))
print("All required keys present:", not missing)

Missing checkpoint keys: []
Completed epoch: 3
Next epoch: 4
Global step: 3354
Optimizer state entries: 218
Optimizer parameter groups: 1
Scheduler last epoch: 3354
GradScaler state: {'scale': 131072.0, 'growth_factor': 2.0, 'backoff_factor': 0.5, 'growth_interval': 2000, '_growth_tracker': 1354}
Decoder extension shape: (95, 1024)
LM-head extension shape: (95, 1024)
All required keys present: True


In [192]:
import random
import numpy as np
import torch

state = torch.load(
    os.path.join(LATEST_CHECKPOINT_DIR, "training_state.pt"),
    map_location="cpu",
    weights_only=False,
)

# Restore the custom vocabulary rows.
base = extended_model.base_model.model
decoder_embedding = base.model.decoder.embed_tokens
lm_head = base.lm_head

with torch.no_grad():
    decoder_embedding.new_weight.copy_(
        state["decoder_new_weight"].to(
            device=decoder_embedding.new_weight.device,
            dtype=decoder_embedding.new_weight.dtype,
        )
    )
    lm_head.new_weight.copy_(
        state["lm_head_new_weight"].to(
            device=lm_head.new_weight.device,
            dtype=lm_head.new_weight.dtype,
        )
    )

# Restore training state.
optimizer.load_state_dict(state["optimizer_state_dict"])
scheduler.load_state_dict(state["scheduler_state_dict"])
scaler.load_state_dict(state["scaler_state_dict"])

# Restore random-number-generator states.
random.setstate(state["python_rng_state"])
np.random.set_state(state["numpy_rng_state"])
torch.set_rng_state(state["torch_rng_state"])

if torch.cuda.is_available() and state["cuda_rng_state_all"] is not None:
    torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])

# Restore experiment progress.
start_epoch = state["completed_epoch"] + 1
global_step = state["global_step"]
best_val_loss = state["best_val_loss"]
epochs_without_improvement = state["epochs_without_improvement"]
training_history = state["training_history"]

# Verify restored extension matrices.
print("Resume epoch:", start_epoch)
print("Global step:", global_step)
print("Best validation loss:", best_val_loss)
print("Epochs without improvement:", epochs_without_improvement)
print("History epochs:", [item.get("epoch") for item in training_history])
print("Decoder extension finite:", torch.isfinite(decoder_embedding.new_weight).all().item())
print("LM-head extension finite:", torch.isfinite(lm_head.new_weight).all().item())
print("Optimizer state entries:", len(optimizer.state))
print("Scheduler last epoch:", scheduler.last_epoch)
print("GradScaler scale:", scaler.get_scale())
print("Training has NOT started.")

Resume epoch: 4
Global step: 3354
Best validation loss: 2.5691809874461837
Epochs without improvement: 0
History epochs: [1, 2, 3]
Decoder extension finite: True
LM-head extension finite: True
Optimizer state entries: 218
Scheduler last epoch: 3354
GradScaler scale: 131072.0
Training has NOT started.


In [194]:
import torch

# Rebuild the expected trainable-parameter list without changing the optimizer.
current_lora_params = [
    p for name, p in extended_model.named_parameters()
    if "lora_" in name and p.requires_grad
]

base = extended_model.base_model.model
current_extension_params = [
    base.model.decoder.embed_tokens.new_weight,
    base.lm_head.new_weight,
]

expected_params = current_lora_params + current_extension_params
optimizer_params = [
    p
    for group in optimizer.param_groups
    for p in group["params"]
]

expected_ids = {id(p) for p in expected_params}
optimizer_ids = {id(p) for p in optimizer_params}

print("Expected trainable tensors:", len(expected_params))
print("Optimizer parameter tensors:", len(optimizer_params))
print("Optimizer matches current parameters:", expected_ids == optimizer_ids)
print("Trainable parameter count:", sum(p.numel() for p in expected_params))
print("Optimizer state entries:", len(optimizer.state))

bad_state = []
for param, param_state in optimizer.state.items():
    for key, value in param_state.items():
        if torch.is_tensor(value) and value.is_floating_point():
            if not torch.isfinite(value).all().item():
                bad_state.append(key)

print("Non-finite optimizer-state tensors:", len(bad_state))
print("Examples:", bad_state[:5])
print("Global step:", global_step)
print("Next epoch:", start_epoch)

Expected trainable tensors: 218
Optimizer parameter tensors: 218
Optimizer matches current parameters: True
Trainable parameter count: 3733504
Optimizer state entries: 218
Non-finite optimizer-state tensors: 0
Examples: []
Global step: 3354
Next epoch: 4


In [197]:
# Reset GradScaler's per-optimizer state after the interrupted loop.
# Preserve its learned scale and growth tracker.
scaler._per_optimizer_states.clear()

print("GradScaler state reset.")
print("Current scale:", scaler.get_scale())
print("Global step:", global_step)
print("Next epoch:", start_epoch)

GradScaler state reset.
Current scale: 131072.0
Global step: 3354
Next epoch: 4


In [198]:
print("Current epoch:", start_epoch)
print("Global step:", global_step)
print("Gradient accumulation steps:", GRAD_ACCUMULATION_STEPS)
print("GradScaler scale:", scaler.get_scale())

print("Optimizer gradients currently present:",
      sum(p.grad is not None for p in trainable_params))

print("Trainable parameters finite:",
      all(torch.isfinite(p).all().item() for p in trainable_params))

print("Optimizer state entries:", len(optimizer.state))
print("Scheduler step:", scheduler.last_epoch)

Current epoch: 4
Global step: 3354
Gradient accumulation steps: 4
GradScaler scale: 131072.0
Optimizer gradients currently present: 218
Trainable parameters finite: True
Optimizer state entries: 218
Scheduler step: 3354


In [199]:
import os
import torch
import random
import numpy as np

checkpoint_path = os.path.join(
    LATEST_CHECKPOINT_DIR, "training_state.pt"
)

state = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Restore the vocabulary-extension weights
base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

with torch.no_grad():
    decoder_embedding.new_weight.copy_(
        state["decoder_new_weight"].to(
            device=decoder_embedding.new_weight.device,
            dtype=decoder_embedding.new_weight.dtype
        )
    )
    lm_head.new_weight.copy_(
        state["lm_head_new_weight"].to(
            device=lm_head.new_weight.device,
            dtype=lm_head.new_weight.dtype
        )
    )

# Restore optimizer, scheduler, and GradScaler
optimizer.load_state_dict(state["optimizer_state_dict"])
scheduler.load_state_dict(state["scheduler_state_dict"])
scaler.load_state_dict(state["scaler_state_dict"])
scaler._per_optimizer_states.clear()

# Restore random-number-generator states
random.setstate(state["python_rng_state"])
np.random.set_state(state["numpy_rng_state"])
torch.set_rng_state(state["torch_rng_state"])

if torch.cuda.is_available() and state["cuda_rng_state_all"] is not None:
    torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])

# Restore completed-epoch metadata
start_epoch = state["completed_epoch"] + 1
global_step = state["global_step"]
best_val_loss = state["best_val_loss"]
epochs_without_improvement = state["epochs_without_improvement"]
training_history = state["training_history"]

# Discard gradients left by the interrupted run
optimizer.zero_grad(set_to_none=True)

print("Restored from:", checkpoint_path)
print("Completed epoch:", state["completed_epoch"])
print("Next epoch:", start_epoch)
print("Global step:", global_step)
print("Best validation loss:", best_val_loss)
print("GradScaler scale:", scaler.get_scale())
print(
    "Parameters with gradients:",
    sum(p.grad is not None for p in trainable_params)
)

Restored from: /kaggle/working/santali_vocab_extended_lora/latest_checkpoint/training_state.pt
Completed epoch: 3
Next epoch: 4
Global step: 3354
Best validation loss: 2.5691809874461837
GradScaler scale: 131072.0
Parameters with gradients: 0


In [200]:

# ============================================================
# Resumable training loop — numerically safer gradient clipping
# ============================================================

import time
import json
import os
import torch
from tqdm.auto import tqdm

# Safety checks: confirm this is the restored epoch-3 run.
assert start_epoch == 4, f"Expected resume at epoch 4, got {start_epoch}"
assert global_step == 3354, f"Unexpected global step: {global_step}"
assert len(training_history) == 3, "Expected history for epochs 1–3"
assert all(torch.isfinite(p).all().item() for p in trainable_params), \
    "A trainable parameter is non-finite before training."

# Recreate the parameter list from the current model.
base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

lora_params = [
    p for name, p in extended_model.named_parameters()
    if "lora_" in name and p.requires_grad
]
extension_params = [
    decoder_embedding.new_weight,
    lm_head.new_weight,
]
trainable_params = lora_params + extension_params

optimizer_param_ids = {
    id(p) for group in optimizer.param_groups for p in group["params"]
}
assert optimizer_param_ids == {id(p) for p in trainable_params}, \
    "Optimizer parameters do not match the current model."

run_start = time.time()
stop_early = False
skipped_updates_this_run = 0

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    extended_model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    processed_batches = 0
    epoch_start = time.time()

    progress = tqdm(
        enumerate(train_loader, start=1),
        total=len(train_loader),
        desc=f"Epoch {epoch}/{NUM_EPOCHS} [TRAIN]",
    )

    for batch_idx, batch in progress:
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            outputs = extended_model(
                **batch,
                use_cache=False,
                return_dict=True,
            )
            loss = outputs.loss

        if not torch.isfinite(loss).item():
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}, batch {batch_idx}. "
                "Training stopped; checkpoint remains available."
            )

        # Preserve correct scaling for the final partial accumulation group.
        group_start = (
            (batch_idx - 1) // GRAD_ACCUMULATION_STEPS
        ) * GRAD_ACCUMULATION_STEPS
        group_size = min(
            GRAD_ACCUMULATION_STEPS,
            len(train_loader) - group_start,
        )

        scaled_loss = loss / group_size
        scaler.scale(scaled_loss).backward()

        running_loss += loss.detach().float().item()
        processed_batches += 1

        is_group_end = (
            batch_idx % GRAD_ACCUMULATION_STEPS == 0
            or batch_idx == len(train_loader)
        )

        if is_group_end:
            # Unscale once per accumulation group. GradScaler records
            # non-finite gradients here and can skip the optimizer update.
            scaler.unscale_(optimizer)

            gradients_finite = True
            grad_sq_sum = torch.zeros(
                (), device=device, dtype=torch.float64
            )

            for p in trainable_params:
                if p.grad is None:
                    continue

                grad = p.grad.detach()

                if not torch.isfinite(grad).all().item():
                    gradients_finite = False
                    break

                # FP64 accumulation avoids overflow in the norm reduction.
                grad_sq_sum += grad.to(torch.float64).square().sum()

            if gradients_finite:
                grad_norm = grad_sq_sum.sqrt()

                if not torch.isfinite(grad_norm).item():
                    gradients_finite = False

            if gradients_finite:
                # Equivalent to global-norm clipping, without an FP32
                # sum-of-squares reduction that may overflow.
                clip_coefficient = min(
                    1.0,
                    MAX_GRAD_NORM / (grad_norm.item() + 1e-12),
                )

                if clip_coefficient < 1.0:
                    for p in trainable_params:
                        if p.grad is not None:
                            p.grad.mul_(clip_coefficient)

            if gradients_finite:
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                new_scale = scaler.get_scale()

                # A scale decrease indicates GradScaler skipped the update.
                # Do not advance the scheduler or global step in that case.
                update_succeeded = new_scale >= old_scale

                if update_succeeded:
                    scheduler.step()
                    global_step += 1
                else:
                    skipped_updates_this_run += 1
                    print(
                        f"GradScaler skipped update at epoch {epoch}, "
                        f"batch {batch_idx}; scale {old_scale} -> {new_scale}"
                    )
            else:
                # GradScaler.unscale_ has recorded non-finite gradients.
                # scaler.step() skips the optimizer update; scaler.update()
                # lowers the scale. No scheduler/global-step advance.
                scaler.step(optimizer)
                scaler.update()
                skipped_updates_this_run += 1

                print(
                    f"Skipped non-finite gradient update at epoch {epoch}, "
                    f"batch {batch_idx}; new scale={scaler.get_scale()}"
                )

            optimizer.zero_grad(set_to_none=True)

            # Stop if parameters became invalid despite the safeguards.
            for name, p in extended_model.named_parameters():
                if p.requires_grad and not torch.isfinite(p).all().item():
                    raise RuntimeError(
                        f"Non-finite trainable parameter after update: {name}"
                    )

        avg_loss = running_loss / max(processed_batches, 1)
        percent = 100.0 * batch_idx / len(train_loader)
        elapsed = (time.time() - epoch_start) / 60
        eta = elapsed / max(percent / 100.0, 1e-8) - elapsed

        gpu_mem = (
            torch.cuda.memory_allocated() / (1024 ** 3)
            if torch.cuda.is_available() else 0.0
        )

        progress.set_postfix({
            "loss": f"{loss.detach().float().item():.4f}",
            "avg": f"{avg_loss:.4f}",
            "done": f"{percent:.1f}%",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
            "elapsed": f"{elapsed:.1f}m",
            "ETA": f"{max(eta, 0):.1f}m",
            "GPU": f"{gpu_mem:.2f}G",
            "skips": skipped_updates_this_run,
        })

        del outputs, loss, scaled_loss

    # Validation after each completed epoch.
    train_loss = running_loss / max(processed_batches, 1)
    val_loss = evaluate_validation()
    epoch_minutes = (time.time() - epoch_start) / 60

    improved = val_loss < best_val_loss - MIN_DELTA

    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val_loss,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "global_step": global_step,
        "epoch_minutes": epoch_minutes,
        "improved": improved,
        "skipped_updates_this_run": skipped_updates_this_run,
    }
    training_history.append(epoch_record)

    # Save the latest completed epoch first.
    save_checkpoint(
        LATEST_CHECKPOINT_DIR,
        epoch,
        best_val_loss,
        epochs_without_improvement,
        training_history,
    )

    # Replace the best checkpoint only when validation improves sufficiently.
    if improved:
        save_checkpoint(
            BEST_CHECKPOINT_DIR,
            epoch,
            best_val_loss,
            epochs_without_improvement,
            training_history,
        )

    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(training_history, f, indent=2)

    print(
        f"\nEpoch {epoch}/{NUM_EPOCHS} complete | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"best_val_loss={best_val_loss:.4f} | "
        f"epoch_time={epoch_minutes:.1f} min"
    )
    print(
        f"Completed: {100.0 * epoch / NUM_EPOCHS:.1f}% of planned epochs | "
        f"patience={epochs_without_improvement}/{PATIENCE} | "
        f"global_step={global_step} | "
        f"skipped updates={skipped_updates_this_run}"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        stop_early = True

    if stop_early:
        break

# Save the final completed state.
if training_history:
    final_epoch = training_history[-1]["epoch"]
    save_checkpoint(
        FINAL_CHECKPOINT_DIR,
        final_epoch,
        best_val_loss,
        epochs_without_improvement,
        training_history,
    )

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(training_history, f, indent=2)

print("\nTraining finished.")
print("Epochs in history:", [item["epoch"] for item in training_history])
print("Best validation loss:", best_val_loss)
print("Global step:", global_step)
print("Skipped updates this run:", skipped_updates_this_run)
print("Latest checkpoint:", LATEST_CHECKPOINT_DIR)
print("Best checkpoint:", BEST_CHECKPOINT_DIR)
print("Final checkpoint:", FINAL_CHECKPOINT_DIR)
print("History:", HISTORY_PATH)


Epoch 4/10 [TRAIN]:   0%|          | 0/4471 [00:00<?, ?it/s]

Skipped non-finite gradient update at epoch 4, batch 2588; new scale=131072.0


VALIDATION:   0%|          | 0/249 [00:00<?, ?it/s]


Epoch 4/10 complete | train_loss=2.7395 | val_loss=2.5515 | best_val_loss=2.5515 | epoch_time=15.9 min
Completed: 40.0% of planned epochs | patience=0/2 | global_step=4471 | skipped updates=1


Epoch 5/10 [TRAIN]:   0%|          | 0/4471 [00:00<?, ?it/s]

...

VALIDATION:   0%|          | 0/249 [00:00<?, ?it/s]


Epoch 9/10 complete | train_loss=2.6547 | val_loss=2.5163 | best_val_loss=2.5163 | epoch_time=15.8 min
Completed: 90.0% of planned epochs | patience=0/2 | global_step=10059 | skipped updates=3


Epoch 10/10 [TRAIN]:   0%|          | 0/4471 [00:00<?, ?it/s]

VALIDATION:   0%|          | 0/249 [00:00<?, ?it/s]


Epoch 10/10 complete | train_loss=2.6497 | val_loss=2.5149 | best_val_loss=2.5149 | epoch_time=15.8 min
Completed: 100.0% of planned epochs | patience=0/2 | global_step=11177 | skipped updates=3

Training finished.
Epochs in history: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Best validation loss: 2.514895272542195
Global step: 11177
Skipped updates this run: 3
Latest checkpoint: /kaggle/working/santali_vocab_extended_lora/latest_checkpoint
Best checkpoint: /kaggle/working/santali_vocab_extended_lora/best_checkpoint
Final checkpoint: /kaggle/working/santali_vocab_extended_lora/final_checkpoint
History: /kaggle/working/santali_vocab_extended_lora/training_history.json


In [201]:

import os
import json
import torch

print("=== Extended Vocabulary Checkpoints ===")

for name, path in [
    ("Best", BEST_CHECKPOINT_DIR),
    ("Latest", LATEST_CHECKPOINT_DIR),
    ("Final", FINAL_CHECKPOINT_DIR),
]:
    print(f"\n{name} checkpoint: {path}")
    print("Directory exists:", os.path.isdir(path))

    if os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            level = root.replace(path, "").count(os.sep)
            indent = "  " * level
            print(f"{indent}{os.path.basename(root)}/")
            for file in files:
                print(f"{indent}  {file}")

print("\n=== Training History ===")

with open(HISTORY_PATH, "r") as f:
    history_data = json.load(f)

print("History file loaded successfully.")
print("History type:", type(history_data).__name__)

if isinstance(history_data, dict):
    print("Available keys:", list(history_data.keys()))
    print("Best validation loss:", best_val_loss)
    print("Global step:", global_step)


=== Extended Vocabulary Checkpoints ===

Best checkpoint: /kaggle/working/santali_vocab_extended_lora/best_checkpoint
Directory exists: True
best_checkpoint/
  training_state.pt
  adapter/
    adapter_config.json
    adapter_model.safetensors
    README.md

Latest checkpoint: /kaggle/working/santali_vocab_extended_lora/latest_checkpoint
Directory exists: True
latest_checkpoint/
  training_state.pt
  adapter/
    adapter_config.json
    adapter_model.safetensors
    README.md

Final checkpoint: /kaggle/working/santali_vocab_extended_lora/final_checkpoint
Directory exists: True
final_checkpoint/
  training_state.pt
  adapter/
    adapter_config.json
    adapter_model.safetensors
    README.md

=== Training History ===
History file loaded successfully.
History type: list


In [202]:

import os
import torch

best_state_path = os.path.join(
    BEST_CHECKPOINT_DIR, "training_state.pt"
)

best_state = torch.load(
    best_state_path,
    map_location="cpu",
    weights_only=False
)

print("=== Best Checkpoint Metadata ===")
print("Completed epoch:", best_state["completed_epoch"])
print("Global step:", best_state["global_step"])
print("Best validation loss:", best_state["best_val_loss"])

print("\n=== Vocabulary Extension Weights ===")

for key in ["decoder_new_weight", "lm_head_new_weight"]:
    tensor = best_state.get(key)

    if tensor is None:
        print(f"{key}: MISSING")
    else:
        print(
            f"{key}: shape={tuple(tensor.shape)}, "
            f"dtype={tensor.dtype}, "
            f"finite={torch.isfinite(tensor).all().item()}"
        )

print("\n=== Adapter Files ===")
adapter_dir = os.path.join(BEST_CHECKPOINT_DIR, "adapter")

for filename in ["adapter_config.json", "adapter_model.safetensors"]:
    filepath = os.path.join(adapter_dir, filename)
    print(f"{filename}: exists={os.path.isfile(filepath)}")

print("\nCheckpoint inspection complete.")


=== Best Checkpoint Metadata ===
Completed epoch: 10
Global step: 11177
Best validation loss: 2.514895272542195

=== Vocabulary Extension Weights ===
decoder_new_weight: shape=(95, 1024), dtype=torch.float32, finite=True
lm_head_new_weight: shape=(95, 1024), dtype=torch.float32, finite=True

=== Adapter Files ===
adapter_config.json: exists=True
adapter_model.safetensors: exists=True

Checkpoint inspection complete.


In [203]:

import os
import torch
from safetensors.torch import load_file
from peft import set_peft_model_state_dict

# 1. Load the best LoRA adapter weights
best_adapter_file = os.path.join(
    BEST_CHECKPOINT_DIR,
    "adapter",
    "adapter_model.safetensors"
)

best_adapter_state = load_file(best_adapter_file)

adapter_result = set_peft_model_state_dict(
    extended_model,
    best_adapter_state
)

# 2. Restore the custom vocabulary rows from the same checkpoint
base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

with torch.no_grad():
    decoder_embedding.new_weight.copy_(
        best_state["decoder_new_weight"].to(
            device=decoder_embedding.new_weight.device,
            dtype=decoder_embedding.new_weight.dtype
        )
    )

    lm_head.new_weight.copy_(
        best_state["lm_head_new_weight"].to(
            device=lm_head.new_weight.device,
            dtype=lm_head.new_weight.dtype
        )
    )

# 3. Verify restored weights
print("Best adapter loaded.")
print("Best checkpoint epoch:", best_state["completed_epoch"])
print("Best validation loss:", best_state["best_val_loss"])

print(
    "Decoder extension finite:",
    torch.isfinite(decoder_embedding.new_weight).all().item()
)
print(
    "LM-head extension finite:",
    torch.isfinite(lm_head.new_weight).all().item()
)
print(
    "Decoder extension shape:",
    tuple(decoder_embedding.new_weight.shape)
)
print(
    "LM-head extension shape:",
    tuple(lm_head.new_weight.shape)
)


Best adapter loaded.
Best checkpoint epoch: 10
Best validation loss: 2.514895272542195
Decoder extension finite: True
LM-head extension finite: True
Decoder extension shape: (95, 1024)
LM-head extension shape: (95, 1024)


In [205]:
import os
import glob

matches = glob.glob(
    "/kaggle/input/datasets/sanjeevanijaiswal/"
    "common-tribal-preprocessing/splits/*santali*test*"
)

print("Matching Santali test files:")
for path in matches:
    print(path)

if not matches:
    print("\nChecking all mounted datasets:")
    for path in glob.glob("/kaggle/input/**/*santali*test*", recursive=True):
        print(path)

Matching Santali test files:
/kaggle/input/datasets/sanjeevanijaiswal/common-tribal-preprocessing/splits/santali_test.csv
/kaggle/input/datasets/sanjeevanijaiswal/common-tribal-preprocessing/splits/santali_test.parquet


In [206]:
import pandas as pd

TEST_CSV_PATH = (
    "/kaggle/input/datasets/sanjeevanijaiswal/"
    "common-tribal-preprocessing/splits/santali_test.csv"
)

test_df = pd.read_csv(TEST_CSV_PATH)

print("Test shape:", test_df.shape)
print("\nColumns:", test_df.columns.tolist())
print("\nFirst 3 rows:")
display(test_df.head(3))

print("\nMissing values:")
print(test_df.isna().sum())

Test shape: (994, 2)

Columns: ['English', 'Santali']

First 3 rows:


,English,Santali
0,Puducherry is one of the most popular tourist ...,ᱡᱟᱹᱛᱤᱭᱟᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱫ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ ᱠᱚᱴᱷᱮᱱ ᱯᱩ...
1,She is widely regarded in modern Indian histor...,ᱱᱟᱦᱟᱜ ᱵᱷᱟᱨᱚᱛᱤᱭᱚ ᱱᱟᱜᱟᱢ ᱚᱞ ᱨᱮ ᱩᱱᱤ ᱫᱚ ᱟᱹᱰᱤ ᱛᱮᱫ ᱜᱮ...
2,"This way, the Robo advisor can go from giving ...","ᱱᱚᱶᱟ ᱞᱮᱠᱟᱛᱮ, ᱯᱚᱨᱟᱢᱮᱥ ᱮᱢ ᱠᱷᱚᱱ ᱮᱦᱚᱵ ᱠᱟᱛᱮᱜ ᱯᱨᱚᱠᱤᱛ..."



Missing values:
English    0
Santali    0
dtype: int64


In [207]:

print("=== Tokenizer ===")
print("Tokenizer type:", type(tok).__name__)
print("Tokenizer vocab size:", len(tok))

print("\n=== Model ===")
print("Model type:", type(extended_model).__name__)
print("Model device:", next(extended_model.parameters()).device)

print("\n=== Extended vocabulary ===")
print("Extension tokenizer function exists:",
      "tokenize_target_with_extension_v2" in globals())

for name in [
    "LANGUAGE_CODE",
    "SRC_LANG",
    "TGT_LANG",
    "INFERENCE_BATCH_SIZE",
    "MAX_SOURCE_LENGTH",
    "MAX_TARGET_LENGTH",
]:
    if name in globals():
        print(f"{name} =", globals()[name])

print("\n=== Existing generation functions ===")
for name in [
    "generate_translation",
    "generate_translations",
    "postprocess",
    "apply_chat_template",
]:
    print(f"{name}: {name in globals()}")


=== Tokenizer ===
Tokenizer type: IndicTransTokenizer
Tokenizer vocab size: 122706

=== Model ===
Model type: PeftModelForSeq2SeqLM
Model device: cuda:0

=== Extended vocabulary ===
Extension tokenizer function exists: True
SRC_LANG = eng_Latn
TGT_LANG = sat_Olck
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

=== Existing generation functions ===
generate_translation: False
generate_translations: False
postprocess: False
apply_chat_template: False


In [209]:
print("=== Decoder embedding wrapper ===")
print(type(decoder_embedding).__name__)
print("Attributes/modules:")

print("Child modules:")
for name, module in decoder_embedding.named_children():
    print(f"  {name}: {type(module).__name__}")

print("\nParameters:")
for name, param in decoder_embedding.named_parameters(recurse=False):
    print(f"  {name}: shape={tuple(param.shape)}, dtype={param.dtype}")

print("\nBuffers:")
for name, buffer in decoder_embedding.named_buffers(recurse=False):
    print(f"  {name}: shape={tuple(buffer.shape)}, dtype={buffer.dtype}")

print("\n=== LM-head wrapper ===")
print(type(lm_head).__name__)

print("Child modules:")
for name, module in lm_head.named_children():
    print(f"  {name}: {type(module).__name__}")

print("\nParameters:")
for name, param in lm_head.named_parameters(recurse=False):
    print(f"  {name}: shape={tuple(param.shape)}, dtype={param.dtype}")

=== Decoder embedding wrapper ===
FrozenBaseExtendedEmbedding
Attributes/modules:
Child modules:

Parameters:
  original_weight: shape=(122672, 1024), dtype=torch.float16
  new_weight: shape=(95, 1024), dtype=torch.float32

Buffers:

=== LM-head wrapper ===
FrozenBaseExtendedLMHead
Child modules:

Parameters:
  original_weight: shape=(122672, 1024), dtype=torch.float16
  new_weight: shape=(95, 1024), dtype=torch.float32


In [210]:
import inspect

print(inspect.getsource(tokenize_target_with_extension_v2))

def tokenize_target_with_extension_v2(text, add_eos=True):
    token_ids = []
    cursor = 0

    for match in _extension_pattern.finditer(text):
        # Encode the text preceding the extension candidate.
        prefix = text[cursor:match.start()]
        token_ids.extend(encode_original_target_segment(prefix))

        # Insert the dedicated ID for the extended vocabulary word.
        candidate = match.group(1)
        token_ids.append(candidate_to_id[candidate])

        cursor = match.end()

    # Encode the remaining text.
    token_ids.extend(encode_original_target_segment(text[cursor:]))

    # Add EOS exactly once, using the model's configured EOS ID.
    eos_id = extended_model.config.eos_token_id
    if add_eos and (not token_ids or token_ids[-1] != eos_id):
        token_ids.append(eos_id)

    return token_ids



In [211]:
print("=== Extension ID mapping ===")
print("Number of candidates:", len(candidate_to_id))
print("Unique extension IDs:", len(set(candidate_to_id.values())))
print("Minimum extension ID:", min(candidate_to_id.values()))
print("Maximum extension ID:", max(candidate_to_id.values()))

print("\n=== Mapping checks ===")
print(
    "All IDs within extension range:",
    all(122672 <= i < 122767 for i in candidate_to_id.values())
)

print("\nSample candidate mappings:")
for word, token_id in list(candidate_to_id.items())[:10]:
    print(repr(word), "->", token_id)

print("\n=== Original target encoder ===")
import inspect
print(inspect.getsource(encode_original_target_segment))

=== Extension ID mapping ===
Number of candidates: 95
Unique extension IDs: 95
Minimum extension ID: 122672
Maximum extension ID: 122766

=== Mapping checks ===
All IDs within extension range: True

Sample candidate mappings:
'ᱤᱧᱟᱹᱜ' -> 122672
'ᱤᱧᱤᱧ' -> 122673
'ᱩᱱᱫᱚ' -> 122674
'ᱫᱳᱠᱟᱱ' -> 122675
'ᱢᱩᱜᱷᱚᱞ' -> 122676
'ᱱᱩᱱᱟᱹᱜ' -> 122677
'ᱟᱞᱮᱭᱟᱜ' -> 122678
'ᱤᱧᱨᱮᱱ' -> 122679
'ᱟᱹᱲᱤᱥ' -> 122680
'ᱦᱟᱦᱟᱲᱟᱜ' -> 122681

=== Original target encoder ===
def encode_original_target_segment(segment):
    if not segment:
        return []

    # Use the tokenizer's actual target-vocabulary mapping.
    with tok.as_target_tokenizer():
        pieces = tok.tokenize(segment)
        token_ids = tok.convert_tokens_to_ids(pieces)

    return token_ids



In [212]:
print("=== Candidate reverse mapping ===")

id_to_candidate = {v: k for k, v in candidate_to_id.items()}

print("Reverse mapping size:", len(id_to_candidate))
print("Unique reverse IDs:", len(id_to_candidate))
print("Sample reverse mappings:")

for token_id, word in list(id_to_candidate.items())[:10]:
    print(token_id, "->", repr(word))

print("\n=== Existing decoding helpers ===")

for name, obj in list(globals().items()):
    if (
        callable(obj)
        and any(term in name.lower() for term in ["decode", "detoken", "extension"])
        and not name.startswith("_")
    ):
        print(name)

=== Candidate reverse mapping ===
Reverse mapping size: 95
Unique reverse IDs: 95
Sample reverse mappings:
122672 -> 'ᱤᱧᱟᱹᱜ'
122673 -> 'ᱤᱧᱤᱧ'
122674 -> 'ᱩᱱᱫᱚ'
122675 -> 'ᱫᱳᱠᱟᱱ'
122676 -> 'ᱢᱩᱜᱷᱚᱞ'
122677 -> 'ᱱᱩᱱᱟᱹᱜ'
122678 -> 'ᱟᱞᱮᱭᱟᱜ'
122679 -> 'ᱤᱧᱨᱮᱱ'
122680 -> 'ᱟᱹᱲᱤᱥ'
122681 -> 'ᱦᱟᱦᱟᱲᱟᱜ'

=== Existing decoding helpers ===
decoder_embeddings
decoder_emb
test_decoder
expanded_decoder
old_decoder
new_decoder
extension_probe
tokenize_target_with_extension
tokenize_target_with_extension_v2
decoder_embedding
decoder
old_decoder_embedding
new_decoder_embedding


In [213]:
import torch

BASE_VOCAB_SIZE = 122672
EXTENDED_VOCAB_SIZE = 122767
id_to_candidate = {v: k for k, v in candidate_to_id.items()}

def decode_extended_ids(token_ids):
    # Convert tensors to a regular Python list.
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.detach().cpu().tolist()

    # Remove padding and stop at EOS.
    eos_id = extended_model.config.eos_token_id
    pad_id = extended_model.config.pad_token_id

    clean_ids = []
    for token_id in token_ids:
        token_id = int(token_id)

        if token_id == eos_id:
            break
        if pad_id is not None and token_id == pad_id:
            continue

        clean_ids.append(token_id)

    # Decode contiguous original-token segments and insert extension words.
    output_parts = []
    original_segment = []

    def flush_original_segment():
        if original_segment:
            output_parts.append(
                tok.decode(
                    original_segment,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False
                )
            )
            original_segment.clear()

    for token_id in clean_ids:
        if token_id >= BASE_VOCAB_SIZE:
            flush_original_segment()
            word = id_to_candidate.get(token_id)
            if word is None:
                raise ValueError(f"Unknown extension token ID: {token_id}")
            output_parts.append(word)
        else:
            original_segment.append(token_id)

    flush_original_segment()
    return "".join(output_parts).strip()

# Round-trip test: extension IDs should recover their exact candidate words.
sample_words = list(candidate_to_id.keys())[:5]
sample_ids = [candidate_to_id[word] for word in sample_words]

print("Expected words:", sample_words)
print("Decoded text:", decode_extended_ids(sample_ids))
print("Exact match:", decode_extended_ids(sample_ids) == "".join(sample_words))

Expected words: ['ᱤᱧᱟᱹᱜ', 'ᱤᱧᱤᱧ', 'ᱩᱱᱫᱚ', 'ᱫᱳᱠᱟᱱ', 'ᱢᱩᱜᱷᱚᱞ']
Decoded text: ᱤᱧᱟᱹᱜᱤᱧᱤᱧᱩᱱᱫᱚᱫᱳᱠᱟᱱᱢᱩᱜᱷᱚᱞ
Exact match: True


In [215]:
print("Tokenizer class:", type(tok))
print("Tokenizer methods related to language/source/target:")

print([
    name for name in dir(tok)
    if any(term in name.lower() for term in [
        "source", "target", "lang", "token", "preprocess"
    ])
])

print("\nTokenizer call signature:")
import inspect
print(inspect.signature(tok.__call__))

print("\nTokenizer initialization/config attributes:")
for name in ["src_lang", "tgt_lang", "language", "lang_code_to_id"]:
    print(f"{name}:", getattr(tok, name, "Not available"))

Tokenizer class: <class 'transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.tokenization_indictrans.IndicTransTokenizer'>
Tokenizer methods related to language/source/target:
['SPECIAL_TOKENS_ATTRIBUTES', '_add_tokens', '_added_tokens_decoder', '_added_tokens_encoder', '_additional_special_tokens', '_bos_token', '_cls_token', '_convert_id_to_token', '_convert_token_to_id', '_convert_token_to_id_with_added_voc', '_decode_use_source_tokenizer', '_eos_token', '_in_target_context_manager', '_mask_token', '_pad_token', '_pad_token_type_id', '_sep_token', '_src_tokenize', '_switch_to_target_mode', '_tgt_tokenize', '_tokenize', '_unk_token', 'add_new_language_tags', 'add_special_tokens', 'add_tokens', 'added_tokens_decoder', 'added_tokens_encoder', 'additional_special_tokens', 'additional_special_tokens_ids', 'all_special_tokens', 'all_special_tokens_extended', 'as_target_tokenizer', 'bos_token', 'bos_token_id', 'build_inputs_with_special_tokens

In [216]:
import inspect

print("Source tokenization method:")
print(inspect.getsource(type(tok)._src_tokenize))

print("\nTokenizer preparation method:")
print(inspect.getsource(type(tok).prepare_for_tokenization))

print("\nSource preprocessing helper candidates:")
print([
    name for name in globals()
    if any(term in name.lower() for term in [
        "preprocess", "source", "tokenize", "lang"
    ])
])


Source tokenization method:
    def _src_tokenize(self, text: str) -> List[str]:
        src_lang, tgt_lang, text = text.split(" ", 2)
        assert src_lang in LANGUAGE_TAGS, f"Invalid source language tag: {src_lang}"
        assert tgt_lang in LANGUAGE_TAGS, f"Invalid target language tag: {tgt_lang}"
        return [src_lang, tgt_lang] + self.spm.EncodeAsPieces(text)


Tokenizer preparation method:
    def prepare_for_tokenization(
        self, text: str, is_split_into_words: bool = False, **kwargs
    ) -> Tuple[str, Dict[str, Any]]:
        """
        Performs any necessary transformations before tokenization.

        This method should pop the arguments from kwargs and return the remaining `kwargs` as well. We test the
        `kwargs` at the end of the encoding process to be sure all the arguments have been used.

        Args:
            text (`str`):
                The text to prepare.
            is_split_into_words (`bool`, *optional*, defaults to `False`):
            

In [217]:
source_text = str(test_df.iloc[0]["English"])

formatted_source = f"{SRC_LANG} {TGT_LANG} {source_text}"

encoded = tok(
    formatted_source,
    truncation=True,
    max_length=MAX_SOURCE_LENGTH,
    return_tensors="pt"
)

print("Original English:", source_text)
print("Formatted source:", formatted_source)
print("Input IDs shape:", encoded["input_ids"].shape)
print("Attention mask shape:", encoded["attention_mask"].shape)
print("Decoded input:", tok.decode(
    encoded["input_ids"][0],
    skip_special_tokens=False
))

Original English: Puducherry is one of the most popular tourist spots in India for national and international tourists.
Formatted source: eng_Latn sat_Olck Puducherry is one of the most popular tourist spots in India for national and international tourists.
Input IDs shape: torch.Size([1, 21])
Attention mask shape: torch.Size([1, 21])
Decoded input: . ग्रौ चाहै परिणामकारताणॆन्नुंऱ् वॆळ्ळंills पहरे ꯃꯤꯁꯤꯡप्पळ्ळि ठेवावेس पहिल नियुक्ति तत्कालिन पाबेनशींदा इत्यादिना पणि</s>


In [218]:
source_text = str(test_df.iloc[0]["English"])
formatted_source = f"{SRC_LANG} {TGT_LANG} {source_text}"

print("Source language:", SRC_LANG)
print("Target language:", TGT_LANG)
print("Special tokens:", tok.special_tokens_map)
print("Vocabulary size:", len(tok))

print("\nSource pieces:")
print(tok._src_tokenize(formatted_source))

print("\nEncoded token IDs:")
print(encoded["input_ids"][0].tolist())

print("\nTokens corresponding to IDs:")
print(tok.convert_ids_to_tokens(encoded["input_ids"][0].tolist()))

print("\nDecoded with special tokens:")
print(tok.decode(encoded["input_ids"][0], skip_special_tokens=False))

Source language: eng_Latn
Target language: sat_Olck
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}
Vocabulary size: 122706

Source pieces:
['eng_Latn', 'sat_Olck', '▁Puducherry', '▁is', '▁one', '▁of', '▁the', '▁most', '▁popular', '▁tourist', '▁sp', 'ots', '▁in', '▁India', '▁for', '▁national', '▁and', '▁international', '▁tourists', '.']

Encoded token IDs:
[4, 29925, 99584, 14469, 46755, 4123, 5975, 66956, 77567, 91312, 32844, 43533, 3332, 14331, 13422, 67609, 6879, 71837, 104125, 6379, 2]

Tokens corresponding to IDs:
['eng_Latn', 'sat_Olck', '▁Puducherry', '▁is', '▁one', '▁of', '▁the', '▁most', '▁popular', '▁tourist', '▁sp', 'ots', '▁in', '▁India', '▁for', '▁national', '▁and', '▁international', '▁tourists', '.', '</s>']

Decoded with special tokens:
. ग्रौ चाहै परिणामकारताणॆन्नुंऱ् वॆळ्ळंills पहरे ꯃꯤꯁꯤꯡप्पळ्ळि ठेवावेس पहिल नियुक्ति तत्कालिन पाबेनशींदा इत्यादिना पणि</s>


In [219]:
import inspect

print("Decode implementation:")
print(inspect.getsource(type(tok).decode))

print("\nDecode source-tokenizer flag:")
print(getattr(tok, "_decode_use_source_tokenizer", "Not available"))

print("\nSource tokenization round-trip variables:")
for name in ["source_roundtrip", "source_ids", "source_encoding", "formatted_source"]:
    if name in globals():
        print(f"{name}: {type(globals()[name]).__name__}")

Decode implementation:
    def decode(
        self,
        token_ids: Union[int, List[int], "np.ndarray", "torch.Tensor", "tf.Tensor"],
        skip_special_tokens: bool = False,
        clean_up_tokenization_spaces: bool = None,
        **kwargs,
    ) -> str:
        """
        Converts a sequence of ids in a string, using the tokenizer and vocabulary with options to remove special
        tokens and clean up tokenization spaces.

        Similar to doing `self.convert_tokens_to_string(self.convert_ids_to_tokens(token_ids))`.

        Args:
            token_ids (`Union[int, List[int], np.ndarray, torch.Tensor, tf.Tensor]`):
                List of tokenized input ids. Can be obtained using the `__call__` method.
            skip_special_tokens (`bool`, *optional*, defaults to `False`):
                Whether or not to remove special tokens in the decoding.
            clean_up_tokenization_spaces (`bool`, *optional*):
                Whether or not to clean up the tokenization s

In [220]:
for name in ["source_roundtrip", "source_ids", "source_encoding", "formatted_source"]:
    if name in globals():
        value = globals()[name]
        print(f"\n{name}:")
        if isinstance(value, str):
            print(repr(value[:300]))
        elif hasattr(value, "shape"):
            print("shape:", tuple(value.shape))
            print("first IDs:", value.flatten()[:25].tolist())
        elif hasattr(value, "keys"):
            print("keys:", list(value.keys()))
            for key, item in value.items():
                if hasattr(item, "shape"):
                    print(f"{key} shape:", tuple(item.shape))


source_roundtrip:
"According to Minhaj, Iltutmish re-asserted Delhi's control over Badaun, Awadh, Banaras, and Siwalik in a series of campaigns."

source_ids:
shape: (2, 4)
first IDs: [5, 10, 20, 2, 7, 12, 25, 2]

source_encoding:
keys: ['input_ids', 'attention_mask']
input_ids shape: (1, 24)
attention_mask shape: (1, 24)

formatted_source:
'eng_Latn sat_Olck Puducherry is one of the most popular tourist spots in India for national and international tourists.'


In [222]:
import torch

base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

print("Model parameter dtype:", next(extended_model.parameters()).dtype)
print("Decoder embedding class:", type(decoder_embedding).__name__)
print("LM head class:", type(lm_head).__name__)

print("\nDecoder embedding:")
for name, param in decoder_embedding.named_parameters(recurse=False):
    print(name, param.dtype, tuple(param.shape))

print("\nLM head:")
for name, param in lm_head.named_parameters(recurse=False):
    print(name, param.dtype, tuple(param.shape))

print("\nDecoder layer norm:")
norm = base_model.model.decoder.layers[0].self_attn_layer_norm
print("weight:", norm.weight.dtype)
print("bias:", norm.bias.dtype if norm.bias is not None else None)

print("\nModel config dtype:", getattr(extended_model.config, "torch_dtype", None))

Model parameter dtype: torch.float16
Decoder embedding class: FrozenBaseExtendedEmbedding
LM head class: FrozenBaseExtendedLMHead

Decoder embedding:
original_weight torch.float16 (122672, 1024)
new_weight torch.float32 (95, 1024)

LM head:
original_weight torch.float16 (122672, 1024)
new_weight torch.float32 (95, 1024)

Decoder layer norm:
weight: torch.float16
bias: torch.float16

Model config dtype: torch.float16


In [223]:
import inspect

print("=== Extended decoder embedding forward ===")
print(inspect.getsource(type(decoder_embedding).forward))

print("\n=== Extended LM head forward ===")
print(inspect.getsource(type(lm_head).forward))

=== Extended decoder embedding forward ===
    def forward(self, input_ids):
        combined_weight = torch.cat(
            [self.original_weight, self.new_weight],
            dim=0
        )

        return F.embedding(
            input_ids,
            combined_weight,
            padding_idx=self.padding_idx
        )


=== Extended LM head forward ===
    def forward(self, hidden_states):
        combined_weight = torch.cat(
            [self.original_weight, self.new_weight],
            dim=0
        )

        return F.linear(
            hidden_states,
            combined_weight,
            self.bias
        )



In [224]:
import torch
import torch.nn.functional as F

def embedding_forward_fp16_safe(self, input_ids):
    combined_weight = torch.cat(
        [self.original_weight, self.new_weight],
        dim=0
    ).to(dtype=self.original_weight.dtype)

    return F.embedding(
        input_ids,
        combined_weight,
        padding_idx=self.padding_idx
    )

def lm_head_forward_fp16_safe(self, hidden_states):
    combined_weight = torch.cat(
        [self.original_weight, self.new_weight],
        dim=0
    ).to(dtype=hidden_states.dtype)

    bias = self.bias
    if bias is not None:
        bias = bias.to(dtype=hidden_states.dtype)

    return F.linear(hidden_states, combined_weight, bias)

# Patch only the extended model's two custom wrappers.
import types

decoder_embedding.forward = types.MethodType(
    embedding_forward_fp16_safe, decoder_embedding
)
lm_head.forward = types.MethodType(
    lm_head_forward_fp16_safe, lm_head
)

print("Decoder extension parameters remain FP32:",
      decoder_embedding.new_weight.dtype)
print("LM-head extension parameters remain FP32:",
      lm_head.new_weight.dtype)
print("Base model dtype:", next(extended_model.parameters()).dtype)
print("FP16-safe wrapper methods installed.")

Decoder extension parameters remain FP32: torch.float32
LM-head extension parameters remain FP32: torch.float32
Base model dtype: torch.float16
FP16-safe wrapper methods installed.


In [225]:
import torch

extended_model.eval()
device = next(extended_model.parameters()).device

source_text = str(test_df.iloc[0]["English"])
reference_text = str(test_df.iloc[0]["Santali"])

encoded = tok(
    f"{SRC_LANG} {TGT_LANG} {source_text}",
    truncation=True,
    max_length=MAX_SOURCE_LENGTH,
    return_tensors="pt"
)
encoded = {key: value.to(device) for key, value in encoded.items()}

with torch.inference_mode():
    generated_ids = extended_model.generate(
        **encoded,
        max_length=MAX_TARGET_LENGTH,
        num_beams=4,
        early_stopping=True
    )

prediction = decode_extended_ids(generated_ids[0])

print("English:", source_text)
print("Reference Santali:", reference_text)
print("Predicted Santali:", prediction)
print("Generated token IDs:", generated_ids[0].detach().cpu().tolist())

English: Puducherry is one of the most popular tourist spots in India for national and international tourists.
Reference Santali: ᱡᱟᱹᱛᱤᱭᱟᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱫ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ ᱠᱚᱴᱷᱮᱱ ᱯᱩᱫᱩᱪᱮᱨᱤ ᱫᱚ ᱦᱩᱭᱩᱜ ᱠᱟᱱᱟ ᱵᱷᱟᱨᱚᱛ ᱨᱮᱭᱟᱜ ᱥᱚᱨᱮᱥ ᱧᱩᱢᱡᱚᱝ ᱥᱟᱸᱜᱷᱟᱨ ᱴᱷᱟᱣ ᱾
Predicted Santali: 
Generated token IDs: [2, 98621, 99345, 105544, 92596, 44857, 88011, 99508, 104182, 51412, 60440, 74948, 48806, 90180, 74934, 101142, 96137, 85364, 63364, 31428, 2]


In [226]:
print("EOS token ID:", extended_model.config.eos_token_id)
print("Decoder start token ID:", extended_model.config.decoder_start_token_id)
print("BOS token ID:", extended_model.config.bos_token_id)
print("Pad token ID:", extended_model.config.pad_token_id)

print("\nGeneration config:")
print("EOS:", extended_model.generation_config.eos_token_id)
print("Decoder start:", extended_model.generation_config.decoder_start_token_id)
print("BOS:", extended_model.generation_config.bos_token_id)
print("Pad:", extended_model.generation_config.pad_token_id)

print("\nGenerated IDs:", generated_ids[0].detach().cpu().tolist())
print("\nRaw tokens:", tok.convert_ids_to_tokens(
    generated_ids[0].detach().cpu().tolist()
))

EOS token ID: 2
Decoder start token ID: 2
BOS token ID: 0
Pad token ID: 1

Generation config:
EOS: 2
Decoder start: 2
BOS: 0
Pad: 1

Generated IDs: [2, 98621, 99345, 105544, 92596, 44857, 88011, 99508, 104182, 51412, 60440, 74948, 48806, 90180, 74934, 101142, 96137, 85364, 63364, 31428, 2]

Raw tokens: ['</s>', '▁चीजन', '▁ᱚᱴ', 'rawn', '▁بخشڻ', '▁फोग', '▁avoid', '▁छिय', '▁लबार', '▁7.4', 'दरां', '▁Porn', '▁स्विट्जरलैंड', 'स़हिर', '▁ক', '▁ڪیلینڊر', '▁آھ', '▁adv', '▁जनु', '▁फोऱ', '</s>']


In [227]:
def decode_extended_ids(token_ids):
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.detach().cpu().tolist()

    eos_id = extended_model.config.eos_token_id
    pad_id = extended_model.config.pad_token_id
    decoder_start_id = extended_model.config.decoder_start_token_id

    # Remove only the initial decoder-start token.
    if token_ids and int(token_ids[0]) == decoder_start_id:
        token_ids = token_ids[1:]

    clean_ids = []
    for token_id in token_ids:
        token_id = int(token_id)

        if token_id == eos_id:
            break
        if pad_id is not None and token_id == pad_id:
            continue

        clean_ids.append(token_id)

    output_parts = []
    original_segment = []

    def flush_original_segment():
        if original_segment:
            output_parts.append(
                tok.decode(
                    original_segment,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False
                )
            )
            original_segment.clear()

    for token_id in clean_ids:
        if token_id >= BASE_VOCAB_SIZE:
            flush_original_segment()
            word = id_to_candidate.get(token_id)
            if word is None:
                raise ValueError(f"Unknown extension token ID: {token_id}")
            output_parts.append(word)
        else:
            original_segment.append(token_id)

    flush_original_segment()
    return "".join(output_parts).strip()

print("Corrected prediction:")
print(decode_extended_ids(generated_ids[0]))

Corrected prediction:
ᱯᱩᱫᱩᱪᱮᱨᱤ ᱡᱟᱹᱛᱤᱭᱟᱹᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱛ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ ᱠᱚ ᱞᱟᱹᱜᱤᱫ ᱵᱷᱟᱨᱚᱛ ᱨᱮᱭᱟᱜ ᱥᱚᱨᱮᱥ ᱦᱚᱲ ᱠᱩᱥᱤ ᱥᱟᱸᱜᱷᱟᱨ ᱴᱷᱟᱶ ᱠᱟᱱᱟ ᱾


In [228]:
import os
import json
import torch

print("Best checkpoint:", BEST_CHECKPOINT_DIR)

print("\nTraining history summary:")
with open(HISTORY_PATH, "r") as f:
    history = json.load(f)

print("History type:", type(history).__name__)
if isinstance(history, dict):
    print("Keys:", list(history.keys()))
    print(json.dumps(history, indent=2)[:4000])
elif isinstance(history, list):
    print("Number of records:", len(history))
    print(json.dumps(history[-2:], indent=2)[:4000])

print("\nSaved training-state keys:")
print(list(best_state.keys()))

Best checkpoint: /kaggle/working/santali_vocab_extended_lora/best_checkpoint

Training history summary:
History type: list
Number of records: 10
[
  {
    "epoch": 9,
    "train_loss": 2.654723513555644,
    "val_loss": 2.516281647854541,
    "best_val_loss": 2.516281647854541,
    "learning_rate": 1.0336560627017055e-05,
    "global_step": 10059,
    "epoch_minutes": 15.767508431275685,
    "improved": true,
    "skipped_updates_this_run": 3
  },
  {
    "epoch": 10,
    "train_loss": 2.649658505400937,
    "val_loss": 2.514895272542195,
    "best_val_loss": 2.514895272542195,
    "learning_rate": 2.766251728907321e-08,
    "global_step": 11177,
    "epoch_minutes": 15.77529573837916,
    "improved": true,
    "skipped_updates_this_run": 3
  }
]

Saved training-state keys:
['completed_epoch', 'global_step', 'best_val_loss', 'epochs_without_improvement', 'training_history', 'decoder_new_weight', 'lm_head_new_weight', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 

In [229]:
import torch

best_state = torch.load(
    os.path.join(BEST_CHECKPOINT_DIR, "training_state.pt"),
    map_location="cpu",
    weights_only=False
)

base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

decoder_saved = best_state["decoder_new_weight"]
lm_saved = best_state["lm_head_new_weight"]

print("Decoder extension matches checkpoint:",
      torch.allclose(
          decoder_embedding.new_weight.detach().cpu(),
          decoder_saved.float(),
          rtol=1e-3,
          atol=1e-3
      ))

print("LM-head extension matches checkpoint:",
      torch.allclose(
          lm_head.new_weight.detach().cpu(),
          lm_saved.float(),
          rtol=1e-3,
          atol=1e-3
      ))

print("Saved decoder shape:", tuple(decoder_saved.shape))
print("Saved LM-head shape:", tuple(lm_saved.shape))
print("Best validation loss:", best_state["best_val_loss"])

Decoder extension matches checkpoint: True
LM-head extension matches checkpoint: True
Saved decoder shape: (95, 1024)
Saved LM-head shape: (95, 1024)
Best validation loss: 2.514895272542195


In [230]:
import torch
import pandas as pd
from tqdm.auto import tqdm

extended_model.eval()
device = next(extended_model.parameters()).device

sample_results = []

for i in tqdm(range(4), desc="Testing extended model"):
    source_text = str(test_df.iloc[i]["English"])
    reference_text = str(test_df.iloc[i]["Santali"])

    encoded = tok(
        f"{SRC_LANG} {TGT_LANG} {source_text}",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt"
    )
    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.inference_mode():
        generated_ids = extended_model.generate(
            **encoded,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    prediction = decode_extended_ids(generated_ids[0])

    sample_results.append({
        "English": source_text,
        "Reference Santali": reference_text,
        "Predicted Santali": prediction
    })

display(pd.DataFrame(sample_results))

Testing extended model:   0%|          | 0/4 [00:00<?, ?it/s]

,English,Reference Santali,Predicted Santali
0,Puducherry is one of the most popular tourist ...,ᱡᱟᱹᱛᱤᱭᱟᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱫ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ ᱠᱚᱴᱷᱮᱱ ᱯᱩ...,ᱯᱩᱫᱩᱪᱮᱨᱤ ᱡᱟᱹᱛᱤᱭᱟᱹᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱛ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ...
1,She is widely regarded in modern Indian histor...,ᱱᱟᱦᱟᱜ ᱵᱷᱟᱨᱚᱛᱤᱭᱚ ᱱᱟᱜᱟᱢ ᱚᱞ ᱨᱮ ᱩᱱᱤ ᱫᱚ ᱟᱹᱰᱤ ᱛᱮᱫ ᱜᱮ...,ᱩᱱᱤ ᱫᱚ ᱱᱟᱦᱟᱜ ᱵᱷᱟᱨᱚᱛ ᱨᱮᱭᱟᱜ ᱱᱟᱜᱟᱢ ᱪᱤᱛᱟᱹᱨ ᱨᱮ ᱟᱠᱵᱚ...
2,"This way, the Robo advisor can go from giving ...","ᱱᱚᱶᱟ ᱞᱮᱠᱟᱛᱮ, ᱯᱚᱨᱟᱢᱮᱥ ᱮᱢ ᱠᱷᱚᱱ ᱮᱦᱚᱵ ᱠᱟᱛᱮᱜ ᱯᱨᱚᱠᱤᱛ...",ᱱᱚᱶᱟ ᱞᱮᱠᱟᱛᱮ ᱨᱳᱵᱳ ᱫᱤᱥᱦᱩᱫᱤᱥ ᱮᱢᱚᱜᱤᱡ ᱟᱢᱟᱜ ᱠᱟᱹᱣᱰᱤ ᱫ...
3,He hung the portraits of all these three perso...,ᱩᱱᱤ ᱫᱚ ᱟᱭᱟᱜ ᱜᱳᱣᱨᱤ ᱠᱩᱧᱡᱚ ᱨᱮᱱᱟᱜ ᱛᱟᱦᱮᱱ ᱚᱲᱟᱜ ᱨᱮ ᱱᱩ...,ᱩᱱᱤ ᱫᱚ ᱟᱭᱟᱜ ᱜᱳᱣᱨᱤ ᱠᱩᱱ ᱟᱜ ᱚᱲᱟᱜ ᱨᱮ ᱱᱚᱣᱟ ᱯᱮᱭᱟ ᱦᱚᱲ...


In [232]:
print("=== Preprocessed test data ===")
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
display(test_df.head(3))

print("\n=== Null values ===")
print(test_df.isnull().sum())

print("\n=== Existing test data files ===")
import os

for root, dirs, files in os.walk(
    "/kaggle/input/datasets/sanjeevanijaiswal/common-tribal-preprocessing"
):
    for file in files:
        if "santali_test" in file.lower():
            print(os.path.join(root, file))

=== Preprocessed test data ===
Shape: (994, 2)
Columns: ['English', 'Santali']


,English,Santali
0,Puducherry is one of the most popular tourist ...,ᱡᱟᱹᱛᱤᱭᱟᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱫ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ ᱠᱚᱴᱷᱮᱱ ᱯᱩ...
1,She is widely regarded in modern Indian histor...,ᱱᱟᱦᱟᱜ ᱵᱷᱟᱨᱚᱛᱤᱭᱚ ᱱᱟᱜᱟᱢ ᱚᱞ ᱨᱮ ᱩᱱᱤ ᱫᱚ ᱟᱹᱰᱤ ᱛᱮᱫ ᱜᱮ...
2,"This way, the Robo advisor can go from giving ...","ᱱᱚᱶᱟ ᱞᱮᱠᱟᱛᱮ, ᱯᱚᱨᱟᱢᱮᱥ ᱮᱢ ᱠᱷᱚᱱ ᱮᱦᱚᱵ ᱠᱟᱛᱮᱜ ᱯᱨᱚᱠᱤᱛ..."



=== Null values ===
English    0
Santali    0
dtype: int64

=== Existing test data files ===
/kaggle/input/datasets/sanjeevanijaiswal/common-tribal-preprocessing/splits/santali_test.csv
/kaggle/input/datasets/sanjeevanijaiswal/common-tribal-preprocessing/splits/santali_test.parquet


In [233]:
import pandas as pd
import torch
from tqdm.auto import tqdm

extended_model.eval()
device = next(extended_model.parameters()).device

extended_predictions = []
extended_references = test_df["Santali"].astype(str).tolist()
extended_sources = test_df["English"].astype(str).tolist()

for i, source_text in enumerate(
    tqdm(extended_sources, desc="Extended model inference")
):
    encoded = tok(
        f"{SRC_LANG} {TGT_LANG} {source_text}",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt"
    )
    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.inference_mode():
        generated_ids = extended_model.generate(
            **encoded,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    prediction = decode_extended_ids(generated_ids[0])
    extended_predictions.append(prediction)

extended_results_df = pd.DataFrame({
    "English": extended_sources,
    "Reference Santali": extended_references,
    "Extended Prediction": extended_predictions
})

print("Total examples:", len(extended_results_df))
print("Empty predictions:", (extended_results_df["Extended Prediction"].str.strip() == "").sum())

display(extended_results_df.head(5))

Extended model inference:   0%|          | 0/994 [00:00<?, ?it/s]

Total examples: 994
Empty predictions: 0


,English,Reference Santali,Extended Prediction
0,Puducherry is one of the most popular tourist ...,ᱡᱟᱹᱛᱤᱭᱟᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱫ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ ᱠᱚᱴᱷᱮᱱ ᱯᱩ...,ᱯᱩᱫᱩᱪᱮᱨᱤ ᱡᱟᱹᱛᱤᱭᱟᱹᱨᱤ ᱟᱨ ᱡᱮᱜᱮᱫ ᱡᱟᱠᱟᱛ ᱥᱟᱸᱜᱷᱟᱨᱤᱭᱟᱹ...
1,She is widely regarded in modern Indian histor...,ᱱᱟᱦᱟᱜ ᱵᱷᱟᱨᱚᱛᱤᱭᱚ ᱱᱟᱜᱟᱢ ᱚᱞ ᱨᱮ ᱩᱱᱤ ᱫᱚ ᱟᱹᱰᱤ ᱛᱮᱫ ᱜᱮ...,ᱩᱱᱤ ᱫᱚ ᱱᱟᱦᱟᱜ ᱵᱷᱟᱨᱚᱛ ᱨᱮᱭᱟᱜ ᱱᱟᱜᱟᱢ ᱪᱤᱛᱟᱹᱨ ᱨᱮ ᱟᱠᱵᱚ...
2,"This way, the Robo advisor can go from giving ...","ᱱᱚᱶᱟ ᱞᱮᱠᱟᱛᱮ, ᱯᱚᱨᱟᱢᱮᱥ ᱮᱢ ᱠᱷᱚᱱ ᱮᱦᱚᱵ ᱠᱟᱛᱮᱜ ᱯᱨᱚᱠᱤᱛ...",ᱱᱚᱶᱟ ᱞᱮᱠᱟᱛᱮ ᱨᱳᱵᱳ ᱫᱤᱥᱦᱩᱫᱤᱥ ᱮᱢᱚᱜᱤᱡ ᱟᱢᱟᱜ ᱠᱟᱹᱣᱰᱤ ᱫ...
3,He hung the portraits of all these three perso...,ᱩᱱᱤ ᱫᱚ ᱟᱭᱟᱜ ᱜᱳᱣᱨᱤ ᱠᱩᱧᱡᱚ ᱨᱮᱱᱟᱜ ᱛᱟᱦᱮᱱ ᱚᱲᱟᱜ ᱨᱮ ᱱᱩ...,ᱩᱱᱤ ᱫᱚ ᱟᱭᱟᱜ ᱜᱳᱣᱨᱤ ᱠᱩᱱ ᱟᱜ ᱚᱲᱟᱜ ᱨᱮ ᱱᱚᱣᱟ ᱯᱮᱭᱟ ᱦᱚᱲ...
4,I am unsure about driving in the US after driv...,ᱤᱧ ᱵᱷᱟᱨᱚᱛ ᱨᱮ ᱜᱟᱹᱰᱤ ᱪᱟᱞᱟᱣ ᱛᱟᱭᱚᱢ ᱢᱟᱨᱠᱤᱱ ᱡᱩᱠᱛᱚ ᱨᱟ...,ᱵᱷᱟᱨᱚᱛ ᱨᱮ ᱰᱨᱟᱭᱵᱷᱤᱝ ᱛᱟᱭᱚᱢ ᱤᱧ ᱢᱟᱨᱠᱤᱱ ᱡᱩᱠᱛᱚᱨᱟᱥᱴᱨᱚ...


In [234]:
import sacrebleu
import numpy as np
import pandas as pd

references = extended_results_df["Reference Santali"].astype(str).tolist()
predictions = extended_results_df["Extended Prediction"].astype(str).tolist()

# Corpus-level translation metrics
bleu = sacrebleu.corpus_bleu(predictions, [references])
chrf = sacrebleu.corpus_chrf(predictions, [references])

# Exact match after stripping leading/trailing whitespace
exact_match = np.mean([
    pred.strip() == ref.strip()
    for pred, ref in zip(predictions, references)
]) * 100

# Character-length ratio, using the same definition as the baseline
hypothesis_length = sum(len(pred) for pred in predictions)
reference_length = sum(len(ref) for ref in references)
length_ratio = (
    hypothesis_length / reference_length
    if reference_length > 0 else float("nan")
)

extended_metrics = {
    "Model": "Extended-vocabulary LoRA",
    "Test examples": len(references),
    "SacreBLEU": round(bleu.score, 4),
    "chrF": round(chrf.score, 4),
    "Exact match (%)": round(exact_match, 4),
    "Hypothesis/reference length ratio": round(length_ratio, 4)
}

print("=== Extended-model test metrics ===")
display(pd.DataFrame([extended_metrics]))

print("\nBLEU signature:", bleu.get_signature())
print("chrF signature:", chrf.get_signature())

=== Extended-model test metrics ===


,Model,Test examples,SacreBLEU,chrF,Exact match (%),Hypothesis/reference length ratio
0,Extended-vocabulary LoRA,994,10.8258,44.8445,0.1006,1.0261


AttributeError: 'BLEUScore' object has no attribute 'get_signature'

In [236]:
import sacrebleu

bleu_metric = sacrebleu.metrics.BLEU()
bleu_metric.corpus_score(predictions, [references])

chrf_metric = sacrebleu.metrics.CHRF()
chrf_metric.corpus_score(predictions, [references])

print("SacreBLEU signature:", bleu_metric.get_signature())
print("chrF signature:", chrf_metric.get_signature())

SacreBLEU signature: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0
chrF signature: nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0


In [237]:
import os
import json
import pandas as pd

RESULTS_DIR = "/kaggle/working/santali_vocab_extended_lora/evaluation"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save predictions for all 994 test examples
predictions_path = os.path.join(
    RESULTS_DIR, "extended_model_test_predictions.csv"
)
extended_results_df.to_csv(predictions_path, index=False, encoding="utf-8-sig")

# Add reproducibility details to the existing metrics
extended_metrics["BLEU signature"] = bleu_metric.get_signature().__str__()
extended_metrics["chrF signature"] = chrf_metric.get_signature().__str__()
extended_metrics["SacreBLEU version"] = sacrebleu.__version__

metrics_path = os.path.join(RESULTS_DIR, "extended_model_test_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(extended_metrics, f, ensure_ascii=False, indent=2)

print("Saved predictions:", predictions_path)
print("Saved metrics:", metrics_path)
print("Prediction rows:", len(extended_results_df))
print("\nSaved metrics:")
print(json.dumps(extended_metrics, ensure_ascii=False, indent=2))

Saved predictions: /kaggle/working/santali_vocab_extended_lora/evaluation/extended_model_test_predictions.csv
Saved metrics: /kaggle/working/santali_vocab_extended_lora/evaluation/extended_model_test_metrics.json
Prediction rows: 994

Saved metrics:
{
  "Model": "Extended-vocabulary LoRA",
  "Test examples": 994,
  "SacreBLEU": 10.8258,
  "chrF": 44.8445,
  "Exact match (%)": 0.1006,
  "Hypothesis/reference length ratio": 1.0261,
  "BLEU signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
  "chrF signature": "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0",
  "SacreBLEU version": "2.6.0"
}



# Extended-Vocabulary LoRA: Results and Comparative Analysis

## 1. Experimental Objective

The objective of this experiment is to investigate whether extending the vocabulary of a pretrained multilingual neural machine translation model and learning representations for the newly added tokens can improve English-to-Santali translation.

The experiment compares two approaches:

1. **Baseline LoRA:** Fine-tuning the pretrained IndicTrans2 model using LoRA adapters while retaining its original vocabulary.
2. **Extended-Vocabulary LoRA:** Fine-tuning the model with an extended vocabulary containing 95 additional Santali tokens, together with trainable embeddings and output-projection weights for these tokens.

The base model used is `ai4bharat/indictrans2-indic-indic-1B`.

## 2. Experimental Setup

| Parameter | Configuration |
|---|---|
| Base model | IndicTrans2 Indic 1B |
| Translation direction | English → Santali |
| Source language code | `eng_Latn` |
| Target language code | `sat_Olck` |
| Additional vocabulary tokens | 95 |
| Original vocabulary size | 122,672 |
| Extended vocabulary size | 122,767 |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| LoRA target modules | `q_proj`, `v_proj` |
| Maximum source length | 256 tokens |
| Maximum target length | 256 tokens |
| Training epochs completed | 10 |
| Best validation loss | 2.5149 |
| Test examples | 994 |

The extended-vocabulary model was evaluated on the preprocessed test set containing 994 English–Santali sentence pairs. The same test references were used for comparison with the baseline results.

## 3. Evaluation Results

| Evaluation Metric | Baseline LoRA | Extended-Vocabulary LoRA |
|---|---:|---:|
| SacreBLEU | 8.3859 | **10.8258** |
| chrF | 41.0353 | **44.8445** |
| Exact Match (%) | 0.0000 | **0.1006** |
| Hypothesis/Reference Length Ratio | 1.2350 | **1.0261** |

### Metric Interpretation

- **SacreBLEU:** Measures n-gram overlap between generated translations and reference translations. A higher score indicates better overlap. The extended model achieved a score of 10.8258 compared with 8.3859 for the baseline.

- **chrF:** Measures character n-gram similarity. It is particularly useful for languages with rich morphology and differences in word formation. The score increased from 41.0353 to 44.8445.

- **Exact Match:** Measures the percentage of predictions that exactly match their references after stripping leading and trailing whitespace. The extended model achieved 0.1006%, equivalent to 1 exact match out of 994 examples when calculated at sentence level.

- **Hypothesis/Reference Length Ratio:** Measures the total generated character length divided by the total reference character length. A value closer to 1 indicates a closer match in overall output length, although it does not directly measure translation correctness.

## 4. Comparative Analysis

### 4.1 SacreBLEU Improvement

The BLEU score increased from 8.3859 to 10.8258.

- Absolute improvement: **2.4399 BLEU points**
- Relative improvement: **approximately 29.1%**

This indicates that the extended-vocabulary model produced translations with greater n-gram overlap with the reference translations. The improvement is encouraging, although the absolute BLEU score remains relatively low.

### 4.2 chrF Improvement

The chrF score increased from 41.0353 to 44.8445.

- Absolute improvement: **3.8092 points**
- Relative improvement: **approximately 9.3%**

The improvement in character-level similarity suggests that the extended model generated text that more closely resembles the reference Santali translations at the character level.

### 4.3 Output Length

The length ratio decreased from 1.2350 to 1.0261.

The baseline generated approximately 23.5% more characters than the reference corpus in aggregate, whereas the extended model generated approximately 2.6% more characters.

This is a substantial improvement in aggregate output-length alignment. However, a ratio close to 1 does not guarantee that individual translations are accurate or semantically equivalent.

### 4.4 Exact Match

The extended model achieved an exact-match rate of 0.1006%, compared with 0% for the baseline.

Although this represents one exact sentence match, exact match is a strict metric: a translation can be correct or understandable without matching the reference character for character.

## 5. Summary of Findings

The extended-vocabulary model outperformed the baseline on all four reported metrics:

1. Higher SacreBLEU score.
2. Higher chrF score.
3. A nonzero exact-match rate.
4. An aggregate output-length ratio closer to 1.

These findings suggest that vocabulary extension is a promising approach for improving low-resource English-to-Santali translation. The simultaneous improvements in BLEU and chrF provide evidence that the gains are not limited to output length alone.

However, these results do not yet establish that vocabulary extension by itself caused the improvements. Differences in training conditions, initialization, or other implementation details may also contribute.

## 6. Research Significance

This experiment investigates a representation-level adaptation strategy rather than relying exclusively on conventional LoRA fine-tuning with a fixed vocabulary.

The extended model adds 95 Santali tokens and learns corresponding input embeddings and output-projection weights. This provides a basis for studying whether more direct token representations can improve translation quality in a low-resource language.

The experiment is relevant to the broader research question:

**Can vocabulary extension and appropriate initialization of newly introduced token representations improve translation quality for low-resource Indian languages?**

The current results are preliminary evidence supporting further investigation of this question.

## 7. Limitations and Next Steps

The following limitations should be addressed before drawing strong conclusions:

1. **Controlled comparison:** Verify that both models use identical train/validation/test partitions, source formatting, decoding parameters, and metric definitions.
2. **Initialization ablation:** Compare the current extension initialization with alternative initialization methods, such as random initialization and initialization informed by related-language representations.
3. **Multiple seeds:** Repeat training with multiple random seeds and report mean and standard deviation to assess whether improvements are robust.
4. **Statistical significance:** Use paired bootstrap resampling or another appropriate paired evaluation method to determine whether the metric differences are statistically meaningful.
5. **Qualitative analysis:** Inspect representative successful and unsuccessful translations, including cases where newly added tokens occur.
6. **Vocabulary analysis:** Measure tokenizer fertility and examine whether the added tokens reduce fragmentation of relevant Santali words.
7. **Data leakage:** Confirm that training data does not overlap with the evaluation references. This is essential for interpreting test performance reliably.
8. **Generalization:** Evaluate on an independently sourced test set, if available, to assess performance beyond the current benchmark.

## 8. Conclusion

On the 994-example test set, the extended-vocabulary LoRA model achieved a SacreBLEU score of 10.8258 and a chrF score of 44.8445, compared with 8.3859 and 41.0353 for the baseline, respectively. Its aggregate hypothesis/reference length ratio also improved from 1.2350 to 1.0261.

The results are promising and motivate further investigation of vocabulary-aware adaptation for low-resource Santali translation. Additional controlled experiments, initialization ablations, and statistical testing are needed to establish the robustness and cause of the observed improvements.


# SAVING MODEL

In [240]:
import os
import json
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

ROOT_DIR = Path("/kaggle/working/santali_vocab_extended_lora")
BUNDLE_DIR = ROOT_DIR / "portable_model_bundle"
EVAL_DIR = ROOT_DIR / "evaluation"
ZIP_PATH = Path("/kaggle/working/santali_extended_model_and_results.zip")

BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Copy the best adapter
import shutil

adapter_src = ROOT_DIR / "best_checkpoint" / "adapter"
adapter_dst = BUNDLE_DIR / "adapter"

if not (adapter_src / "adapter_config.json").exists():
    raise FileNotFoundError(f"Missing best adapter: {adapter_src}")

if adapter_dst.exists():
    shutil.rmtree(adapter_dst)
shutil.copytree(adapter_src, adapter_dst)

# 2. Save the learned extension weights
base_model = extended_model.base_model.model
decoder_embedding = base_model.model.decoder.embed_tokens
lm_head = base_model.lm_head

torch.save(
    {
        "decoder_new_weight": decoder_embedding.new_weight.detach().cpu(),
        "lm_head_new_weight": lm_head.new_weight.detach().cpu(),
    },
    BUNDLE_DIR / "extension_weights.pt"
)

# 3. Save the vocabulary mapping
with open(BUNDLE_DIR / "extended_vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "base_vocab_size": int(BASE_VOCAB_SIZE),
            "extended_vocab_size": int(EXTENDED_VOCAB_SIZE),
            "candidate_to_id": candidate_to_id,
            "id_to_candidate": {
                str(token_id): word
                for word, token_id in candidate_to_id.items()
            },
        },
        f,
        ensure_ascii=False,
        indent=2
    )

# 4. Save model metadata using the correct checkpoint key
metadata = {
    "base_model": "ai4bharat/indictrans2-indic-indic-1B",
    "source_language": SRC_LANG,
    "target_language": TGT_LANG,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "base_vocab_size": int(BASE_VOCAB_SIZE),
    "extended_vocab_size": int(EXTENDED_VOCAB_SIZE),
    "number_of_added_tokens": len(candidate_to_id),
    "best_epoch": int(best_state["completed_epoch"]),
    "best_validation_loss": float(best_state["best_val_loss"]),
    "global_step": int(best_state["global_step"]),
    "note": (
        "The base model is not included. Reconstruct using the matching "
        "IndicTrans2 implementation, adapter, vocabulary, and extension weights."
    ),
}

with open(BUNDLE_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# 5. Ensure predictions and metrics exist
extended_results_df.to_csv(
    EVAL_DIR / "extended_model_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(EVAL_DIR / "extended_model_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(extended_metrics, f, ensure_ascii=False, indent=2)

with open(EVAL_DIR / "results_summary.txt", "w", encoding="utf-8") as f:
    f.write("Santali Extended-Vocabulary LoRA Evaluation\n")
    f.write("Test examples: 994\n")
    f.write("SacreBLEU: 10.8258\n")
    f.write("chrF: 44.8445\n")
    f.write("Exact match (%): 0.1006\n")
    f.write("Hypothesis/reference length ratio: 1.0261\n")

# 6. Package best, final, latest, evaluation, and portable bundle
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

folders = [
    ROOT_DIR / "best_checkpoint",
    ROOT_DIR / "final_checkpoint",
    ROOT_DIR / "latest_checkpoint",
    EVAL_DIR,
    BUNDLE_DIR,
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        if folder.exists():
            for path in folder.rglob("*"):
                if path.is_file() and path != ZIP_PATH:
                    zf.write(path, path.relative_to(ROOT_DIR))

# 7. Verify archive and show download
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()
    if bad_file is not None:
        raise RuntimeError(f"ZIP integrity check failed: {bad_file}")
    print("Files packaged:", len(zf.namelist()))

print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print("ZIP verified:", ZIP_PATH.exists())
display(FileLink(str(ZIP_PATH), result_html_prefix="Download your model and results: "))

Files packaged: 21
ZIP size: 1902.12 MB
ZIP verified: True


/kaggle/working/santali_extended_model_and_results.zip